#Official-Final Notebook

### ***Time Series Analysis for Residual Chlorine Forecast at Los Filtros Water Treatment Plant***

External generalization and transfer learning from the Alexander Orr source plant to the Los Filtros target plant (Caguas Outlet)


## Dataset description:

Los Filtros hourly dataset currently including:

1. Timestamp
2. Residual chlorine (mg/L)
3. Turbidity (NTU)

Flowrate will use the standardized name `finishwater` when received.


##Standards for residual chlorine: Min of 0.2 mg/L to 4.0 mg/L (EPA)


In [ ]:
#Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import calendar

#Interactive plot
import plotly.graph_objects as go

#For seasonal decompositions
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import month_plot, quarter_plot, plot_acf, plot_pacf

#Holt-Winters
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing

#ARIMA Models
from statsmodels.tsa.stattools import adfuller
!pip install pymannkendall
import pymannkendall as mk
from statsmodels.tsa.statespace.sarimax import SARIMAX
from scipy.stats import norm

#White test for heteroscedasticity
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_white

#For error measurement
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_percentage_error, mean_absolute_error, r2_score

#For cross-validation
from sklearn.model_selection import TimeSeriesSplit, ParameterGrid
from sklearn.base import clone

#For parameter tuning
from sklearn.model_selection import train_test_split, ParameterGrid
from itertools import product #Generates combinations of elements from multiple lists

#For feature selection
from sklearn.feature_selection import SequentialFeatureSelector

#For data standardization
from sklearn.preprocessing import StandardScaler, QuantileTransformer

#Deep learning
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras import layers
from tensorflow.keras.utils import plot_model
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.gofplots import qqplot
#Architecture plots
!apt-get install graphviz -y
!pip install pydot
from graphviz import Digraph

#TFT
!pip install tft-torch
import torch
from torch.utils.data import Dataset, DataLoader
from tft_torch.tft import TemporalFusionTransformer
from omegaconf import OmegaConf
import tft_torch.loss as tft_loss

#Classic ML libraries
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from itertools import product
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
from scipy.stats import jarque_bera
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch


#Hypothesis testing
from pathlib import Path
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations
from statsmodels.stats.multitest import multipletests


import os
import gc
import glob
import joblib
import hashlib
import json
import warnings

#Radar/Spider plot
from math import pi
import re


## 1. Environment and Google Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import json
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, confusion_matrix
)

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

import tensorflow as tf
import torch
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

BASE_OUTPUT = "/content/drive/MyDrive/Forecasting_AlexanderOrr"
SAVED_MODEL_DIR = os.path.join(BASE_OUTPUT, "Saved_Models")
EXTERNAL_OUTPUT = (
    "/content/drive/MyDrive/External_Validation_LosFiltrosCaguas"
)
os.makedirs(EXTERNAL_OUTPUT, exist_ok=True)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Saved models:", SAVED_MODEL_DIR)
print("External outputs:", EXTERNAL_OUTPUT)


## 2. Dataset settings

The notebook loads the los filtros dataset directly from Google Drive.
In the next cell, confirm only the Drive path, timestamp column and source
column names.

The currently available measurements are:

- `totalchlorine`: residual chlorine in mg/L
- `turbidity`: turbidity in NTU

Flow rate is not yet available. When received, map its original name to the
Alexander Orr standardized model name `finishwater`:

```python
"flowrate_losfiltros": "finishwater"
```

Until then, flow-dependent saved configurations are automatically skipped.
Chlorine-only and chlorine-plus-turbidity configurations continue to run.


In [ ]:
EXTERNAL_DATA_PATH = (
    "/content/drive/MyDrive/LosFiltros_External_Validation_Caguas/"
    "losfiltros_hourly_data_caguas.xlsx"
)

# The source provides both timestamps. Use `timestamp` because it is UTC;
# `timestamp_local` represents the corresponding Puerto Rico local time.
TIMESTAMP_COL = "timestamp"
SOURCE_TIMEZONE = "UTC"
LOCAL_TIMEZONE = "America/Puerto_Rico"

# Unit contract used throughout EDA, transfer learning, and evaluation.
# No unit conversion is performed in this notebook.
VARIABLE_UNITS = {
    "totalchlorine": "mg/L",
    "turbidity": "NTU",
    "finishwater": "mgd",
    "chlorine_conversion": "binary 0/1",
}

COLUMN_MAP = {
    # Update these source names only if your file uses different labels:
    "chlorine_losfiltros": "totalchlorine",
    "turbidity_losfiltros": "turbidity",

    # Flow rate is not currently available. Uncomment when received:
    # "flowrate_losfiltros": "finishwater",

    # Optional future operational indicator:
    # "original_conversion_name": "chlorine_conversion",
}

DATASET_NAME = "Los_Filtros_Caguas"


ORIGINAL_UPPER_THRESHOLD = 4.0
EXTERNAL_OPERATIONAL_THRESHOLD = None


## 2A. Optional model-selection controls

Leave a setting as `None` to run everything available. Enter a list to run only selected items.

Examples:

- `ACTIVE_SEASONS = ["Winter"]`
- `ACTIVE_CONFIGURATIONS = ["full_model"]`
- `ACTIVE_MODEL_FAMILIES = ["LSTM"]`

Names must match the saved folder metadata exactly.


In [ ]:
# DEFAULT: automatically run every saved model, configuration, and season.
ACTIVE_MODEL_FAMILIES = None
ACTIVE_SEASONS = None
ACTIVE_CONFIGURATIONS = None
ACTIVE_CLASSIFICATION_MODELS = None

# Example for one limited run:
# ACTIVE_MODEL_FAMILIES = ["ANN", "LSTM", "TFT"]
# ACTIVE_SEASONS = ["Winter"]
# ACTIVE_CONFIGURATIONS = ["full_model"]
# ACTIVE_CLASSIFICATION_MODELS = ["RandomForest"]


def normalize_configuration_name(configuration):
    normalized = (
        str(configuration)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    aliases = {
        "univariate": "univariate",
        "turbidity": "turbidity",
        "flowrate": "flowrate",
        "flowrate_turbidity": "flowrate_turbidity",
        "full_model": "full_model"
    }

    return aliases.get(normalized, normalized)


def selection_is_active(
    model_family,
    season,
    configuration
):
    configuration = normalize_configuration_name(
        configuration
    )

    normalized_active_configurations = (
        None
        if ACTIVE_CONFIGURATIONS is None
        else [
            normalize_configuration_name(value)
            for value in ACTIVE_CONFIGURATIONS
        ]
    )

    if (
        ACTIVE_MODEL_FAMILIES is not None
        and model_family not in ACTIVE_MODEL_FAMILIES
    ):
        return False

    if (
        ACTIVE_SEASONS is not None
        and season not in ACTIVE_SEASONS
    ):
        return False

    if (
        normalized_active_configurations is not None
        and configuration not in normalized_active_configurations
    ):
        return False

    return True


print("Active model families:", ACTIVE_MODEL_FAMILIES or "ALL")
print("Active seasons:", ACTIVE_SEASONS or "ALL")
print("Active configurations:", ACTIVE_CONFIGURATIONS or "ALL")
print(
    "Active classification models:",
    ACTIVE_CLASSIFICATION_MODELS or "ALL"
)


## 3. Import the external dataset


In [ ]:
suffix = Path(EXTERNAL_DATA_PATH).suffix.lower()

if suffix in {".xlsx", ".xls"}:
    external_raw = pd.read_excel(EXTERNAL_DATA_PATH)
elif suffix == ".csv":
    external_raw = pd.read_csv(EXTERNAL_DATA_PATH)
else:
    raise ValueError("Use an Excel or CSV external dataset.")

external_raw = external_raw.rename(columns=COLUMN_MAP)

if TIMESTAMP_COL not in external_raw.columns:
    raise ValueError(
        f"Timestamp column not found: {TIMESTAMP_COL}"
    )

# los filtros timestamps are already UTC.
external_raw[TIMESTAMP_COL] = pd.to_datetime(
    external_raw[TIMESTAMP_COL],
    utc=True
)

# Verify the supplied Puerto Rico local clock against the UTC timestamp.
if "timestamp_local" in external_raw.columns:
    supplied_local_clock = pd.to_datetime(
        external_raw["timestamp_local"],
        errors="coerce"
    )
    expected_local_clock = (
        external_raw[TIMESTAMP_COL]
        .dt.tz_convert(LOCAL_TIMEZONE)
        .dt.tz_localize(None)
    )
    timestamp_mismatches = int(
        (supplied_local_clock != expected_local_clock).sum()
    )
    print("UTC/local timestamp mismatches:", timestamp_mismatches)
    assert timestamp_mismatches == 0, (
        "timestamp_local does not match timestamp converted to Puerto Rico time."
    )
external_raw = external_raw.set_index(TIMESTAMP_COL)

external_raw.head()


## 4. Standardize timestamps, frequency, and shared time features


In [ ]:
def prepare_external_dataframe(
    data,
    source_timezone,
    freq="h"
):
    # Create a separate working copy and preserve the original input.
    out = data.copy().sort_index()

    if str(source_timezone).upper() != "UTC":
        raise ValueError(
            "This los filtros workflow expects source timestamps already in UTC."
        )

    # The source `timestamp` column is UTC. Assign UTC directly rather than
    # interpreting it as Puerto Rico local time.
    out.index = pd.to_datetime(out.index, utc=True)
    out = out.sort_index()

    # Duplicate UTC timestamps cannot represent separate hourly forecast
    # origins. Average duplicate numeric observations if any are present.
    if out.index.duplicated().any():
        print("Duplicate UTC timestamps:", int(out.index.duplicated().sum()))
        out = out.groupby(level=0).mean(numeric_only=True)

    # Identify true missing UTC hours before enforcing a regular hourly index.
    complete_index = pd.date_range(
        start=out.index.min(),
        end=out.index.max(),
        freq=freq,
        tz="UTC"
    )
    missing_timestamps = complete_index.difference(out.index)
    out = out.reindex(complete_index)
    out.index.name = data.index.name

    # Fill continuous measurements using past observations only. Forward
    # filling avoids using a future measurement to construct an earlier model
    # input. Audit flags identify every value filled by this operation.
    continuous_columns = [
        column
        for column in ["totalchlorine", "turbidity", "finishwater"]
        if column in out.columns
    ]

    filling_rows = []
    for column in continuous_columns:
        out[column] = pd.to_numeric(out[column], errors="coerce")
        flag_column = f"{column}_forward_filled"
        missing_before_mask = out[column].isna()
        missing_before = int(missing_before_mask.sum())

        out[column] = out[column].ffill()
        out[flag_column] = missing_before_mask & out[column].notna()

        missing_after = int(out[column].isna().sum())
        filling_rows.append({
            "Variable": column,
            "MissingBeforeForwardFill": missing_before,
            "ForwardFilledValues": missing_before - missing_after,
            "MissingAfterForwardFill": missing_after
        })

    # Audit the calendar columns supplied with the dataset against Puerto Rico
    # local time. Forecasting features are created separately from UTC.
    supplied_time_columns = ["day", "year", "month", "hour"]
    missing_time_columns = [
        column for column in supplied_time_columns
        if column not in out.columns
    ]
    if missing_time_columns:
        raise ValueError(
            "Expected los filtros time columns are missing: "
            f"{missing_time_columns}"
        )

    local_index = out.index.tz_convert(LOCAL_TIMEZONE)
    local_calendar_values = {
        "day": local_index.day,
        "year": local_index.year,
        "month": local_index.month,
        "hour": local_index.hour
    }
    calendar_mismatches = {}
    for column, local_values in local_calendar_values.items():
        supplied_values = pd.to_numeric(out[column], errors="coerce")
        expected_values = pd.Series(local_values, index=out.index)
        comparison_mask = supplied_values.notna()
        calendar_mismatches[column] = int(
            (
                supplied_values[comparison_mask] != expected_values[comparison_mask]
            ).sum()
        )

    print(
        "Supplied calendar mismatches versus Puerto Rico time:",
        calendar_mismatches
    )

    # Reconstruct the complete Puerto Rico local clock from the canonical
    # UTC index, including rows inserted for missing source hours.
    out["timestamp_local"] = local_index.tz_localize(None)

    # Explicit UTC features are the only calendar features used by forecasting
    # and classification models.
    out["utc_year"] = out.index.year
    out["utc_month"] = out.index.month
    out["utc_day"] = out.index.day
    out["utc_day_of_week"] = out.index.dayofweek
    out["utc_hour"] = out.index.hour
    out["utc_month_year"] = out.index.strftime("%Y-%m")
    out["utc_hour_sin"] = np.sin(2 * np.pi * out["utc_hour"] / 24.0)
    out["utc_hour_cos"] = np.cos(2 * np.pi * out["utc_hour"] / 24.0)
    out["utc_dow_sin"] = np.sin(
        2 * np.pi * out["utc_day_of_week"] / 7.0
    )
    out["utc_dow_cos"] = np.cos(
        2 * np.pi * out["utc_day_of_week"] / 7.0
    )

    # Explicit Puerto Rico features support operational EDA. Puerto Rico does
    # not observe DST, but its local clock is four hours behind UTC.
    out["pr_year"] = local_index.year
    out["pr_month"] = local_index.month
    out["pr_day"] = local_index.day
    out["pr_day_of_week"] = local_index.dayofweek
    out["pr_hour"] = local_index.hour
    out["pr_month_year"] = local_index.strftime("%Y-%m")
    out["pr_hour_sin"] = np.sin(2 * np.pi * out["pr_hour"] / 24.0)
    out["pr_hour_cos"] = np.cos(2 * np.pi * out["pr_hour"] / 24.0)
    out["pr_dow_sin"] = np.sin(
        2 * np.pi * out["pr_day_of_week"] / 7.0
    )
    out["pr_dow_cos"] = np.cos(
        2 * np.pi * out["pr_day_of_week"] / 7.0
    )

    # Generic calendar names remain Puerto Rico-local for EDA, matching the
    # Alexander Orr convention of local EDA plus explicit UTC model features.
    out["year"] = out["pr_year"]
    out["month"] = out["pr_month"]
    out["day"] = out["pr_day"]
    out["day_of_week"] = out["pr_day_of_week"]
    out["dayofweek"] = out["pr_day_of_week"]
    out["hour"] = out["pr_hour"]
    out["hour_sin"] = out["pr_hour_sin"]
    out["hour_cos"] = out["pr_hour_cos"]
    out["dow_sin"] = out["pr_dow_sin"]
    out["dow_cos"] = out["pr_dow_cos"]

    # Remove only leading rows that cannot be filled from prior observations.
    out = out.dropna(subset=continuous_columns)

    out.attrs["missing_timestamps_before_filling"] = missing_timestamps
    out.attrs["forward_fill_summary"] = pd.DataFrame(filling_rows)

    out["static_dummy"] = 0.0

    # This operational condition is unavailable for los filtros. A zero
    # placeholder maintains compatible multivariable DL input dimensions but
    # must not be interpreted as a measured Puerto Rico operating condition.
    if "chlorine_conversion" not in out.columns:
        out["chlorine_conversion"] = 0
        print(
            "chlorine_conversion was unavailable and was set to 0. "
            "Interpret multivariable transfer cautiously."
        )

    print("Missing hourly timestamps before filling:", len(missing_timestamps))
    print("Past-only forward-fill summary:")
    display(pd.DataFrame(filling_rows))

    return out


losfiltroscaguas = prepare_external_dataframe(
    external_raw,
    SOURCE_TIMEZONE
)

# Residual chlorine and turbidity are required for the current analysis.
required_current_columns = ["totalchlorine", "turbidity"]
missing_current_columns = [
    column for column in required_current_columns
    if column not in losfiltroscaguas.columns
]
if missing_current_columns:
    raise ValueError(
        "Required los filtros columns are missing: "
        f"{missing_current_columns}. Check COLUMN_MAP."
    )

# Add Puerto Rico-local seasons to the master dataframe for operational EDA.
season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
}
losfiltroscaguas["season"] = losfiltroscaguas["pr_month"].map(season_map)

print("\n" + "=" * 70)
print("MASTER LOS FILTROS CAGUAS DATAFRAME: losfiltroscaguas")
print("=" * 70)
display(losfiltroscaguas.head())
display(losfiltroscaguas.tail())

print("Shape:", losfiltroscaguas.shape)
print("Date range:", losfiltroscaguas.index.min(), "to", losfiltroscaguas.index.max())
print("Timezone:", losfiltroscaguas.index.tz)
print("Frequency:", pd.infer_freq(losfiltroscaguas.index))
print("Configured measurement units:", VARIABLE_UNITS)
print("Duplicate timestamps:", losfiltroscaguas.index.duplicated().sum())
print("Total missing values:", int(losfiltroscaguas.isna().sum().sum()))

# Confirm that explicit model-calendar features match the canonical UTC index.
assert str(losfiltroscaguas.index.tz) == "UTC"
assert losfiltroscaguas.index.is_monotonic_increasing
assert not losfiltroscaguas.index.has_duplicates
assert np.array_equal(
    losfiltroscaguas["utc_year"].to_numpy(),
    losfiltroscaguas.index.year
)
assert np.array_equal(
    losfiltroscaguas["utc_month"].to_numpy(),
    losfiltroscaguas.index.month
)
assert np.array_equal(
    losfiltroscaguas["utc_hour"].to_numpy(),
    losfiltroscaguas.index.hour
)
assert (
    losfiltroscaguas.index.to_series().diff().dropna()
    == pd.Timedelta(hours=1)
).all()
print("UTC timeline and calendar-feature integrity checks passed.")

# Confirm Puerto Rico-local features and create a separate local-index EDA copy.
pr_index = losfiltroscaguas.index.tz_convert(LOCAL_TIMEZONE)
assert np.array_equal(losfiltroscaguas["pr_year"].to_numpy(), pr_index.year)
assert np.array_equal(losfiltroscaguas["pr_month"].to_numpy(), pr_index.month)
assert np.array_equal(losfiltroscaguas["pr_hour"].to_numpy(), pr_index.hour)

losfiltroscaguas_pr = losfiltroscaguas.copy(deep=True)
losfiltroscaguas_pr.index = pr_index

print("Puerto Rico calendar-feature integrity checks passed.")
print("\nDataframe information:")
losfiltroscaguas.info()


In [ ]:
#Plot improvement function
def setup_times_new_roman():
    available_fonts = {f.name for f in fm.fontManager.ttflist}
    if "Times New Roman" in available_fonts:
        chosen_font = "Times New Roman"
        print("Using Times New Roman")
    else:
        chosen_font = "DejaVu Serif"
        print("Using fallback:", chosen_font)
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": [chosen_font],
        "figure.dpi": 300,
        "savefig.dpi": 500,
        "figure.figsize": (14, 6),
        "font.size": 20,
        "axes.titlesize": 24,
        "axes.labelsize": 18,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "legend.fontsize": 16,
        "lines.linewidth": 2.5,
        "lines.markersize": 6,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "figure.autolayout": True
    })

setup_times_new_roman()


##***Exploratory Data Analysis***


In [ ]:
#Initial inspection and missing value assessment
print('Summary')
print(losfiltroscaguas.head())
print(losfiltroscaguas.tail())
print('Missing values')
print(losfiltroscaguas.isna().sum())
print(len(losfiltroscaguas))
print('Date range:', losfiltroscaguas.index.min(), 'to', losfiltroscaguas.index.max())


### Missing timestamps and past-only forward filling

los filtrosimestamps are already UTC. Gaps represent missing observations rather than daylight-saving transitions.


### Missing timestamps and past-only forward filling

los filtros timestamps are supplied in UTC. Missing hours represent absent
observations rather than daylight-saving transitions because Puerto Rico does
not observe DST. Continuous measurements are forward-filled from past values
only, and audit flags identify every filled observation.


In [ ]:
# Audit missing hourly timestamps and values filled from prior observations.
missing_timestamps = losfiltroscaguas.attrs.get(
    "missing_timestamps_before_filling",
    pd.DatetimeIndex([])
)

forward_fill_summary = losfiltroscaguas.attrs.get(
    "forward_fill_summary",
    pd.DataFrame()
)

print("Inferred frequency after filling:", pd.infer_freq(losfiltroscaguas.index))
print("Missing hourly timestamps found:", len(missing_timestamps))

if len(missing_timestamps) > 0:
    print("First missing UTC timestamps:")
    display(pd.Series(missing_timestamps[:20], name="MissingTimestampUTC"))

display(forward_fill_summary)

forward_fill_flag_columns = [
    column for column in losfiltroscaguas.columns
    if column.endswith("_forward_filled")
]

if forward_fill_flag_columns:
    monthly_forward_fill_counts = (
        losfiltroscaguas_pr[forward_fill_flag_columns]
        .resample("ME")
        .sum()
    )

    display(monthly_forward_fill_counts)

    monthly_forward_fill_counts.plot(
        figsize=(12, 5),
        marker="o"
    )
    plt.title("Past-Only Filled Observations by Puerto Rico Local Month")
    plt.xlabel("Month (Puerto Rico Local Time)")
    plt.ylabel("Forward-Filled Hourly Values")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# Preflight audit for UTC modeling and Puerto Rico-local EDA.
assert isinstance(losfiltroscaguas.index, pd.DatetimeIndex)
assert str(losfiltroscaguas.index.tz) == "UTC"
assert losfiltroscaguas.index.is_monotonic_increasing
assert not losfiltroscaguas.index.has_duplicates
assert (
    losfiltroscaguas.index.to_series().diff().dropna()
    == pd.Timedelta(hours=1)
).all()

required_utc_features = [
    "utc_year", "utc_month", "utc_day",
    "utc_day_of_week", "utc_hour",
    "utc_hour_sin", "utc_hour_cos",
    "utc_dow_sin", "utc_dow_cos"
]
required_pr_features = [
    "pr_year", "pr_month", "pr_day",
    "pr_day_of_week", "pr_hour"
]
missing_features = [
    column for column in required_utc_features + required_pr_features
    if column not in losfiltroscaguas.columns
]
assert not missing_features, f"Missing time features: {missing_features}"

assert losfiltroscaguas_pr.index.tz is not None
assert str(losfiltroscaguas_pr.index.tz) == LOCAL_TIMEZONE
assert np.array_equal(
    losfiltroscaguas["utc_hour"].to_numpy(),
    losfiltroscaguas.index.hour
)
assert np.array_equal(
    losfiltroscaguas["pr_hour"].to_numpy(),
    losfiltroscaguas_pr.index.hour
)

print("los filtrosTC/local-time preflight audit passed.")


##Central Tendency and Distribution


In [ ]:
#Residual Chlorine central tendency metrics
print(losfiltroscaguas['totalchlorine'].describe())
mode1 = losfiltroscaguas['totalchlorine'].mode()
print(mode1)
skew1 = losfiltroscaguas['totalchlorine'].skew()
kurt1 = losfiltroscaguas['totalchlorine'].kurt()
print(skew1)
print(kurt1)


In [ ]:
#Turbidity central tendency metrics
print(losfiltroscaguas['turbidity'].describe())
mode3 = losfiltroscaguas['turbidity'].mode()
print(mode3)
skew3 = losfiltroscaguas['turbidity'].skew()
kurt3 = losfiltroscaguas['turbidity'].kurt()
print(skew3)
print(kurt3)


In [ ]:
if "finishwater" in losfiltroscaguas.columns:
    #Flowrate central tendency metrics
    print(losfiltroscaguas['finishwater'].describe())
    mode2 = losfiltroscaguas['finishwater'].mode()
    print(mode2)
    skew2 = losfiltroscaguas['finishwater'].skew()
    kurt2 = losfiltroscaguas['finishwater'].kurt()
    print(skew2)
    print(kurt2)
else:
    print("Flowrate EDA paused until finishwater is available.")


##Univariate Visualizations: Line Graph


In [ ]:
#Residual chlorine
plt.figure(figsize=(15,10))
plt.plot(losfiltroscaguas.index, losfiltroscaguas['totalchlorine'], color='tab:blue')
plt.title('Residual Chlorine (mg/L)')
plt.xlabel('Date (UTC)')
plt.ylabel('Residual Chlorine (mg/L)')
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.show()


In [ ]:
#Turbidity
plt.figure(figsize=(15,10))
plt.plot(losfiltroscaguas.index, losfiltroscaguas['turbidity'], color='tab:blue')
plt.title('Turbidity (NTU)')
plt.xlabel('Date (UTC)')
plt.ylabel('Turbidity (NTU)')
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.show()


In [ ]:
if "finishwater" in losfiltroscaguas.columns:
    #Flowrate
    plt.figure(figsize=(15,10))
    plt.plot(losfiltroscaguas.index, losfiltroscaguas['finishwater'], color='tab:blue')
    plt.title('Flowrate (mgd)')
    plt.xlabel('Date (UTC)')
    plt.ylabel('Flowrate (mgd)')
    plt.xticks(rotation=45)
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("Flowrate EDA paused until finishwater is available.")


##Interactive Plots


In [ ]:
#Function definition
def interactive_timeseries_plot(
    df,
    y_col,
    title,
    yaxis_title,
    xaxis_title="Date (UTC)",
    width=1400,
    height=650,
    font_family="Times New Roman",
    base_font_size=32,
    title_font_size=36,
    axis_title_font_size=32,
    tick_font_size=32,
    legend_font_size=26,
    line_width=3,
    show_rangeslider=True
):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df.index,
        y=df[y_col],
        mode='lines',
        name=yaxis_title,
        line=dict(width=line_width)
    ))

    fig.update_layout(
        width=width,
        height=height,

        # Global font
        font=dict(
            family=font_family,
            size=base_font_size
        ),

        title=dict(
            text=title,
            x=0.5,
            xanchor='center',
            font=dict(size=title_font_size, family=font_family)
        ),

        xaxis_title=dict(
            text=xaxis_title,
            font=dict(size=axis_title_font_size, family=font_family)
        ),
        yaxis_title=dict(
            text=yaxis_title,
            font=dict(size=axis_title_font_size, family=font_family)
        ),

        xaxis=dict(
            tickfont=dict(size=tick_font_size, family=font_family),
            rangeslider=dict(visible=show_rangeslider)
        ),
        yaxis=dict(
            tickfont=dict(size=tick_font_size, family=font_family)
        ),

        legend=dict(
            font=dict(size=legend_font_size, family=font_family)
        ),

        hovermode='x unified',
        template='plotly_white'
    )

    return fig


In [ ]:
#Residual Chlorine
fig = interactive_timeseries_plot(
    df=losfiltroscaguas,
    y_col='totalchlorine',
    title='Residual Chlorine (mg/L)',
    yaxis_title='Residual Chlorine (mg/L)'
)
fig.show()


In [ ]:
#Turbidity
fig = interactive_timeseries_plot(
    df=losfiltroscaguas,
    y_col='turbidity',
    title='Turbidity (NTU)',
    yaxis_title='Turbidity (NTU)'
)
fig.show()


In [ ]:
if "finishwater" in losfiltroscaguas.columns:
    #Flowrate
    fig = interactive_timeseries_plot(
        df=losfiltroscaguas,
        y_col='finishwater',
        title='Flowrate (mgd)',
        yaxis_title='Flowrate (mgd)'
    )
    fig.show()
else:
    print("Flowrate EDA paused until finishwater is available.")


##Monthly Trends


In [ ]:
#Residual Chlorine
monthly_avg = losfiltroscaguas.groupby(['year','month'])['totalchlorine'].mean().reset_index()
monthly_avg['Sample Date'] = pd.to_datetime(monthly_avg[['year','month']].assign(day=1))

plt.figure(figsize=(10,5))
plt.plot(monthly_avg['Sample Date'], monthly_avg['totalchlorine'], marker='o')
plt.title('Monthly Average Residual Chlorine (mg/L)')
plt.ylabel('Average Residual Chlorine (mg/L)')
plt.gca().yaxis.set_label_coords(-0.10, 0.35)
plt.xlabel('Month (Puerto Rico Local Time)')
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.subplots_adjust(left=0.25)
plt.tight_layout()
plt.show()


In [ ]:
#Turbidity
monthly_avg = losfiltroscaguas.groupby(['year','month'])['turbidity'].mean().reset_index()
monthly_avg['Sample Date'] = pd.to_datetime(monthly_avg[['year','month']].assign(day=1))

plt.figure(figsize=(10,5))
plt.plot(monthly_avg['Sample Date'], monthly_avg['turbidity'], marker='o')
plt.title('Monthly Average Turbidity (NTU)')
plt.ylabel('Turbidity (NTU)')
plt.xlabel('Month (Puerto Rico Local Time)')
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.show()


In [ ]:
if "finishwater" in losfiltroscaguas.columns:
    #Flowrate
    monthly_avg = losfiltroscaguas.groupby(['year','month'])['finishwater'].mean().reset_index()
    monthly_avg['Sample Date'] = pd.to_datetime(monthly_avg[['year','month']].assign(day=1))

    plt.figure(figsize=(10,5))
    plt.plot(monthly_avg['Sample Date'], monthly_avg['finishwater'], marker='o')
    plt.title('Monthly Average Flowrate (mgd)')
    plt.ylabel('Flowrate (mgd)')
    plt.xlabel('Month (Puerto Rico Local Time)')
    plt.xticks(rotation=45)
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("Flowrate EDA paused until finishwater is available.")


##Distribution Information-Histograms


In [ ]:
#Residual Chlorine
plt.figure(figsize=(8,5))
sns.histplot(losfiltroscaguas['totalchlorine'], bins=100, kde=True, color='teal')
plt.title('Residual Chlorine (mg/L)')
plt.xlabel('Residual Chlorine (mg/L)')
plt.ylabel('Frequency')
plt.show()


In [ ]:
#Turbidity
plt.figure(figsize=(8,5))
sns.histplot(losfiltroscaguas['turbidity'], bins=100, kde=True, color='teal')
plt.title('Turbidity (NTU)')
plt.xlabel('Turbidity (NTU)')
plt.ylabel('Frequency')
plt.show()


In [ ]:
if "finishwater" in losfiltroscaguas.columns:
    #Flowrate
    plt.figure(figsize=(8,5))
    sns.histplot(losfiltroscaguas['finishwater'], bins=100, kde=True, color='teal')
    plt.title('Flowrate (mgd)')
    plt.xlabel('Flowrate (mgd)')
    plt.ylabel('Frequency')
    plt.show()
else:
    print("Flowrate EDA paused until finishwater is available.")


##Distribution Information-Boxplots


In [ ]:
#Residual Chlorine
plt.figure(figsize=(10,5))
sns.boxplot(x='month', y='totalchlorine', data=losfiltroscaguas)
plt.title('Monthly Variation in Residual Chlorine (mg/L)')
plt.xlabel('Month (Puerto Rico Local Time)')
plt.ylabel('Residual Chlorine (mg/L)')
plt.show()


In [ ]:
#Turbidity
plt.figure(figsize=(10,5))
sns.boxplot(x='month', y='turbidity', data=losfiltroscaguas)
plt.title('Monthly Variation in Turbidity (NTU)')
plt.xlabel('Month (Puerto Rico Local Time)')
plt.ylabel('Turbidity (NTU)')
plt.show()


In [ ]:
#Turbidity (monthly variation supressing outliers)
plt.figure(figsize=(10,5))
sns.boxplot(x='month', y='turbidity', data=losfiltroscaguas, showfliers=False)

plt.title('Monthly Variation in Turbidity (NTU) (Outliers Suppressed)')
plt.xlabel('Month (Puerto Rico Local Time)')
plt.ylabel('Turbidity (NTU)')
plt.show()


In [ ]:
if "finishwater" in losfiltroscaguas.columns:
    #Flowrate
    plt.figure(figsize=(10,5))
    sns.boxplot(x='month', y='finishwater', data=losfiltroscaguas)
    plt.title('Monthly Variation in Flowrate (mgd)')
    plt.xlabel('Month (Puerto Rico Local Time)')
    plt.ylabel('Flowrate (mgd)')
    plt.show()
else:
    print("Flowrate EDA paused until finishwater is available.")


In [ ]:
#Monthly trends during the study period
#Residual Chlorine
losfiltroscaguas['month_year'] = losfiltroscaguas['pr_month_year']

plt.figure(figsize=(14,6))
sns.boxplot(x='month_year', y='totalchlorine', data=losfiltroscaguas)
plt.title('Distribution by Month of Residual Chlorine (mg/L)')
plt.xlabel('Month (Puerto Rico Local Time)')
plt.ylabel('Residual Chlorine (mg/L)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
#Turbidity
losfiltroscaguas['month_year'] = losfiltroscaguas['pr_month_year']

plt.figure(figsize=(14,6))
sns.boxplot(x='month_year', y='turbidity', data=losfiltroscaguas)
plt.title('Distribution by Month of Turbidity (NTU)')
plt.xlabel('Month (Puerto Rico Local Time)')
plt.ylabel('Turbidity (NTU)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
#Hiding outliers on turbidity graph to better observe the patterns in Turbidity data
losfiltroscaguas['month_year'] = losfiltroscaguas['pr_month_year']

plt.figure(figsize=(14,6))
sns.boxplot(x='month_year', y='turbidity', data=losfiltroscaguas, showfliers=False)
plt.title('Distribution by Month (Outliers Hidden) of Turbidity (NTU)')
plt.xlabel('Month (Puerto Rico Local Time)')
plt.ylabel('Turbidity (NTU)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
if "finishwater" in losfiltroscaguas.columns:
    #Flowrate
    losfiltroscaguas['month_year'] = losfiltroscaguas['pr_month_year']

    plt.figure(figsize=(14,6))
    sns.boxplot(x='month_year', y='finishwater', data=losfiltroscaguas)
    plt.title('Distribution by Month of Flowrate (mgd)')
    plt.xlabel('Month (Puerto Rico Local Time)')
    plt.ylabel('Flowrate (mgd)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Flowrate EDA paused until finishwater is available.")


##Calendar Heatmaps


Grouping all months together


In [ ]:
#Definition
def plot_heatmap(df, value_col, title):
    pivot = df.pivot_table(
        values=value_col,
        index="hour",
        columns="month",
        aggfunc='mean'
    )

    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot, cmap='coolwarm', annot=False)
    plt.title(title)
    plt.xlabel('Month (Puerto Rico Local Time)')
    plt.ylabel('Hour of Day (Puerto Rico Local Time)')
    plt.tight_layout()
    plt.show()


#Heatmap plots
plot_heatmap(losfiltroscaguas, 'totalchlorine', 'Residual Chlorine (mg/L) Heatmap')
plot_heatmap(losfiltroscaguas, 'turbidity', 'Turbidity (NTU) Heatmap')
if "finishwater" in losfiltroscaguas.columns:
    plot_heatmap(losfiltroscaguas, 'finishwater', 'Flowrate (mgd) Heatmap')
else:
    print("Flowrate heatmap paused until finishwater is available.")


Visualizing data for each year and each month independently


In [ ]:
def calendar_heatmap(df, value_col, year, aggfunc='mean', cmap='coolwarm', ylabel=None):
    """Calendar-style heatmap for one variable in one year (expects DatetimeIndex)."""

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("calendar_heatmap expects df.index to be a DatetimeIndex. Set df = df.set_index('date') first.")

    # Filter using the Puerto Rico local calendar while preserving the
    # canonical UTC index in the main modeling dataframe.
    local_index = df.index.tz_convert(LOCAL_TIMEZONE)
    year_mask = local_index.year == year
    df_year = df.loc[year_mask, [value_col]].copy()

    if df_year.empty:
        print(f"No data available for {value_col} in {year}")
        return

    # ISO week and weekday are derived from Puerto Rico local time.
    local_year_index = local_index[year_mask]
    iso_week = local_year_index.isocalendar().week.astype(int)
    dow = local_year_index.dayofweek  # Mon=0 ... Sun=6

    pivot = pd.pivot_table(
        df_year,
        values=value_col,
        index=dow,
        columns=iso_week,
        aggfunc=aggfunc
    )

    plt.figure(figsize=(16, 4))
    sns.heatmap(
        pivot,
        cmap=cmap,
        cbar_kws={'label': ylabel if ylabel else value_col}
    )
    plt.yticks(
        np.arange(7) + 0.5,
        ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
        rotation=0
    )
    plt.title(f'Calendar Heatmap of {ylabel if ylabel else value_col} ({year})')
    plt.xlabel('ISO Week of Year (Puerto Rico Local Time)')
    plt.ylabel('Day of Week (Puerto Rico Local Time)')
    plt.tight_layout()
    plt.show()

variables = {
    'totalchlorine': 'Residual Chlorine (mg/L)',
    'turbidity': 'Turbidity (NTU)'
}

if 'finishwater' in losfiltroscaguas.columns:
    variables['finishwater'] = 'Flowrate (mgd)'

for col, label in variables.items():
    for y in sorted(losfiltroscaguas['pr_year'].unique()):
        calendar_heatmap(losfiltroscaguas, col, y, ylabel=label)


##Anomalies and Outliers


In [ ]:
#Residual Chlorine
losfiltroscaguas['z_totalchlorine'] = (
    losfiltroscaguas['totalchlorine'] - losfiltroscaguas['totalchlorine'].mean()
) / losfiltroscaguas['totalchlorine'].std()

threshold = 3

anomalies_rc = losfiltroscaguas[
    np.abs(losfiltroscaguas['z_totalchlorine']) > threshold
]

plt.figure(figsize=(12,6))
plt.plot(losfiltroscaguas.index, losfiltroscaguas['totalchlorine'], label='Residual Chlorine (mg/L)')
plt.scatter(anomalies_rc.index, anomalies_rc['totalchlorine'], color='red', label='Outliers', zorder=5)
plt.title('Outliers/Anomalies in Residual Chlorine (mg/L)')
plt.ylabel('Residual Chlorine (mg/L)')
plt.xlabel('Date (UTC)')
plt.xticks(rotation=45)
plt.legend()
plt.show()


In [ ]:
#Turbidity
losfiltroscaguas['z_turbidity'] = (
    losfiltroscaguas['turbidity'] - losfiltroscaguas['turbidity'].mean()
) / losfiltroscaguas['turbidity'].std()

anomalies_t = losfiltroscaguas[
    np.abs(losfiltroscaguas['z_turbidity']) > threshold
]

plt.figure(figsize=(12,6))
plt.plot(losfiltroscaguas.index, losfiltroscaguas['turbidity'], label='Turbidity (NTU)')
plt.scatter(anomalies_t.index, anomalies_t['turbidity'], color='red', label='Outliers', zorder=5)
plt.title('Outliers/Anomalies in Turbidity (NTU)')
plt.ylabel('Turbidity (NTU)')
plt.xlabel('Date (UTC)')
plt.xticks(rotation=45)
plt.legend()
plt.show()


In [ ]:
if "finishwater" in losfiltroscaguas.columns:
    #Flowrate
    losfiltroscaguas['z_finishwater'] = (
        losfiltroscaguas['finishwater'] - losfiltroscaguas['finishwater'].mean()
    ) / losfiltroscaguas['finishwater'].std()

    anomalies_fr = losfiltroscaguas[
        np.abs(losfiltroscaguas['z_finishwater']) > threshold
    ]

    plt.figure(figsize=(12,6))
    plt.plot(losfiltroscaguas.index, losfiltroscaguas['finishwater'], label='Flowrate (mgd)')
    plt.scatter(anomalies_fr.index, anomalies_fr['finishwater'], color='red', label='Outliers', zorder=5)
    plt.title('Outliers/Anomalies in Flowrate (mgd)')
    plt.ylabel('Flowrate (mgd)')
    plt.xlabel('Date (UTC)')
    plt.legend()
    plt.xticks(rotation=45)
    plt.show()
else:
    print("Flowrate EDA paused until finishwater is available.")


In [ ]:
#Counting the amount of outliers per month
def flag_iqr_anomalies(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)


losfiltroscaguas['chlorine_anomaly'] = flag_iqr_anomalies(
    losfiltroscaguas['totalchlorine']
)
losfiltroscaguas['turbidity_anomaly'] = flag_iqr_anomalies(
    losfiltroscaguas['turbidity']
)

losfiltroscaguas_pr['chlorine_anomaly'] = losfiltroscaguas['chlorine_anomaly'].to_numpy()
losfiltroscaguas_pr['turbidity_anomaly'] = losfiltroscaguas['turbidity_anomaly'].to_numpy()
chlorine_monthly = losfiltroscaguas_pr[
    losfiltroscaguas_pr['chlorine_anomaly']
].resample('ME').size()
turbidity_monthly = losfiltroscaguas_pr[
    losfiltroscaguas_pr['turbidity_anomaly']
].resample('ME').size()

print('Chlorine anomalies per month:')
print(chlorine_monthly)
print('\nTurbidity anomalies per month:')
print(turbidity_monthly)

if "finishwater" in losfiltroscaguas.columns:
    losfiltroscaguas['flowrate_anomaly'] = flag_iqr_anomalies(
        losfiltroscaguas['finishwater']
    )
    losfiltroscaguas_pr['flowrate_anomaly'] = losfiltroscaguas['flowrate_anomaly'].to_numpy()
    flowrate_monthly = losfiltroscaguas_pr[
        losfiltroscaguas_pr['flowrate_anomaly']
    ].resample('ME').size()
    print('\nFlowrate anomalies per month:')
    print(flowrate_monthly)
else:
    flowrate_monthly = None
    print("Flowrate anomaly analysis paused until finishwater is available.")


Plot of Outliers per Month


In [ ]:
#Plot of outliers per month
def plot_monthly_anomalies(
    monthly_series,
    title,
    ylabel='Number of Outliers'
):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(monthly_series.index, monthly_series.values, width=20)
    ax.set_title(title)
    ax.set_xlabel('Month (Puerto Rico Local Time)')
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=60)
    plt.tight_layout()
    plt.show()


plot_monthly_anomalies(
    chlorine_monthly,
    'Monthly Residual Chlorine (mg/L) Outliers'
)
plot_monthly_anomalies(
    turbidity_monthly,
    'Monthly Turbidity (NTU) Outliers'
)
if flowrate_monthly is not None:
    plot_monthly_anomalies(
        flowrate_monthly,
        'Monthly Flowrate (mgd) Outliers'
    )


##Residual Chlorine Anomalies According to Upper Threshold


In [ ]:
#Residual Chlorine
lower_limit_rc = 0.2
upper_limit_rc = 4.0

#Flag out of range residual chlorine levels
losfiltroscaguas['chlorine_out_of_range'] = (
    (losfiltroscaguas['totalchlorine'] < lower_limit_rc) |
    (losfiltroscaguas['totalchlorine'] > upper_limit_rc)
)

chlor_num_anomalies = int(losfiltroscaguas['chlorine_out_of_range'].sum())

plt.figure(figsize=(12, 6))

#Line plot
plt.plot(
    losfiltroscaguas.index,
    losfiltroscaguas['totalchlorine'],
    color='tab:blue',
    label='Residual Chlorine (mg/L)'
)

#Scatter of flagged points
outliers = losfiltroscaguas[losfiltroscaguas['chlorine_out_of_range']]

plt.scatter(
    outliers.index,
    outliers['totalchlorine'],
    color='red',
    label='Out of Range',
    zorder=5
)

#Including threshold lines
plt.axhline(lower_limit_rc, color='blue', linestyle='--', label='Lower Limit (0.2 mg/L)')
plt.axhline(upper_limit_rc, color='orange', linestyle='--', label='Upper Limit (4.0 mg/L)')

plt.title('Residual Chlorine Concentrations (mg/L) Outside of Thresholds')
plt.xlabel('Date (UTC)')
plt.xticks(rotation=45)
plt.ylabel('Residual Chlorine (mg/L)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(chlor_num_anomalies)


In [ ]:
#Turbidity
lower_limit_t = 0.0
upper_limit_t = 1.0

#Flag out of range residual chlorine levels
losfiltroscaguas["turbidity_out_of_range"] = (
    (losfiltroscaguas["turbidity"] < lower_limit_t) |
    (losfiltroscaguas["turbidity"] > upper_limit_t)
)

turb_num_anomalies = int(losfiltroscaguas["turbidity_out_of_range"].sum())

plt.figure(figsize=(12, 6))

#Line plot
plt.plot(
    losfiltroscaguas.index,
    losfiltroscaguas["turbidity"],
    color="tab:blue",
    label="Turbidity (NTU)"
)

#Scatter of flagged points
outliers = losfiltroscaguas[losfiltroscaguas["turbidity_out_of_range"]]

plt.scatter(
    outliers.index,
    outliers["turbidity"],
    color="red",
    label="Out of Range",
    zorder=5
)

#Including threshold lines
plt.axhline(lower_limit_t, color="blue", linestyle="--", label="Lower Limit (0.0 NTU)")
plt.axhline(upper_limit_t, color="orange", linestyle="--", label="Upper Limit (1.0 NTU)")

plt.title("Turbidity (NTU) Outside of Regulatory Thresholds")
plt.xlabel("Date (UTC)")
plt.xticks(rotation=45)
plt.ylabel("Turbidity (NTU)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(turb_num_anomalies)


In [ ]:
# Count lower- and upper-threshold chlorine observations by Puerto Rico month.
chlorine_below_lower_limit = losfiltroscaguas[
    losfiltroscaguas["totalchlorine"] < lower_limit_rc
]
chlorine_above_upper_limit = losfiltroscaguas[
    losfiltroscaguas["totalchlorine"] > upper_limit_rc
]

chlorine_below_lower_limit_pr = losfiltroscaguas_pr.loc[
    losfiltroscaguas["totalchlorine"].to_numpy() < lower_limit_rc
]
chlorine_above_upper_limit_pr = losfiltroscaguas_pr.loc[
    losfiltroscaguas["totalchlorine"].to_numpy() > upper_limit_rc
]

lower_monthly_counts = (
    chlorine_below_lower_limit_pr
    .resample("ME")
    .size()
)
upper_monthly_counts = (
    chlorine_above_upper_limit_pr
    .resample("ME")
    .size()
)

print("Residual chlorine values below 0.2 mg/L per Puerto Rico month:")
print(lower_monthly_counts)

print("\nResidual chlorine values above 4.0 mg/L per Puerto Rico month:")
print(upper_monthly_counts)


In [ ]:
# Plot monthly counts for lower- and upper-threshold chlorine observations.
def plot_monthly_threshold_counts(
    monthly_counts,
    title,
    output_name=None
):
    monthly_counts = monthly_counts.copy()
    fig, ax = plt.subplots(figsize=(14, 6))

    bars = ax.bar(
        monthly_counts.index,
        monthly_counts.values,
        width=20
    )

    maximum_count = int(monthly_counts.max()) if len(monthly_counts) else 0
    label_offset = max(0.2, maximum_count * 0.02)

    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + label_offset,
            str(int(height)),
            ha="center",
            va="bottom"
        )

    ax.set_ylim(0, max(1, maximum_count * 1.20))
    ax.set_title(title)
    ax.set_xlabel("Month (Puerto Rico Local Time)")
    ax.set_ylabel("Number of Observations")
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.tick_params(axis="x", rotation=45)

    fig.tight_layout()

    if output_name is not None:
        fig.savefig(
            os.path.join(EXTERNAL_OUTPUT, output_name),
            dpi=300,
            bbox_inches="tight"
        )

    plt.show()
    plt.close(fig)


plot_monthly_threshold_counts(
    lower_monthly_counts,
    "Monthly Count of Residual Chlorine Values Below 0.2 mg/L",
    "monthly_chlorine_below_0_2_mgL.png"
)

plot_monthly_threshold_counts(
    upper_monthly_counts,
    "Monthly Count of Residual Chlorine Values Above 4.0 mg/L",
    "monthly_chlorine_above_4_0_mgL.png"
)


##Rolling Averages


In [ ]:
#Residual Chlorine andT Turbidity
losfiltroscaguas['totalchlorine'].rolling(window=24).mean().plot(figsize=(12,6), legend=True)
losfiltroscaguas['turbidity'].rolling(window=24).mean().plot(secondary_y=True, figsize=(12,6), legend=True)
plt.title('Rolling Average of Residual Chlorine (mg/L) & Turbidity (NTU)')
plt.xlabel('Date (UTC)')
plt.show()


In [ ]:
if "finishwater" in losfiltroscaguas.columns:
    #Flowrate
    losfiltroscaguas['finishwater'].rolling(window=24).mean().plot(figsize=(12,6))
    plt.title('Rolling Average of Flowrate (mgd)')
    plt.xlabel('Date (UTC)')
    plt.ylabel('Flowrate (mgd)')
else:
    print("Flowrate EDA paused until finishwater is available.")


##Percent Changes


In [ ]:
#Residual chlorine
losfiltroscaguas['chlorine_pct_change'] = (losfiltroscaguas['totalchlorine'].pct_change()* 100)

plt.figure(figsize=(12,6))

plt.plot(
    losfiltroscaguas.index,
    losfiltroscaguas['chlorine_pct_change'],
    color='tab:blue'
)
plt.title('Percent Change in Residual Chlorine (mg/L)')
plt.xlabel('Date (UTC)')
plt.ylabel('Percentage Change (%)')


In [ ]:
#Turbidity
losfiltroscaguas['turbidity_pct_change'] = (losfiltroscaguas['turbidity'].pct_change()* 100)

plt.figure(figsize=(12,6))

plt.plot(
    losfiltroscaguas.index,
    losfiltroscaguas['turbidity_pct_change'],
    color='tab:blue'
)
plt.title('Percent Change in Turbidity (NTU)')
plt.xlabel('Date (UTC)')
plt.ylabel('Percentage Change (%)')


In [ ]:
if "finishwater" in losfiltroscaguas.columns:
    #Flowrate
    losfiltroscaguas['flowrate_pct_change'] = (losfiltroscaguas['finishwater'].pct_change()* 100)

    plt.figure(figsize=(12,6))

    plt.plot(
        losfiltroscaguas.index,
        losfiltroscaguas['flowrate_pct_change'],
        color='tab:blue'
    )
    plt.title('Percent Change in Flowrate (mgd)')
    plt.xlabel('Date (UTC)')
    plt.ylabel('Percentage Change (%)')
else:
    print("Flowrate EDA paused until finishwater is available.")


In [ ]:
#Day to day Residual Chlorine percentage change
daily = losfiltroscaguas_pr['totalchlorine'].resample('D').mean()
daily_pct_change = daily.pct_change()*100

plt.figure(figsize=(12,6))
plt.plot(daily.index, daily_pct_change)
plt.title('Daily Percentage Change in Residual Chlorine (mg/L)')
plt.xlabel('Date (Puerto Rico Local Time)')
plt.ylabel('Percentage Change (%)')
plt.grid(alpha=0.3)
plt.show()


In [ ]:
#Day to day Turbidity percentage change
daily = losfiltroscaguas_pr['turbidity'].resample('D').mean()
daily_pct_change = daily.pct_change()*100

plt.figure(figsize=(12,6))
plt.plot(daily.index, daily_pct_change)
plt.title('Daily Percentage Change in Turbidity (NTU)')
plt.xlabel('Date (Puerto Rico Local Time)')
plt.ylabel('Percentage Change (%)')
plt.grid(alpha=0.3)
plt.show()


In [ ]:
if "finishwater" in losfiltroscaguas.columns:
    #Day to day Flowrate percentage change
    daily = losfiltroscaguas_pr['finishwater'].resample('D').mean()
    daily_pct_change = daily.pct_change()*100

    plt.figure(figsize=(12,6))
    plt.plot(daily.index, daily_pct_change)
    plt.title('Daily Percentage Change in Flowrate (mgd)')
    plt.xlabel('Date (Puerto Rico Local Time)')
    plt.ylabel('Percentage Change (%)')
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("Flowrate EDA paused until finishwater is available.")


##Correlations


In [ ]:
#Spearman correlation
correlation_columns = ['totalchlorine', 'turbidity']
if 'finishwater' in losfiltroscaguas.columns:
    correlation_columns.append('finishwater')

spearman_corr = losfiltroscaguas[
    correlation_columns
].corr(method='spearman')

print(spearman_corr)

plt.figure(figsize=(6,5))
sns.heatmap(
    spearman_corr,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f'
)
plt.title('Spearman Correlation Matrix')
plt.tight_layout()
plt.show()


In [ ]:
#Cross-correlation for time dependency

#Lagged correlation
def plot_cross_correlations(
    df,
    col_pairs,
    max_lag=168,
    figsize=(12, 5)
):

    for x_col, y_col in col_pairs:
        lags = list(range(0, max_lag + 1))
        corrs = []

        for lag in lags:
            x = df[x_col]
            y = df[y_col].shift(-lag)   # y at future time

            valid = pd.concat([x, y], axis=1).dropna()
            if len(valid) > 1:
                corr = valid.iloc[:, 0].corr(valid.iloc[:, 1])
            else:
                corr = np.nan

            corrs.append(corr)

        # Plot
        plt.figure(figsize=figsize)
        plt.plot(lags, corrs, marker='o', markersize=3)
        plt.axhline(0, linestyle='--')
        plt.title(f'Cross-Correlation: {x_col} (t) vs {y_col} (t + lag)')
        plt.xlabel('Lag')
        plt.ylabel('Correlation')
        plt.grid(alpha=0.3)
        plt.show()

        # Report strongest lag
        corrs_arr = np.array(corrs, dtype=float)
        if np.isfinite(corrs_arr).any():
            best_idx = np.nanargmax(np.abs(corrs_arr))
            print(
                f'{x_col} -> {y_col}: strongest correlation at lag {lags[best_idx]} '
                f'= {corrs_arr[best_idx]:.4f}'
            )
            print('-' * 60)

pairs = [('turbidity', 'totalchlorine')]

if 'finishwater' in losfiltroscaguas.columns:
    pairs.extend([
        ('turbidity', 'finishwater'),
        ('finishwater', 'turbidity')
    ])

plot_cross_correlations(losfiltroscaguas, pairs, max_lag=168)


##Seasonal Observations


In [ ]:
#Residual Chlorine Monthly Seasonality
monthly = losfiltroscaguas_pr['totalchlorine'].resample('ME').mean()
fig, ax = plt.subplots(figsize=(10, 6))

month_plot(monthly, ax=ax)

ax.set_title('Monthly Seasonality of Residual Chlorine (mg/L)')
ax.set_xlabel('Month')
ax.set_ylabel('Residual Chlorine (mg/L)')

#Residual Chlorine Quarterly Seasonality
quarterly = losfiltroscaguas_pr['totalchlorine'].resample('QE').mean()
fig, ax = plt.subplots(figsize=(10, 6))
quarter_plot(quarterly, ax=ax)

ax.set_title('Quarterly Seasonality of Residual Chlorine (mg/L)')
ax.set_xlabel('Quarter')
ax.set_ylabel('Residual Chlorine (mg/L)')

plt.tight_layout()
plt.show()


In [ ]:
#Turbidity Monthly Seasonality
monthly_turbidity = losfiltroscaguas_pr['turbidity'].resample('ME').mean()

fig, ax = plt.subplots(figsize=(10, 6))
month_plot(monthly_turbidity, ax=ax)

ax.set_title('Monthly Seasonality of Turbidity (NTU)')
ax.set_xlabel('Month')
ax.set_ylabel('Turbidity (NTU)')

plt.tight_layout()
plt.show()

#Turbidity Quarterly Seasonality
quarterly = losfiltroscaguas_pr['turbidity'].resample('QE').mean()
fig, ax = plt.subplots(figsize=(10, 6))
quarter_plot(quarterly, ax=ax)

ax.set_title('Quarterly Seasonality of Turbidity (NTU)')
ax.set_xlabel('Quarter')
ax.set_ylabel('Turbidity (NTU)')

plt.tight_layout()
plt.show()


In [ ]:
if "finishwater" in losfiltroscaguas.columns:
    #Flowrate Monthly Seasonality
    monthly_finishwater = losfiltroscaguas_pr['finishwater'].resample('ME').mean()

    fig, ax = plt.subplots(figsize=(10, 6))
    month_plot(monthly_finishwater, ax=ax)

    ax.set_title('Monthly Seasonality of Flowrate (mgd)')
    ax.set_xlabel('Month')
    ax.set_ylabel('Flowrate (mgd)')

    plt.tight_layout()
    plt.show()

    #Flowrate Quarterly Seasonality
    quarterly_finishwater = losfiltroscaguas_pr['finishwater'].resample('QE').mean()

    fig, ax = plt.subplots(figsize=(10, 6))
    quarter_plot(quarterly_finishwater, ax=ax)

    ax.set_title('Quarterly Seasonality of Flowrate (mgd)')
    ax.set_xlabel('Quarter')
    ax.set_ylabel('Flowrate (mgd)')

    plt.tight_layout()
    plt.show()
else:
    print("Flowrate EDA paused until finishwater is available.")


##Seasonal Decomposition


In [ ]:
#Residual Chlorine
decomposition_chlorine = seasonal_decompose(losfiltroscaguas['totalchlorine'], model='additive', period=2600)
fig = decomposition_chlorine.plot()
fig.suptitle('Seasonal Decomposition of Residual Chlorine (mg/L)', fontsize=60)
fig.set_size_inches(35, 18)
plt.show()


#Periods to check
#daily: 24 hours
#weekly: 168 hours
#yearly: 8760 hours


In [ ]:
#Turbidity
decomposition_turbidity = seasonal_decompose(losfiltroscaguas['turbidity'], model='additive', period=2600)
fig = decomposition_turbidity.plot()
fig.suptitle('Seasonal Decomposition of Turbidity (NTU)', fontsize=60)
fig.set_size_inches(25, 18)
plt.show()


In [ ]:
if "finishwater" in losfiltroscaguas.columns:
    #Flowrate
    decomposition_flowrate = seasonal_decompose(losfiltroscaguas['finishwater'], model='additive', period=2600)
    fig = decomposition_flowrate.plot()
    fig.suptitle('Seasonal Decomposition of Flowrate (mgd)', fontsize=60)
    fig.set_size_inches(25, 18)
    plt.show()
else:
    print("Flowrate EDA paused until finishwater is available.")


##Autocorrelation and Partial Autocorrelation


In [ ]:
#Plot the autocorrelation function (ACF) for Residual Chlorine
plot_acf(losfiltroscaguas['totalchlorine'], lags=500)  #lags 48; 72; 500; 26,000; 6,000; 6,200 (still outside confidence interval, so significant)
plt.title('Residual Chlorine (mg/L) Autocorrelation')
plt.show()


#Periodic trends observed, and values above 0 in close range (e.g., 24 hours)

#Partial autocorrelation
plot_pacf(losfiltroscaguas['totalchlorine'], lags=200)
plt.title('Residual Chlorine (mg/L) Partial Autocorrelation')
plt.show()


In [ ]:
#Plot the autocorrelation function (ACF) for Turdbidity
plot_acf(losfiltroscaguas['turbidity'], lags=162)
plt.title('Turbidity (NTU) Autocorrelation')
plt.show()

#Declining trend observed; driven by short-time disturbances; not seasonal in nature

#Partial autocorrelation
plot_pacf(losfiltroscaguas['turbidity'], lags=162)
plt.title('Turbidity (NTU) Partial Autocorrelation')
plt.show()


In [ ]:
if "finishwater" in losfiltroscaguas.columns:
    #Plot the autocorrelation function (ACF) for Flowrate
    plot_acf(losfiltroscaguas['finishwater'], lags=48)
    plt.title('Flowrate (mgd) Autocorrelation')
    plt.show()

    #Periodic trends observed but not as strong as with turbidity, and values above 0

    #Partial autocorrelation
    plot_pacf(losfiltroscaguas['finishwater'], lags=48)
    plt.title('Flowrate (mgd) Partial Autocorrelation')
    plt.show()
else:
    print("Flowrate EDA paused until finishwater is available.")


#Sample Size Estimation Based on Wilkis (2011)


In [ ]:
#Wilkis (2011) - Sample Size Estimation (Helps to observe how much independent information can be obtained from the data)
y = losfiltroscaguas['totalchlorine'].asfreq('h')

n = len(y)
r1 = y.autocorr(lag=1)
n_eff = n * (1 - r1) / (1 + r1)

print(f"Total observations (n): {n}")
print(f"Lag-1 autocorrelation (r1): {r1:.4f}")
print(f"Effective sample size (n'): {n_eff:.2f}")


# Dickey-Fuller Test to Check if Time Series is Stationary


In [ ]:
result_dfuller=adfuller(y)
print('p-value: %f' % result_dfuller[1])

if result_dfuller[1] > 0.05:
  print('The time series is not stationary')
else:
  print('The time series is stationary')

#Null hypothesis: time series is not stationary; alternative: time series is stationary
#ARIMA(p,0,q)


#Mann-Kendall Test to Check for Monotonic Upward or Downward Trend


In [ ]:
result_mk = mk.seasonal_test(y,
    period=24
)

print(result_mk)

if result_mk.h:  #statistically significant
    if result_mk.trend == 'increasing':
        print(f"Significant increasing trend (p={result_mk.p:.4f})")
    elif result_mk.trend == 'decreasing':
        print(f"Significant decreasing trend (p={result_mk.p:.4f})")
    else:
        print(f"Significant trend detected but direction unclear (p={result_mk.p:.4f})")
else:
    print(f"No statistically significant trend (p={result_mk.p:.4f})")
#Significant increasing trend observed


# Seasonal Differencing to Confirm that Seasonality is Strong in Residual Chlorine


In [ ]:
#Checking seasonal differencing (a lot changes, so D=1)
seasonal_diff = y.diff(24)

#ACF plot on differenced series
plot_acf(seasonal_diff.dropna(), lags=50)
plt.title('ACF After Seasonal Differencing')
plt.show()
#This confirms that the seasonal component is very present

#PACF on differenced series
plot_pacf(seasonal_diff.dropna(), lags=50)
plt.title('PACF After Seasonal Differencing')
plt.show()



# Baseline Statistical Models

These models are fitted from scratch using los filtros data. They reproduce the Alexander Orr baseline framework, including time-series cross-validation, an untouched fixed holdout, forecast intervals, metrics, plots and residual diagnostics.


## Baseline supporting functions


In [ ]:
#General helpers
def ensure_output_dir(output_dir):
    os.makedirs(output_dir, exist_ok=True)


def clean_memory():
    gc.collect()


def ensure_same_tz(ts, tz):
    ts = pd.Timestamp(ts)

    if tz is None:
        if ts.tzinfo is not None:
            return ts.tz_convert(None)
        return ts

    if ts.tzinfo is None:
        return ts.tz_localize(tz)

    return ts.tz_convert(tz)


def season_name(ts):
    m = pd.Timestamp(ts).month
    if m in [3, 4, 5]:
        return "Spring"
    elif m in [6, 7, 8]:
        return "Summer"
    elif m in [9, 10, 11]:
        return "Autumn"
    else:
        return "Winter"


def evaluate_predictions(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    epsilon = 1e-6
    mape = np.mean(
        np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), epsilon))
    ) * 100

    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
        "MAPE": float(mape)
    }

#added 18 May
def evaluate_predictions_by_conversion(
    y_true,
    y_pred,
    conversion_flag=None
):

    return {}

def stable_run_id(row_dict: Dict) -> str:
    keys = [
        "model_family",
        "eval_month_start",
        "season",
        "train_months",
        "hidden_units",
        "batch_size",
        "num_heads"
    ]
    payload = {k: str(row_dict.get(k, "")) for k in keys}
    text = json.dumps(payload, sort_keys=True)
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:12]


def add_residual_based_intervals(pred_df, actual_col, pred_col, alpha=0.05):
    pred_df = pred_df.copy()
    residuals = pred_df[actual_col] - pred_df[pred_col]
    sigma = residuals.std(ddof=1)
    z = norm.ppf(1 - alpha / 2)

    pred_df["lower_ci"] = pred_df[pred_col] - z * sigma
    pred_df["upper_ci"] = pred_df[pred_col] + z * sigma
    pred_df["sigma"] = sigma

    return pred_df


def add_sarimax_intervals(pred_df, conf_int):
    pred_df = pred_df.copy()
    pred_df["lower_ci"] = conf_int.iloc[:, 0].values
    pred_df["upper_ci"] = conf_int.iloc[:, 1].values
    pred_df["sigma"] = (pred_df["upper_ci"] - pred_df["lower_ci"]) / (2 * 1.96)
    return pred_df


def print_table(title, obj):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    print(obj)


In [ ]:
#Plotting and residual diagnostics
def plot_forecast_vs_actual_plain(
    pred_df,
    title,
    actual_col="actual",
    pred_col="forecast",
    ylabel="Residual Chlorine (mg/L)",
    save_path=None,
    show_plot=True
):
    fig, ax = plt.subplots(figsize=(14, 6))

    ax.plot(pred_df.index, pred_df[actual_col], label="Actual")
    ax.plot(pred_df.index, pred_df[pred_col], label="Forecast")

    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel(ylabel)
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=600)

    if show_plot:
        plt.show()

    plt.close(fig)


def plot_forecast_vs_actual_interval(
    pred_df,
    title,
    actual_col="actual",
    pred_col="forecast",
    lower_col="lower_ci",
    upper_col="upper_ci",
    ylabel="Residual Chlorine (mg/L)",
    save_path=None,
    show_plot=True
):
    fig, ax = plt.subplots(figsize=(14, 6))

    ax.plot(pred_df.index, pred_df[actual_col], label="Actual")
    ax.plot(pred_df.index, pred_df[pred_col], label="Forecast")

    if lower_col in pred_df.columns and upper_col in pred_df.columns:
        ax.fill_between(
            pred_df.index,
            pred_df[lower_col],
            pred_df[upper_col],
            alpha=0.20,
            label="Forecast interval"
        )

    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel(ylabel)
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=600)

    if show_plot:
        plt.show()

    plt.close(fig)


# def save_forecast_plots(
#     pred_df,
#     output_dir,
#     file_stem,
#     model_label,
#     actual_col="actual",
#     pred_col="forecast",
#     show_plot=True
# ):
#     ensure_output_dir(output_dir)

#     plot_forecast_vs_actual_plain(
#         pred_df=pred_df,
#         title=f"{model_label}: Forecast vs Actual",
#         actual_col=actual_col,
#         pred_col=pred_col,
#         save_path=f"{output_dir}/{file_stem}_forecast_vs_actual_plain.png",
#         show_plot=show_plot
#     )

#     plot_forecast_vs_actual_interval(
#         pred_df=pred_df,
#         title=f"{model_label}: Forecast vs Actual with Forecast Interval",
#         actual_col=actual_col,
#         pred_col=pred_col,
#         save_path=f"{output_dir}/{file_stem}_forecast_vs_actual_interval.png",
#         show_plot=show_plot
#     )


# # If older code calls plot_forecast_vs_actual(), this now creates BOTH plots.
# def plot_forecast_vs_actual(
#     pred_df,
#     title,
#     actual_col="actual",
#     pred_col="forecast",
#     lower_col="lower_ci",
#     upper_col="upper_ci",
#     ylabel="Residual Chlorine (mg/L)",
#     save_path=None,
#     show_plot=True
# ):
#     if save_path is not None:
#         save_path = str(save_path)

#         if save_path.endswith(".png"):
#             plain_path = save_path.replace(".png", "_plain.png")
#             interval_path = save_path.replace(".png", "_interval.png")
#         else:
#             plain_path = save_path + "_plain.png"
#             interval_path = save_path + "_interval.png"
#     else:
#         plain_path = None
#         interval_path = None

#     plot_forecast_vs_actual_plain(
#         pred_df=pred_df,
#         title=title,
#         actual_col=actual_col,
#         pred_col=pred_col,
#         ylabel=ylabel,
#         save_path=plain_path,
#         show_plot=show_plot
#     )

#     plot_forecast_vs_actual_interval(
#         pred_df=pred_df,
#         title=f"{title} with Forecast Interval",
#         actual_col=actual_col,
#         pred_col=pred_col,
#         lower_col=lower_col,
#         upper_col=upper_col,
#         ylabel=ylabel,
#         save_path=interval_path,
#         show_plot=show_plot
#     )

# Save plain and interval forecast plots using the Alexander Orr design.
def save_forecast_plots(
    pred_df,
    output_dir,
    file_stem,
    model_label,
    actual_col="actual",
    pred_col="forecast",
    ylabel="Residual Chlorine (mg/L)",
    show_plot=True
):
    ensure_output_dir(output_dir)

    plot_forecast_vs_actual_plain(
        pred_df=pred_df,
        title=f"{model_label}: Forecast vs Actual",
        actual_col=actual_col,
        pred_col=pred_col,
        ylabel=ylabel,
        save_path=(
            f"{output_dir}/"
            f"{file_stem}_forecast_vs_actual_plain.png"
        ),
        show_plot=show_plot
    )

    plot_forecast_vs_actual_interval(
        pred_df=pred_df,
        title=(
            f"{model_label}: "
            "Forecast vs Actual with Forecast Interval"
        ),
        actual_col=actual_col,
        pred_col=pred_col,
        ylabel=ylabel,
        save_path=(
            f"{output_dir}/"
            f"{file_stem}_forecast_vs_actual_interval.png"
        ),
        show_plot=show_plot
    )

def save_residual_outputs(pred_df, actual_col, pred_col, output_dir, file_stem, lags=72, show_plots=True):
    ensure_output_dir(output_dir)

    residual_df = pred_df.copy()
    residual_df["residual"] = residual_df[actual_col] - residual_df[pred_col]
    residual_df.to_csv(f"{output_dir}/{file_stem}_residuals.csv")

    residuals = residual_df["residual"].dropna()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(residual_df.index, residual_df["residual"])
    ax.axhline(0, linestyle="--")
    ax.set_title("Residuals over time")
    ax.set_xlabel("Date")
    ax.set_ylabel("Residual")
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(f"{output_dir}/{file_stem}_residuals_timeseries.png", dpi=1200)
    if show_plots:
        plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(residuals, bins=40)
    ax.set_title("Residual distribution")
    ax.set_xlabel("Residual")
    ax.set_ylabel("Frequency")
    fig.tight_layout()
    fig.savefig(f"{output_dir}/{file_stem}_residuals_histogram.png", dpi=1200)
    if show_plots:
        plt.show()
    plt.close(fig)

    fig = qqplot(residuals, line="s")
    fig.axes[0].set_title("Residual Q-Q plot")
    fig.tight_layout()
    fig.savefig(f"{output_dir}/{file_stem}_residuals_qqplot.png", dpi=1200)
    if show_plots:
        plt.show()
    plt.close(fig)

    if len(residuals) > 2:
        fig, ax = plt.subplots(figsize=(12, 5))
        plot_acf(residuals, lags=min(lags, max(1, len(residuals) - 1)), ax=ax)
        ax.set_title("Residual ACF")
        fig.tight_layout()
        fig.savefig(f"{output_dir}/{file_stem}_residuals_acf.png", dpi=1200)
        if show_plots:
            plt.show()
        plt.close(fig)


In [ ]:
#Splits
def make_date_holdout_split(df, target_col, holdout_start):
    df_model = df.copy()
    df_model[target_col] = pd.to_numeric(df_model[target_col], errors="coerce")
    df_model = df_model.dropna(subset=[target_col])

    holdout_start = ensure_same_tz(holdout_start, df_model.index.tz)

    train = df_model.loc[df_model.index < holdout_start, target_col].copy()
    test = df_model.loc[df_model.index >= holdout_start, target_col].copy()

    return train, test


def split_by_time(
    df,
    train_months,
    eval_start_month,
    val_months=1,
    test_months=1
):
    tz = df.index.tz

    train_end = ensure_same_tz(eval_start_month, tz)
    train_start = train_end - pd.DateOffset(months=train_months)

    val_start = train_end
    requested_val_end = (
        val_start
        + pd.DateOffset(months=val_months)
    )

    val_end = min(
        requested_val_end,
        df.index.max()
    )

    test_start = requested_val_end
    requested_test_end = (
        test_start
        + pd.DateOffset(months=test_months)
    )

    train_df = df[
        (df.index >= train_start)
        & (df.index < train_end)
    ].copy()

    val_df = df[
        (df.index >= val_start)
        & (df.index < requested_val_end)
    ].copy()

    if requested_test_end <= df.index.max():

        # Complete holdout month:
        # include test_start but exclude the next month's boundary.
        test_end = requested_test_end

        test_df = df[
            (df.index >= test_start)
            & (df.index < test_end)
        ].copy()

    else:

        # Partial holdout month:
        # keep every observation available through the dataset's final timestamp.
        test_end = df.index.max()

        test_df = df[
            (df.index >= test_start)
            & (df.index <= test_end)
        ].copy()

        print(
            f'Warning: partial test period. '
            f'Requested end: {requested_test_end}. '
            f'Available data end: {test_end}.'
        )

    return (
        train_df,
        val_df,
        test_df,
        train_start,
        train_end,
        val_start,
        val_end,
        test_start,
        test_end
    )

In [ ]:
#Baseline config
@dataclass
class BaselineConfig:
    model_family: str
    target_col: str
    output_dir: str
    holdout_start: str = "2026-01-01"

    exog_cols: Optional[List[str]] = None

    candidate_models: Optional[
        List[Tuple[Tuple[int, int, int], Tuple[int, int, int, int]]]
    ] = None

    smoothing_models: Tuple[str, ...] = ("ses", "des", "holt_winters")

    n_splits: int = 5
    cv_test_size: int = 720
    maxiter: int = 500
    alpha: float = 0.05
    show_plots: bool = True


In [ ]:
#Exponential smoothing fixed holdout
def fit_smoothing_model(train, model_name):
    if model_name == "ses":
        return SimpleExpSmoothing(
            train,
            initialization_method="estimated"
        ).fit(optimized=True)

    if model_name == "des":
        return ExponentialSmoothing(
            train,
            trend="add",
            seasonal=None,
            initialization_method="estimated"
        ).fit(optimized=True)

    if model_name == "holt_winters":
        return ExponentialSmoothing(
            train,
            trend="add",
            seasonal="add",
            seasonal_periods=24,
            initialization_method="estimated"
        ).fit(optimized=True)

    raise ValueError("model_name must be ses, des, or holt_winters")


def run_smoothing_fixed_holdout(df, cfg: BaselineConfig):
    ensure_output_dir(cfg.output_dir)

    train, test = make_date_holdout_split(
        df=df,
        target_col=cfg.target_col,
        holdout_start=cfg.holdout_start
    )

    results = []
    forecasts = {}

    for model_name in cfg.smoothing_models:
        model = fit_smoothing_model(train, model_name)

        forecast = model.forecast(len(test))
        forecast.index = test.index

        pred_df = pd.DataFrame({
            "actual": test,
            "forecast": forecast
        }, index=test.index)

        pred_df = add_residual_based_intervals(
            pred_df,
            actual_col="actual",
            pred_col="forecast",
            alpha=cfg.alpha
        )

        metrics = evaluate_predictions(pred_df["actual"], pred_df["forecast"])

        row = {
            "model_family": model_name,
            "holdout_start": cfg.holdout_start,
            **metrics
        }

        results.append(row)
        forecasts[model_name] = pred_df

        file_stem = f"{model_name}_fixed_holdout"

        pred_df.to_csv(f"{cfg.output_dir}/{file_stem}_forecast.csv")
        pd.DataFrame([metrics]).to_csv(f"{cfg.output_dir}/{file_stem}_metrics.csv", index=False)

        save_forecast_plots(
            pred_df=pred_df,
            output_dir=cfg.output_dir,
            file_stem=file_stem,
            model_label=model_name.upper(),
            show_plot=cfg.show_plots
        )

        save_residual_outputs(
            pred_df=pred_df,
            actual_col="actual",
            pred_col="forecast",
            output_dir=cfg.output_dir,
            file_stem=file_stem,
            show_plots=cfg.show_plots
        )

        del model
        clean_memory()

    results_df = pd.DataFrame(results)
    results_df.to_csv(f"{cfg.output_dir}/smoothing_fixed_holdout_metrics.csv", index=False)

    print_table("Classical Exponential Smoothing Results", results_df)

    return results_df, forecasts

In [ ]:
#ARIMA/SARIMA Holdouts
ARIMA_CANDIDATE_MODELS = [
    ((1, 0, 1), (0, 0, 0, 0)),
    ((2, 0, 1), (0, 0, 0, 0)),
    ((1, 1, 1), (0, 0, 0, 0)),
    ((2, 1, 1), (0, 0, 0, 0))
]

SARIMA_CANDIDATE_MODELS = [
    ((1, 0, 1), (1, 1, 1, 24)),
    ((1, 0, 1), (1, 1, 2, 24)),
    ((2, 0, 1), (1, 1, 1, 24)),
    ((2, 0, 1), (1, 1, 2, 24)),
    ((1, 1, 1), (1, 1, 1, 24)),
    ((1, 1, 1), (1, 1, 2, 24)),
    ((2, 1, 1), (1, 1, 1, 24)),
    ((2, 1, 1), (1, 1, 2, 24)),
]

SARIMAX_CANDIDATE_MODELS = [
    ((1, 0, 1), (1, 1, 1, 24)),
    ((1, 0, 1), (1, 1, 2, 24)),
    ((2, 0, 1), (1, 1, 1, 24)),
    ((2, 0, 1), (1, 1, 2, 24)),
    ((1, 1, 1), (1, 1, 1, 24)),
    ((1, 1, 1), (1, 1, 2, 24)),
    ((2, 1, 1), (1, 1, 1, 24)),
    ((2, 1, 1), (1, 1, 2, 24)),
]


def get_default_candidate_models(model_family):
    if model_family == "arima":
        return ARIMA_CANDIDATE_MODELS

    if model_family == "sarima":
        return SARIMA_CANDIDATE_MODELS

    if model_family == "sarimax":
        return SARIMAX_CANDIDATE_MODELS

    raise ValueError("model_family must be arima, sarima, or sarimax")


def run_arima_family_cv(df, cfg: BaselineConfig):
    ensure_output_dir(cfg.output_dir)

    candidate_models = cfg.candidate_models or get_default_candidate_models(cfg.model_family)
    exog_cols = cfg.exog_cols or []
    needed_cols = [cfg.target_col] + exog_cols

    df_model = df.copy()

    for col in needed_cols:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

    df_model = df_model.dropna(subset=needed_cols)

    holdout_start = ensure_same_tz(cfg.holdout_start, df_model.index.tz)
    train_pool = df_model.loc[df_model.index < holdout_start].copy()

    tscv = TimeSeriesSplit(
        n_splits=cfg.n_splits,
        test_size=cfg.cv_test_size
    )

    results = []
    failures = []

    for order, seasonal_order in candidate_models:
        fold_metrics = []
        failed = False

        print(f"Testing {cfg.model_family.upper()} order={order}, seasonal={seasonal_order}")

        for fold, (train_idx, test_idx) in enumerate(tscv.split(train_pool), start=1):
            fold_train = train_pool.iloc[train_idx]
            fold_test = train_pool.iloc[test_idx]

            y_train = fold_train[cfg.target_col]
            y_test = fold_test[cfg.target_col]

            X_train = fold_train[exog_cols] if cfg.model_family == "sarimax" else None
            X_test = fold_test[exog_cols] if cfg.model_family == "sarimax" else None

            try:
                model = SARIMAX(
                    y_train,
                    exog=X_train,
                    order=order,
                    seasonal_order=seasonal_order,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                ).fit(maxiter=cfg.maxiter, disp=False)

                forecast = model.forecast(
                    steps=len(y_test),
                    exog=X_test
                )

                forecast.index = y_test.index
                metrics = evaluate_predictions(y_test, forecast)

                fold_metrics.append({
                    "fold": fold,
                    "RMSE": metrics["RMSE"],
                    "MAE": metrics["MAE"],
                    "R2": metrics["R2"],
                    "MAPE": metrics["MAPE"],
                    "converged": model.mle_retvals.get("converged", None)
                })

                del model, forecast
                clean_memory()

            except Exception as e:
                failures.append({
                    "order": order,
                    "seasonal_order": seasonal_order,
                    "fold": fold,
                    "error": str(e)
                })
                failed = True
                clean_memory()
                break

        if not failed and len(fold_metrics) > 0:
            fold_df = pd.DataFrame(fold_metrics)

            results.append({
                "order": order,
                "seasonal_order": seasonal_order,
                "RMSE": fold_df["RMSE"].mean(),
                "MAE": fold_df["MAE"].mean(),
                "R2": fold_df["R2"].mean(),
                "MAPE": fold_df["MAPE"].mean(),
                "all_folds_converged": bool(fold_df["converged"].all())
            })

    results_df = pd.DataFrame(results)

    if not results_df.empty:
        results_df = results_df.sort_values("RMSE").reset_index(drop=True)

    failures_df = pd.DataFrame(failures)

    return results_df, failures_df


def fit_arima_family_holdout(df, cfg: BaselineConfig, order, seasonal_order):
    ensure_output_dir(cfg.output_dir)

    exog_cols = cfg.exog_cols or []
    needed_cols = [cfg.target_col] + exog_cols

    df_model = df.copy()

    for col in needed_cols:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

    df_model = df_model.dropna(subset=needed_cols)

    holdout_start = ensure_same_tz(cfg.holdout_start, df_model.index.tz)

    train_df = df_model.loc[df_model.index < holdout_start].copy()
    test_df = df_model.loc[df_model.index >= holdout_start].copy()

    y_train = train_df[cfg.target_col]
    y_test = test_df[cfg.target_col]

    X_train = train_df[exog_cols] if cfg.model_family == "sarimax" else None
    X_test = test_df[exog_cols] if cfg.model_family == "sarimax" else None

    model = SARIMAX(
        y_train,
        exog=X_train,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(maxiter=cfg.maxiter, disp=False)

    forecast_obj = model.get_forecast(
        steps=len(y_test),
        exog=X_test
    )

    forecast = forecast_obj.predicted_mean
    conf_int = forecast_obj.conf_int(alpha=cfg.alpha)

    forecast.index = y_test.index
    conf_int.index = y_test.index

    pred_df = pd.DataFrame({
        "actual": y_test,
        "forecast": forecast
    }, index=y_test.index)

    pred_df = add_sarimax_intervals(pred_df, conf_int)

    metrics = evaluate_predictions(pred_df["actual"], pred_df["forecast"])

    file_stem = f"{cfg.model_family}_fixed_holdout"

    pred_df.to_csv(f"{cfg.output_dir}/{file_stem}_forecast.csv")
    pd.DataFrame([metrics]).to_csv(f"{cfg.output_dir}/{file_stem}_metrics.csv", index=False)

    save_forecast_plots(
        pred_df=pred_df,
        output_dir=cfg.output_dir,
        file_stem=file_stem,
        model_label=cfg.model_family.upper(),
        show_plot=cfg.show_plots
    )

    save_residual_outputs(
        pred_df=pred_df,
        actual_col="actual",
        pred_col="forecast",
        output_dir=cfg.output_dir,
        file_stem=file_stem,
        show_plots=cfg.show_plots
    )

    return model, pred_df, metrics


def run_arima_family_fixed_holdout(df, cfg: BaselineConfig):
    cv_results_df, cv_failures_df = run_arima_family_cv(df, cfg)

    cv_results_df.to_csv(f"{cfg.output_dir}/{cfg.model_family}_cv_results.csv", index=False)
    cv_failures_df.to_csv(f"{cfg.output_dir}/{cfg.model_family}_cv_failures.csv", index=False)

    print_table(f"{cfg.model_family.upper()} CV Results", cv_results_df)
    print_table(f"{cfg.model_family.upper()} CV Failures", cv_failures_df)

    if cv_results_df.empty:
        raise ValueError(f"No successful CV results for {cfg.model_family}")

    best_order = tuple(cv_results_df.iloc[0]["order"])
    best_seasonal_order = tuple(cv_results_df.iloc[0]["seasonal_order"])

    model, pred_df, metrics = fit_arima_family_holdout(
        df=df,
        cfg=cfg,
        order=best_order,
        seasonal_order=best_seasonal_order
    )

    print_table(f"{cfg.model_family.upper()} Final Holdout Metrics", pd.DataFrame([metrics]))

    return {
        "cv_results": cv_results_df,
        "cv_failures": cv_failures_df,
        "best_order": best_order,
        "best_seasonal_order": best_seasonal_order,
        "pred_df": pred_df,
        "metrics": metrics
    }


def run_baseline_experiment(df, cfg: BaselineConfig):
    ensure_output_dir(cfg.output_dir)

    if cfg.model_family == "smoothing":
        return run_smoothing_fixed_holdout(df, cfg)

    if cfg.model_family in ["arima", "sarima", "sarimax"]:
        return run_arima_family_fixed_holdout(df, cfg)

    raise ValueError("Invalid baseline model_family")


In [ ]:
# los filtrostatistical-baseline settings
BASELINE_OUTPUT = os.path.join(
    EXTERNAL_OUTPUT,
    "Statistical_Baselines"
)
os.makedirs(BASELINE_OUTPUT, exist_ok=True)

# Alexander Orr used a fixed date-based holdout. The equivalent target-plant
# experiment uses the final two available los filtros months (May-June 2026).
BASELINE_HOLDOUT_START = "2026-05-01"

print("Baseline output folder:", BASELINE_OUTPUT)
print("Baseline holdout starts:", BASELINE_HOLDOUT_START)


## Exponential Smoothing Models


In [ ]:
#Exponential Smoothing
smoothing_fixed_cfg = BaselineConfig(
    model_family="smoothing",
    target_col="totalchlorine",
    output_dir=os.path.join(
        BASELINE_OUTPUT,
        "outputs_smoothing_fixed"
    ),
    holdout_start=BASELINE_HOLDOUT_START,
    smoothing_models=("ses", "des", "holt_winters"),
    show_plots=True
)

smoothing_fixed_outputs = run_baseline_experiment(
    df=losfiltroscaguas,
    cfg=smoothing_fixed_cfg
)


## ARIMA


In [ ]:
#ARIMA
arima_fixed_cfg = BaselineConfig(
    model_family="arima",
    target_col="totalchlorine",
    output_dir=os.path.join(
        BASELINE_OUTPUT,
        "outputs_arima_fixed"
    ),
    holdout_start=BASELINE_HOLDOUT_START,
    candidate_models=ARIMA_CANDIDATE_MODELS,
    n_splits=5,
    cv_test_size=720,
    maxiter=1000,
    show_plots=True
)

arima_fixed_outputs = run_baseline_experiment(
    df=losfiltroscaguas,
    cfg=arima_fixed_cfg
)


## SARIMA


In [ ]:
#SARIMA
sarima_fixed_cfg = BaselineConfig(
    model_family="sarima",
    target_col="totalchlorine",
    output_dir=os.path.join(
        BASELINE_OUTPUT,
        "outputs_sarima_fixed"
    ),
    holdout_start=BASELINE_HOLDOUT_START,
    candidate_models=SARIMA_CANDIDATE_MODELS,
    n_splits=5,
    cv_test_size=720,
    maxiter=700,
    show_plots=True
)

sarima_fixed_outputs = run_baseline_experiment(
    df=losfiltroscaguas,
    cfg=sarima_fixed_cfg
)


## SARIMAX (All Predictors)


In [ ]:
#SARIMAX All Predictors
if "finishwater" not in losfiltroscaguas.columns:
    print(
        "SARIMAX all-predictor baseline paused until "
        "finishwater is available."
    )
    sarimax_fixed_outputs = None
else:
    sarimax_fixed_cfg = BaselineConfig(
        model_family="sarimax",
        target_col="totalchlorine",
        output_dir=os.path.join(
            BASELINE_OUTPUT,
            "outputs_sarimax_fixed"
        ),
        holdout_start=BASELINE_HOLDOUT_START,
        exog_cols=[
            "turbidity",
            "finishwater",
            "chlorine_conversion"
        ],
        candidate_models=SARIMAX_CANDIDATE_MODELS,
        n_splits=5,
        cv_test_size=720,
        maxiter=700,
        show_plots=True
    )

    sarimax_fixed_outputs = run_baseline_experiment(
        df=losfiltroscaguas,
        cfg=sarimax_fixed_cfg
    )


# los filtros external Validation and Transfer Learning

The EDA above mirrors the Alexander Orr analytical sequence. The sections below retain the existing frozen-source, target-from-scratch, and weight-transfer workflow unchanged.


## 6. Saved-model inventory


In [ ]:
saved_model_files = [
    str(path.relative_to(SAVED_MODEL_DIR))
    for path in Path(SAVED_MODEL_DIR).rglob("*")
    if path.is_file()
]

if not saved_model_files:
    raise FileNotFoundError(
        "No saved models were found. Run the saving cells "
        "in the Alexander Orr notebook first."
    )

for filename in sorted(saved_model_files):
    print(filename)


## 7. Shared evaluation helpers


In [ ]:
SEASON_MONTHS = {
    "Winter": [12, 1, 2],
    "Spring": [3, 4, 5],
    "Summer": [6, 7, 8],
    "Autumn": [9, 10, 11]
}


def evaluate_forecast(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    epsilon = 1e-6

    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
        "MAPE": float(
            np.mean(
                np.abs(
                    (y_true - y_pred)
                    / np.maximum(np.abs(y_true), epsilon)
                )
            ) * 100
        )
    }


def inverse_target(y_scaled, scaler, model_cols, target_col):
    target_index = model_cols.index(target_col)
    return (
        np.asarray(y_scaled) * scaler.scale_[target_index]
        + scaler.mean_[target_index]
    )


def season_mask(index, season):
    return pd.DatetimeIndex(index).month.isin(
        SEASON_MONTHS[season]
    )


external_metrics_rows = []


## 8. Deep-learning model helper definitions


### ANN helper definitions


In [ ]:
def add_lags(data, cols, lags):
    out = data.copy()
    for col in cols:
        for lag in lags:
            out[f"{col}_lag_{lag}"] = out[col].shift(lag)
    return out


def build_ann_frame(
    df_in,
    target_col,
    numeric_base_cols,
    categorical_cols,
    lag_steps,
    horizon_steps
):
    out = df_in.copy()
    out["hour_proxy"] = out.index.hour
    out = add_lags(
        out,
        numeric_base_cols + [target_col],
        lag_steps
    )
    out["target_t_plus_12h"] = out[target_col].shift(-horizon_steps)
    out["target_timestamp"] = pd.Series(
        out.index,
        index=out.index
    ).shift(-horizon_steps)

    if "totalchlorine_forward_filled" in out.columns:
        out["target_was_forward_filled"] = (
            out["totalchlorine_forward_filled"]
            .shift(-horizon_steps)
        )
    else:
        out["target_was_forward_filled"] = False

    required_columns = list(dict.fromkeys(
        numeric_base_cols
        + categorical_cols
        + [target_col, "target_t_plus_12h", "target_timestamp"]
        + [
            f"{column}_lag_{lag}"
            for column in numeric_base_cols + [target_col]
            for lag in lag_steps
        ]
        + ["hour_proxy"]
    ))
    out = out.dropna(subset=required_columns).copy()
    out = out.loc[~out["target_was_forward_filled"].astype(bool)].copy()
    return out


def evaluate_saved_ann(model_dir, data):
    model_dir = Path(model_dir)
    with open(model_dir / "model_metadata.json") as file:
        metadata = json.load(file)

    model = tf.keras.models.load_model(
        model_dir / "frozen_model.keras"
    )
    preprocessor = joblib.load(
        model_dir / "fitted_preprocessor.joblib"
    )
    target_scaler = joblib.load(
        model_dir / "fitted_target_scaler.joblib"
    )

    frame = build_ann_frame(
        data,
        metadata["target_col"],
        metadata["numeric_base_cols"],
        metadata["categorical_cols"],
        metadata["lag_steps"],
        metadata["horizon_steps"]
    )
    mask = season_mask(frame["target_timestamp"], metadata["season"])
    frame = frame.loc[mask].copy()
    if frame.empty:
        return None, None

    X = preprocessor.transform(
        frame[metadata["all_feature_cols"]]
    )
    pred_scaled = model.predict(X, verbose=0).reshape(-1, 1)
    prediction = target_scaler.inverse_transform(pred_scaled).reshape(-1)
    actual = frame["target_t_plus_12h"].to_numpy()

    predictions = pd.DataFrame({
        "actual": actual,
        "forecast": prediction
    }, index=pd.DatetimeIndex(frame["target_timestamp"]))

    return predictions, evaluate_forecast(actual, prediction)


### LSTM helper definitions


In [ ]:
def build_lstm_sequences(
    values,
    timestamps,
    lookback,
    horizon,
    n_input_features,
    target_idx
):
    X, y, times = [], [], []
    for t in range(lookback - 1, len(values) - horizon):
        X.append(values[t-lookback+1:t+1, :n_input_features])
        y.append(values[t+horizon, target_idx])
        times.append(timestamps[t+horizon])
    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        times
    )


def evaluate_saved_lstm(model_dir, data):
    model_dir = Path(model_dir)
    with open(model_dir / "model_metadata.json") as file:
        metadata = json.load(file)

    model = tf.keras.models.load_model(
        model_dir / "frozen_model.keras"
    )
    scaler = joblib.load(
        model_dir / "fitted_scaler.joblib"
    )
    model_cols = metadata["model_cols"]

    missing = [col for col in model_cols if col not in data.columns]
    if missing:
        raise ValueError(f"LSTM external columns missing: {missing}")

    scaled = data.copy()
    scaled[model_cols] = scaler.transform(data[model_cols])
    target_idx = model_cols.index(metadata["target_col"])

    X, y_scaled, times = build_lstm_sequences(
        scaled[model_cols].to_numpy(),
        scaled.index.to_numpy(),
        metadata["lookback"],
        metadata["horizon_steps"],
        len(metadata["input_cols"]),
        target_idx
    )
    times = np.asarray(times)
    if "totalchlorine_forward_filled" in data.columns:
        observed_target_mask = ~data.loc[
            pd.DatetimeIndex(times),
            "totalchlorine_forward_filled"
        ].to_numpy(dtype=bool)
        X = X[observed_target_mask]
        y_scaled = y_scaled[observed_target_mask]
        times = times[observed_target_mask]
    mask = season_mask(times, metadata["season"])
    X = X[mask]
    y_scaled = y_scaled[mask]
    times = list(np.asarray(times)[mask])
    if len(X) == 0:
        return None, None

    pred_scaled = model.predict(X, verbose=0).reshape(-1)
    prediction = inverse_target(
        pred_scaled, scaler, model_cols, metadata["target_col"]
    )
    actual = inverse_target(
        y_scaled, scaler, model_cols, metadata["target_col"]
    )

    predictions = pd.DataFrame({
        "actual": actual,
        "forecast": prediction
    }, index=pd.to_datetime(times))

    return predictions, evaluate_forecast(actual, prediction)


### TFT helper definitions


In [ ]:
!pip -q install tft-torch omegaconf

from omegaconf import OmegaConf
from tft_torch.tft import TemporalFusionTransformer


class DictTensorDataset(Dataset):
    def __init__(self, array_dict):
        self.keys_list = list(array_dict.keys())
        self.data = {
            key: torch.tensor(value, dtype=torch.float32)
            for key, value in array_dict.items()
        }

    def __len__(self):
        return self.data[self.keys_list[0]].shape[0]

    def __getitem__(self, idx):
        return {
            key: value[idx]
            for key, value in self.data.items()
        }


def build_tft_arrays(
    df_scaled,
    historical_feature_cols,
    future_known_feature_cols,
    static_numeric_cols,
    target_col,
    lookback,
    horizon
):
    hist_values = df_scaled[historical_feature_cols].to_numpy(np.float32)
    future_values = df_scaled[future_known_feature_cols].to_numpy(np.float32)
    target_values = df_scaled[target_col].to_numpy(np.float32)
    static_values = df_scaled[static_numeric_cols].to_numpy(np.float32)

    hist, future, target, static, times = [], [], [], [], []
    max_start = len(df_scaled) - lookback - horizon + 1

    for start in range(max_start):
        hist_end = start + lookback
        future_end = hist_end + horizon
        hist.append(hist_values[start:hist_end])
        future.append(future_values[hist_end:future_end])
        target.append([target_values[future_end - 1]])
        static.append(static_values[hist_end - 1])
        times.append(df_scaled.index[future_end - 1])

    return {
        "historical_ts_numeric": np.asarray(hist, np.float32),
        "future_ts_numeric": np.asarray(future, np.float32),
        "static_feats_numeric": np.asarray(static, np.float32),
        "target": np.asarray(target, np.float32),
        "tgt_times": times
    }


def build_saved_tft(metadata):
    configuration = {
        "task_type": "regression",
        "target_window_start": metadata["horizon_steps"],
        "data_props": {
            "num_historical_numeric": len(
                metadata["historical_feature_cols"]
            ),
            "num_historical_categorical": 0,
            "historical_categorical_cardinalities": [],
            "num_static_numeric": len(metadata["static_numeric_cols"]),
            "num_static_categorical": 0,
            "static_categorical_cardinalities": [],
            "num_future_numeric": len(
                metadata["future_known_feature_cols"]
            ),
            "num_future_categorical": 0,
            "future_categorical_cardinalities": [],
        },
        "model": {
            "attention_heads": metadata["num_heads"],
            "dropout": metadata["dropout"],
            "lstm_layers": metadata["lstm_layers"],
            "output_quantiles": metadata["quantiles"],
            "state_size": metadata["hidden_units"],
        }
    }
    return TemporalFusionTransformer(
        config=OmegaConf.create(configuration)
    )


def evaluate_saved_tft(model_dir, data):
    model_dir = Path(model_dir)
    with open(model_dir / "model_metadata.json") as file:
        metadata = json.load(file)

    # Reject legacy TFT artifacts whose calendar basis is ambiguous.
    # Retrained source models must name UTC future-known features explicitly.
    ambiguous_calendar_features = {
        "hour", "hour_sin", "hour_cos",
        "dayofweek", "month", "dow_sin", "dow_cos"
    }
    ambiguous_used = sorted(
        ambiguous_calendar_features.intersection(
            metadata["future_known_feature_cols"]
        )
    )
    if ambiguous_used:
        raise ValueError(
            "The saved TFT artifact uses ambiguous calendar features: "
            f"{ambiguous_used}. Retrain/export the source TFT with explicit "
            "utc_ calendar features before transfer evaluation."
        )

    joint_scaler = joblib.load(
        model_dir / "fitted_joint_scaler.joblib"
    )
    static_scaler = joblib.load(
        model_dir / "fitted_static_scaler.joblib"
    )
    model = build_saved_tft(metadata).to(DEVICE)
    state = torch.load(
        model_dir / "frozen_model_state_dict.pt",
        map_location=DEVICE,
        weights_only=True
    )
    model.load_state_dict(state)
    model.eval()

    scaled = data.copy()
    joint_cols = metadata["model_cols"]
    static_cols = metadata["static_numeric_cols"]
    scaled[joint_cols] = joint_scaler.transform(data[joint_cols])
    scaled[static_cols] = static_scaler.transform(data[static_cols])

    arrays = build_tft_arrays(
        scaled,
        metadata["historical_feature_cols"],
        metadata["future_known_feature_cols"],
        static_cols,
        metadata["target_col"],
        metadata["lookback"],
        metadata["horizon_steps"]
    )
    target_times = np.asarray(arrays["tgt_times"])
    observed_target_mask = np.ones(len(target_times), dtype=bool)
    if "totalchlorine_forward_filled" in data.columns:
        observed_target_mask = ~data.loc[
            pd.DatetimeIndex(target_times),
            "totalchlorine_forward_filled"
        ].to_numpy(dtype=bool)
    mask = observed_target_mask & season_mask(
        target_times, metadata["season"]
    )
    for key in [
        "historical_ts_numeric",
        "future_ts_numeric",
        "static_feats_numeric",
        "target"
    ]:
        arrays[key] = arrays[key][mask]
    times = list(target_times[mask])
    if len(times) == 0:
        return None, None

    dataset = DictTensorDataset({
        "historical_ts_numeric": arrays["historical_ts_numeric"],
        "future_ts_numeric": arrays["future_ts_numeric"],
        "static_feats_numeric": arrays["static_feats_numeric"],
        "target": arrays["target"]
    })
    loader = DataLoader(
        dataset,
        batch_size=metadata["batch_size"],
        shuffle=False
    )

    q_values = metadata["quantiles"]
    q10_idx = q_values.index(0.1)
    q50_idx = q_values.index(0.5)
    q90_idx = q_values.index(0.9)
    q10, q50, q90 = [], [], []

    with torch.no_grad():
        for batch in loader:
            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            output = model(batch)["predicted_quantiles"].cpu().numpy()
            q10.extend(output[:, 0, q10_idx])
            q50.extend(output[:, 0, q50_idx])
            q90.extend(output[:, 0, q90_idx])

    actual_scaled = arrays["target"][:, 0]
    actual = inverse_target(
        actual_scaled, joint_scaler, joint_cols, metadata["target_col"]
    )
    lower = inverse_target(
        q10, joint_scaler, joint_cols, metadata["target_col"]
    )
    prediction = inverse_target(
        q50, joint_scaler, joint_cols, metadata["target_col"]
    )
    upper = inverse_target(
        q90, joint_scaler, joint_cols, metadata["target_col"]
    )

    predictions = pd.DataFrame({
        "actual": actual,
        "forecast": prediction,
        "q10": lower,
        "q50": prediction,
        "q90": upper,
        "lower_ci": lower,
        "upper_ci": upper
    }, index=pd.to_datetime(times))

    metrics = evaluate_forecast(actual, prediction)
    metrics["IntervalCoverage"] = float(
        np.mean((actual >= lower) & (actual <= upper))
    )
    return predictions, metrics


## 9. Transfer-learning and uncertainty helper definitions


In [ ]:
# ============================================================
# TRANSFER-LEARNING SETTINGS
# ============================================================

RUN_TRANSFER_LEARNING = True

SEASONAL_EVALUATION_MONTHS = {
    "Winter": {
        "validation_month": 1,
        "holdout_month": 2
    },
    "Spring": {
        "validation_month": 4,
        "holdout_month": 5
    },
    "Summer": {
        "validation_month": 7,
        "holdout_month": 8
    },
    "Autumn": {
        "validation_month": 10,
        "holdout_month": 11
    }
}

# A month must retain at least this proportion of its expected hourly targets
# to qualify as the latest complete validation/holdout occurrence.
MIN_MONTH_COVERAGE = 0.90
DEFAULT_TRANSFER_TRAIN_MONTHS = 24

# ANN/LSTM approximate forecast uncertainty using stochastic dropout passes.
MC_DROPOUT_SAMPLES = 100
MC_INTERVAL_LOWER_PERCENTILE = 2.5
MC_INTERVAL_UPPER_PERCENTILE = 97.5

HEAD_EPOCHS = 40
FINE_TUNE_EPOCHS = 60
SCRATCH_EPOCHS = 100

HEAD_LEARNING_RATE = 1e-3
FINE_TUNE_LEARNING_RATE = 1e-5
SCRATCH_LEARNING_RATE = 1e-3

TRANSFER_PATIENCE = 10

TRANSFER_OUTPUT = os.path.join(
    EXTERNAL_OUTPUT,
    "Transfer_Learning"
)
os.makedirs(TRANSFER_OUTPUT, exist_ok=True)


def saved_training_months(metadata):
    possible_keys = [
        "train_months",
        "training_months",
        "train_length_months",
        "selected_train_months"
    ]
    for key in possible_keys:
        if key in metadata and metadata[key] is not None:
            return int(metadata[key])

    print(
        "Training-length metadata was unavailable for",
        metadata.get("season", "unknown season"),
        "- using",
        DEFAULT_TRANSFER_TRAIN_MONTHS,
        "months."
    )
    return DEFAULT_TRANSFER_TRAIN_MONTHS


def month_bounds(year, month, timezone):
    start = pd.Timestamp(
        year=year,
        month=month,
        day=1,
        tz=timezone
    )
    end = start + pd.offsets.MonthBegin(1)
    return start, end


def month_coverage(index, year, month):
    index = pd.DatetimeIndex(index)
    start, end = month_bounds(
        year,
        month,
        index.tz
    )
    observed = int(
        ((index >= start) & (index < end)).sum()
    )
    expected = len(
        pd.date_range(
            start=start,
            end=end,
            freq="h",
            inclusive="left"
        )
    )
    return observed / expected if expected else 0.0


def latest_complete_season_year(index, season):
    index = pd.DatetimeIndex(index).sort_values()
    month_settings = SEASONAL_EVALUATION_MONTHS[season]
    validation_month = month_settings["validation_month"]
    holdout_month = month_settings["holdout_month"]

    candidate_years = sorted(
        set(index.year),
        reverse=True
    )
    for year in candidate_years:
        validation_coverage = month_coverage(
            index,
            year,
            validation_month
        )
        holdout_coverage = month_coverage(
            index,
            year,
            holdout_month
        )
        if (
            validation_coverage >= MIN_MONTH_COVERAGE
            and holdout_coverage >= MIN_MONTH_COVERAGE
        ):
            return year

    raise ValueError(
        f"No complete {season} validation/holdout pair was found. "
        f"Required coverage: {MIN_MONTH_COVERAGE:.0%}."
    )


def month_based_masks(times, metadata):
    index = pd.DatetimeIndex(times)
    season = metadata["season"]
    year = latest_complete_season_year(
        index,
        season
    )
    month_settings = SEASONAL_EVALUATION_MONTHS[season]

    validation_start, validation_end = month_bounds(
        year,
        month_settings["validation_month"],
        index.tz
    )
    holdout_start, holdout_end = month_bounds(
        year,
        month_settings["holdout_month"],
        index.tz
    )

    requested_train_months = saved_training_months(
        metadata
    )
    requested_train_start = (
        validation_start
        - pd.DateOffset(months=requested_train_months)
    )

    train_mask = (
        (index >= requested_train_start)
        & (index < validation_start)
    )
    validation_mask = (
        (index >= validation_start)
        & (index < validation_end)
    )
    holdout_mask = (
        (index >= holdout_start)
        & (index < holdout_end)
    )

    if not train_mask.any():
        raise ValueError(
            f"No losfiltros training samples precede the {season} "
            "validation month."
        )
    if not validation_mask.any() or not holdout_mask.any():
        raise ValueError(
            f"The selected {season} validation or holdout month "
            "contains no usable samples."
        )

    actual_train_start = index[train_mask].min()
    actual_train_end = index[train_mask].max()

    split_info = {
        "requested_train_months": requested_train_months,
        "train_start": actual_train_start,
        "train_end": actual_train_end,
        "val_start": index[validation_mask].min(),
        "val_end": index[validation_mask].max(),
        "test_start": index[holdout_mask].min(),
        "test_end": index[holdout_mask].max(),
        "evaluation_year": year
    }

    print(
        f"{season} {year}: ",
        f"train={int(train_mask.sum())}, ",
        f"validation={int(validation_mask.sum())}, ",
        f"holdout={int(holdout_mask.sum())}, ",
        f"requested_train_months={requested_train_months}"
    )

    return (
        train_mask,
        validation_mask,
        holdout_mask,
        split_info
    )


def month_based_frame_split(frame, metadata):
    frame = frame.sort_index().copy()
    (
        train_mask,
        validation_mask,
        holdout_mask,
        split_info
    ) = month_based_masks(
        frame.index,
        metadata
    )
    return (
        frame.loc[train_mask].copy(),
        frame.loc[validation_mask].copy(),
        frame.loc[holdout_mask].copy(),
        split_info
    )




def keras_has_active_dropout(model):
    for layer in model.layers:
        layer_name = layer.__class__.__name__.lower()
        if "dropout" in layer_name:
            return True
        if float(getattr(layer, "dropout", 0.0) or 0.0) > 0:
            return True
        if float(getattr(layer, "recurrent_dropout", 0.0) or 0.0) > 0:
            return True
    return False


def mc_dropout_scaled_predictions(model, X):
    samples = []
    for _ in range(MC_DROPOUT_SAMPLES):
        stochastic_prediction = model(
            X,
            training=True
        )
        samples.append(
            np.asarray(stochastic_prediction).reshape(-1)
        )

    samples = np.asarray(samples)
    return {
        "mean": samples.mean(axis=0),
        "lower": np.percentile(
            samples,
            MC_INTERVAL_LOWER_PERCENTILE,
            axis=0
        ),
        "upper": np.percentile(
            samples,
            MC_INTERVAL_UPPER_PERCENTILE,
            axis=0
        )
    }


def interval_metrics(predictions):
    if not {
        "lower_ci",
        "upper_ci"
    }.issubset(predictions.columns):
        return {}

    actual = predictions["actual"].to_numpy()
    lower = predictions["lower_ci"].to_numpy()
    upper = predictions["upper_ci"].to_numpy()
    return {
        "IntervalCoverage": float(
            np.mean(
                (actual >= lower)
                & (actual <= upper)
            )
        ),
        "MeanIntervalWidth": float(
            np.mean(upper - lower)
        )
    }


def plot_true_holdout_forecast(
    predictions,
    model_family,
    configuration,
    season,
    adaptation_method,
    output_path,
    interval_label
):
    # Use the same forecast renderer and style as the percentile analysis.
    plot_forecast_vs_actual_interval(
        pred_df=predictions,
        title=(
            f"{model_family} - {configuration} - {season}\n"
            f"{adaptation_method}: True Unseen Holdout Forecast"
        ),
        actual_col="actual",
        pred_col="forecast",
        lower_col="lower_ci",
        upper_col="upper_ci",
        ylabel="Residual Chlorine (mg/L)",
        save_path=output_path,
        show_plot=True
    )

def early_stopping_callback():
    return tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=TRANSFER_PATIENCE,
        restore_best_weights=True
    )


def compile_regression_model(model, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )


def freeze_keras_except_prediction_head(model):
    for layer in model.layers:
        layer.trainable = False
    model.layers[-1].trainable = True


def unfreeze_all_keras_layers(model):
    for layer in model.layers:
        layer.trainable = True


print("Transfer-learning output:", TRANSFER_OUTPUT)
print("Seasonal validation and holdout months:")
display(pd.DataFrame(SEASONAL_EVALUATION_MONTHS).T)
print("Minimum monthly coverage:", MIN_MONTH_COVERAGE)


### ANN transfer functions


In [ ]:
def prepare_ann_target_split(model_dir, data):
    model_dir = Path(model_dir)

    with open(
        model_dir / "model_metadata.json"
    ) as file:
        metadata = json.load(file)

    preprocessor = joblib.load(
        model_dir / "fitted_preprocessor.joblib"
    )
    target_scaler = joblib.load(
        model_dir / "fitted_target_scaler.joblib"
    )

    frame = build_ann_frame(
        data,
        metadata["target_col"],
        metadata["numeric_base_cols"],
        metadata["categorical_cols"],
        metadata["lag_steps"],
        metadata["horizon_steps"]
    )
    # Split ANN samples by their t + horizon target timestamps, not by
    # their earlier forecast-origin timestamps.
    frame = frame.set_index(
        "target_timestamp",
        drop=False
    ).sort_index()
    (
        train_frame,
        val_frame,
        test_frame,
        split_info
    ) = month_based_frame_split(
        frame,
        metadata
    )
    feature_cols = metadata["all_feature_cols"]

    X_train = preprocessor.transform(
        train_frame[feature_cols]
    )
    X_val = preprocessor.transform(
        val_frame[feature_cols]
    )
    X_test = preprocessor.transform(
        test_frame[feature_cols]
    )

    y_train = target_scaler.transform(
        train_frame[["target_t_plus_12h"]]
    ).reshape(-1)
    y_val = target_scaler.transform(
        val_frame[["target_t_plus_12h"]]
    ).reshape(-1)

    return {
        "metadata": metadata,
        "target_scaler": target_scaler,
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "actual_test": test_frame[
            "target_t_plus_12h"
        ].to_numpy(),
        "test_times": test_frame.index,
        "requested_train_months": split_info[
            "requested_train_months"
        ],
        "evaluation_year": split_info[
            "evaluation_year"
        ],
        "train_start": split_info["train_start"],
        "train_end": split_info["train_end"],
        "val_start": split_info["val_start"],
        "val_end": split_info["val_end"],
        "test_start": split_info["test_start"],
        "test_end": split_info["test_end"]
    }


def ann_predictions(model, prepared):
    mc_scaled = mc_dropout_scaled_predictions(
        model,
        prepared["X_test"]
    )
    target_scaler = prepared["target_scaler"]

    forecast = target_scaler.inverse_transform(
        mc_scaled["mean"].reshape(-1, 1)
    ).reshape(-1)
    lower = target_scaler.inverse_transform(
        mc_scaled["lower"].reshape(-1, 1)
    ).reshape(-1)
    upper = target_scaler.inverse_transform(
        mc_scaled["upper"].reshape(-1, 1)
    ).reshape(-1)

    if not keras_has_active_dropout(model):
        print(
            "Warning: ANN contains no active dropout; "
            "its MC interval may collapse to the forecast."
        )

    return pd.DataFrame(
        {
            "actual": prepared["actual_test"],
            "forecast": forecast,
            "lower_ci": lower,
            "upper_ci": upper
        },
        index=prepared["test_times"]
    )


def train_ann_from_scratch(source_model, prepared):
    model = tf.keras.models.clone_model(source_model)
    compile_regression_model(
        model,
        SCRATCH_LEARNING_RATE
    )
    model.fit(
        prepared["X_train"],
        prepared["y_train"],
        validation_data=(
            prepared["X_val"],
            prepared["y_val"]
        ),
        epochs=SCRATCH_EPOCHS,
        batch_size=int(
            prepared["metadata"].get(
                "batch_size",
                32
            )
        ),
        shuffle=False,
        verbose=0,
        callbacks=[early_stopping_callback()]
    )
    return model


def fine_tune_ann(source_model, prepared):
    model = tf.keras.models.clone_model(source_model)
    model.set_weights(source_model.get_weights())

    freeze_keras_except_prediction_head(model)
    compile_regression_model(
        model,
        HEAD_LEARNING_RATE
    )
    model.fit(
        prepared["X_train"],
        prepared["y_train"],
        validation_data=(
            prepared["X_val"],
            prepared["y_val"]
        ),
        epochs=HEAD_EPOCHS,
        batch_size=int(
            prepared["metadata"].get(
                "batch_size",
                32
            )
        ),
        shuffle=False,
        verbose=0,
        callbacks=[early_stopping_callback()]
    )

    unfreeze_all_keras_layers(model)
    compile_regression_model(
        model,
        FINE_TUNE_LEARNING_RATE
    )
    model.fit(
        prepared["X_train"],
        prepared["y_train"],
        validation_data=(
            prepared["X_val"],
            prepared["y_val"]
        ),
        epochs=FINE_TUNE_EPOCHS,
        batch_size=int(
            prepared["metadata"].get(
                "batch_size",
                32
            )
        ),
        shuffle=False,
        verbose=0,
        callbacks=[early_stopping_callback()]
    )
    return model


### LSTM transfer functions


In [ ]:
def prepare_lstm_target_split(model_dir, data):
    model_dir = Path(model_dir)

    with open(
        model_dir / "model_metadata.json"
    ) as file:
        metadata = json.load(file)

    scaler = joblib.load(
        model_dir / "fitted_scaler.joblib"
    )
    model_cols = metadata["model_cols"]
    scaled = data.copy()
    scaled[model_cols] = scaler.transform(
        data[model_cols]
    )

    target_idx = model_cols.index(
        metadata["target_col"]
    )
    X, y, times = build_lstm_sequences(
        scaled[model_cols].to_numpy(),
        scaled.index.to_numpy(),
        metadata["lookback"],
        metadata["horizon_steps"],
        len(metadata["input_cols"]),
        target_idx
    )

    times = np.asarray(times)
    if "totalchlorine_forward_filled" in data.columns:
        observed_target_mask = ~data.loc[
            pd.DatetimeIndex(times),
            "totalchlorine_forward_filled"
        ].to_numpy(dtype=bool)
        X = X[observed_target_mask]
        y = y[observed_target_mask]
        times = times[observed_target_mask]

    (
        train_mask,
        validation_mask,
        holdout_mask,
        split_info
    ) = month_based_masks(
        times,
        metadata
    )

    return {
        "metadata": metadata,
        "scaler": scaler,
        "model_cols": model_cols,
        "X_train": X[train_mask],
        "y_train": y[train_mask],
        "X_val": X[validation_mask],
        "y_val": y[validation_mask],
        "X_test": X[holdout_mask],
        "y_test": y[holdout_mask],
        "test_times": times[holdout_mask],
        "requested_train_months": split_info[
            "requested_train_months"
        ],
        "evaluation_year": split_info[
            "evaluation_year"
        ],
        "train_start": split_info["train_start"],
        "train_end": split_info["train_end"],
        "val_start": split_info["val_start"],
        "val_end": split_info["val_end"],
        "test_start": split_info["test_start"],
        "test_end": split_info["test_end"]
    }


def lstm_predictions(model, prepared):
    mc_scaled = mc_dropout_scaled_predictions(
        model,
        prepared["X_test"]
    )
    scaler = prepared["scaler"]
    model_cols = prepared["model_cols"]
    target_col = prepared["metadata"]["target_col"]

    forecast = inverse_target(
        mc_scaled["mean"],
        scaler,
        model_cols,
        target_col
    )
    lower = inverse_target(
        mc_scaled["lower"],
        scaler,
        model_cols,
        target_col
    )
    upper = inverse_target(
        mc_scaled["upper"],
        scaler,
        model_cols,
        target_col
    )
    actual = inverse_target(
        prepared["y_test"],
        scaler,
        model_cols,
        target_col
    )

    if not keras_has_active_dropout(model):
        print(
            "Warning: LSTM contains no active dropout; "
            "its MC interval may collapse to the forecast."
        )

    return pd.DataFrame(
        {
            "actual": actual,
            "forecast": forecast,
            "lower_ci": lower,
            "upper_ci": upper
        },
        index=pd.to_datetime(
            prepared["test_times"]
        )
    )


def train_lstm_from_scratch(source_model, prepared):
    model = tf.keras.models.clone_model(source_model)
    compile_regression_model(
        model,
        SCRATCH_LEARNING_RATE
    )
    model.fit(
        prepared["X_train"],
        prepared["y_train"],
        validation_data=(
            prepared["X_val"],
            prepared["y_val"]
        ),
        epochs=SCRATCH_EPOCHS,
        batch_size=int(
            prepared["metadata"].get(
                "batch_size",
                32
            )
        ),
        shuffle=False,
        verbose=0,
        callbacks=[early_stopping_callback()]
    )
    return model


def fine_tune_lstm(source_model, prepared):
    model = tf.keras.models.clone_model(source_model)
    model.set_weights(source_model.get_weights())

    freeze_keras_except_prediction_head(model)
    compile_regression_model(
        model,
        HEAD_LEARNING_RATE
    )
    model.fit(
        prepared["X_train"],
        prepared["y_train"],
        validation_data=(
            prepared["X_val"],
            prepared["y_val"]
        ),
        epochs=HEAD_EPOCHS,
        batch_size=int(
            prepared["metadata"].get(
                "batch_size",
                32
            )
        ),
        shuffle=False,
        verbose=0,
        callbacks=[early_stopping_callback()]
    )

    unfreeze_all_keras_layers(model)
    compile_regression_model(
        model,
        FINE_TUNE_LEARNING_RATE
    )
    model.fit(
        prepared["X_train"],
        prepared["y_train"],
        validation_data=(
            prepared["X_val"],
            prepared["y_val"]
        ),
        epochs=FINE_TUNE_EPOCHS,
        batch_size=int(
            prepared["metadata"].get(
                "batch_size",
                32
            )
        ),
        shuffle=False,
        verbose=0,
        callbacks=[early_stopping_callback()]
    )
    return model


### TFT transfer functions


In [ ]:
import copy


def prepare_tft_target_split(model_dir, data):
    model_dir = Path(model_dir)

    with open(
        model_dir / "model_metadata.json"
    ) as file:
        metadata = json.load(file)

    joint_scaler = joblib.load(
        model_dir / "fitted_joint_scaler.joblib"
    )
    static_scaler = joblib.load(
        model_dir / "fitted_static_scaler.joblib"
    )

    joint_cols = metadata["model_cols"]
    static_cols = metadata["static_numeric_cols"]
    scaled = data.copy()
    scaled[joint_cols] = joint_scaler.transform(
        data[joint_cols]
    )
    scaled[static_cols] = static_scaler.transform(
        data[static_cols]
    )

    arrays = build_tft_arrays(
        scaled,
        metadata["historical_feature_cols"],
        metadata["future_known_feature_cols"],
        static_cols,
        metadata["target_col"],
        metadata["lookback"],
        metadata["horizon_steps"]
    )

    array_keys = [
        "historical_ts_numeric",
        "future_ts_numeric",
        "static_feats_numeric",
        "target"
    ]
    times = np.asarray(
        arrays["tgt_times"]
    )
    if "totalchlorine_forward_filled" in data.columns:
        observed_target_mask = ~data.loc[
            pd.DatetimeIndex(times),
            "totalchlorine_forward_filled"
        ].to_numpy(dtype=bool)
        for key in array_keys:
            arrays[key] = arrays[key][observed_target_mask]
        times = times[observed_target_mask]

    (
        train_mask,
        validation_mask,
        holdout_mask,
        split_info
    ) = month_based_masks(
        times,
        metadata
    )

    def subset(mask):
        return {
            key: arrays[key][mask]
            for key in array_keys
        }

    return {
        "metadata": metadata,
        "joint_scaler": joint_scaler,
        "joint_cols": joint_cols,
        "train": subset(train_mask),
        "val": subset(validation_mask),
        "test": subset(holdout_mask),
        "test_times": times[holdout_mask],
        "requested_train_months": split_info[
            "requested_train_months"
        ],
        "evaluation_year": split_info[
            "evaluation_year"
        ],
        "train_start": split_info["train_start"],
        "train_end": split_info["train_end"],
        "val_start": split_info["val_start"],
        "val_end": split_info["val_end"],
        "test_start": split_info["test_start"],
        "test_end": split_info["test_end"]
    }


def tft_loader(array_dict, batch_size, shuffle):
    return DataLoader(
        DictTensorDataset(array_dict),
        batch_size=batch_size,
        shuffle=shuffle
    )


def tft_quantile_loss(output, target, quantiles):
    q = torch.tensor(
        quantiles,
        dtype=output.dtype,
        device=output.device
    ).view(1, 1, -1)
    error = target.unsqueeze(-1) - output
    return torch.maximum(
        q * error,
        (q - 1.0) * error
    ).mean()


def train_tft_stage(
    model,
    train_loader,
    val_loader,
    quantiles,
    learning_rate,
    epochs
):
    optimizer = torch.optim.Adam(
        [
            parameter
            for parameter in model.parameters()
            if parameter.requires_grad
        ],
        lr=learning_rate
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )
    best_val = np.inf
    patience_count = 0

    for _ in range(epochs):
        model.train()
        for batch in train_loader:
            batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }
            optimizer.zero_grad()
            output = model(batch)[
                "predicted_quantiles"
            ]
            loss = tft_quantile_loss(
                output,
                batch["target"],
                quantiles
            )
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                batch = {
                    key: value.to(DEVICE)
                    for key, value in batch.items()
                }
                output = model(batch)[
                    "predicted_quantiles"
                ]
                val_losses.append(
                    tft_quantile_loss(
                        output,
                        batch["target"],
                        quantiles
                    ).item()
                )

        val_loss = float(np.mean(val_losses))
        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(
                model.state_dict()
            )
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= TRANSFER_PATIENCE:
            break

    model.load_state_dict(best_state)
    return model


def freeze_tft_except_prediction_head(model):
    leaf_modules = [
        module
        for module in model.modules()
        if not list(module.children())
        and list(module.parameters(recurse=False))
    ]
    if not leaf_modules:
        raise RuntimeError(
            "No TFT prediction-head module was found."
        )

    for parameter in model.parameters():
        parameter.requires_grad = False

    prediction_head = leaf_modules[-1]
    for parameter in prediction_head.parameters():
        parameter.requires_grad = True


def unfreeze_all_tft_parameters(model):
    for parameter in model.parameters():
        parameter.requires_grad = True


def train_tft_from_scratch(prepared):
    metadata = prepared["metadata"]
    model = build_saved_tft(metadata).to(DEVICE)
    batch_size = int(
        metadata.get("batch_size", 32)
    )
    return train_tft_stage(
        model,
        tft_loader(
            prepared["train"],
            batch_size,
            True
        ),
        tft_loader(
            prepared["val"],
            batch_size,
            False
        ),
        metadata["quantiles"],
        SCRATCH_LEARNING_RATE,
        SCRATCH_EPOCHS
    )


def fine_tune_tft(source_model, prepared):
    metadata = prepared["metadata"]
    model = build_saved_tft(metadata).to(DEVICE)
    model.load_state_dict(
        copy.deepcopy(source_model.state_dict())
    )
    batch_size = int(
        metadata.get("batch_size", 32)
    )
    train_loader = tft_loader(
        prepared["train"],
        batch_size,
        True
    )
    val_loader = tft_loader(
        prepared["val"],
        batch_size,
        False
    )

    freeze_tft_except_prediction_head(model)
    train_tft_stage(
        model,
        train_loader,
        val_loader,
        metadata["quantiles"],
        HEAD_LEARNING_RATE,
        HEAD_EPOCHS
    )

    unfreeze_all_tft_parameters(model)
    train_tft_stage(
        model,
        train_loader,
        val_loader,
        metadata["quantiles"],
        FINE_TUNE_LEARNING_RATE,
        FINE_TUNE_EPOCHS
    )
    return model


def tft_predictions(model, prepared):
    metadata = prepared["metadata"]
    loader = tft_loader(
        prepared["test"],
        int(metadata.get("batch_size", 32)),
        False
    )
    q_values = metadata["quantiles"]
    q10_idx = q_values.index(0.1)
    q50_idx = q_values.index(0.5)
    q90_idx = q_values.index(0.9)
    q10, q50, q90 = [], [], []

    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }
            output = model(batch)[
                "predicted_quantiles"
            ].cpu().numpy()
            q10.extend(output[:, 0, q10_idx])
            q50.extend(output[:, 0, q50_idx])
            q90.extend(output[:, 0, q90_idx])

    target_scaled = prepared["test"][
        "target"
    ][:, 0]
    actual = inverse_target(
        target_scaled,
        prepared["joint_scaler"],
        prepared["joint_cols"],
        metadata["target_col"]
    )
    lower = inverse_target(
        q10,
        prepared["joint_scaler"],
        prepared["joint_cols"],
        metadata["target_col"]
    )
    forecast = inverse_target(
        q50,
        prepared["joint_scaler"],
        prepared["joint_cols"],
        metadata["target_col"]
    )
    upper = inverse_target(
        q90,
        prepared["joint_scaler"],
        prepared["joint_cols"],
        metadata["target_col"]
    )

    return pd.DataFrame(
        {
            "actual": actual,
            "forecast": forecast,
            "q10": lower,
            "q50": forecast,
            "q90": upper,
            "lower_ci": lower,
            "upper_ci": upper
        },
        index=pd.to_datetime(
            prepared["test_times"]
        )
    )


## 10. Chunk runner


In [ ]:
# ============================================================
# CHUNKED EXTERNAL TRANSFER-LEARNING RUNNER
# ============================================================

TRANSFER_RESULT_RECORDS = {}


def _matching_metadata_paths(model_family, configuration, season):
    root = (
        Path(SAVED_MODEL_DIR)
        / "DeepLearning"
        / model_family
    )
    matches = []
    for metadata_path in sorted(
        root.rglob("model_metadata.json")
    ):
        with open(metadata_path) as file:
            metadata = json.load(file)

        saved_configuration = normalize_configuration_name(
            metadata["config_name"]
        )
        if (
            saved_configuration
            == normalize_configuration_name(configuration)
            and metadata["season"] == season
        ):
            matches.append((metadata_path, metadata))
    return matches


def _save_current_transfer_tables():
    global transfer_results
    global ann_transfer_results
    global lstm_transfer_results
    global tft_transfer_results

    transfer_results = pd.DataFrame(
        list(TRANSFER_RESULT_RECORDS.values())
    )

    if transfer_results.empty:
        ann_transfer_results = pd.DataFrame()
        lstm_transfer_results = pd.DataFrame()
        tft_transfer_results = pd.DataFrame()
        return

    transfer_results = transfer_results.sort_values(
        [
            "Configuration",
            "Season",
            "ModelFamily",
            "AdaptationMethod"
        ]
    ).reset_index(drop=True)

    ann_transfer_results = transfer_results.loc[
        transfer_results["ModelFamily"] == "ANN"
    ].copy()
    lstm_transfer_results = transfer_results.loc[
        transfer_results["ModelFamily"] == "LSTM"
    ].copy()
    tft_transfer_results = transfer_results.loc[
        transfer_results["ModelFamily"] == "TFT"
    ].copy()

    transfer_results.to_csv(
        Path(TRANSFER_OUTPUT)
        / "all_transfer_learning_metrics.csv",
        index=False
    )
    ann_transfer_results.to_csv(
        Path(TRANSFER_OUTPUT)
        / "ann_transfer_comparison.csv",
        index=False
    )
    lstm_transfer_results.to_csv(
        Path(TRANSFER_OUTPUT)
        / "lstm_transfer_comparison.csv",
        index=False
    )
    tft_transfer_results.to_csv(
        Path(TRANSFER_OUTPUT)
        / "tft_transfer_comparison.csv",
        index=False
    )


def _store_transfer_result(
    model_family,
    configuration,
    season,
    method,
    prepared,
    metrics
):
    record = {
        "Dataset": DATASET_NAME,
        "ModelFamily": model_family,
        "Configuration": configuration,
        "Season": season,
        "AdaptationMethod": method,
        "RequestedTrainMonths": prepared[
            "requested_train_months"
        ],
        "EvaluationYear": prepared["evaluation_year"],
        "TrainStart": prepared["train_start"],
        "TrainEnd": prepared["train_end"],
        "ValidationStart": prepared["val_start"],
        "ValidationEnd": prepared["val_end"],
        "TestStart": prepared["test_start"],
        "TestEnd": prepared["test_end"],
        **metrics
    }
    key = (
        model_family,
        configuration,
        season,
        method
    )
    TRANSFER_RESULT_RECORDS[key] = record


def _run_ann_chunk(configuration, season):
    chunk_rows = []
    model_matches = _matching_metadata_paths(
        "ANN",
        configuration,
        season
    )

    for metadata_path, metadata in model_matches:
        required_columns = list(dict.fromkeys(
            metadata["numeric_base_cols"]
            + metadata["categorical_cols"]
            + [metadata["target_col"]]
        ))
        missing_columns = [
            column
            for column in required_columns
            if column not in losfiltroscaguas.columns
        ]
        if missing_columns:
            print(
                "Skipping ANN",
                configuration,
                season,
                "- unavailable inputs:",
                missing_columns
            )
            continue

        model_dir = metadata_path.parent
        prepared = prepare_ann_target_split(
            model_dir,
            losfiltroscaguas
        )
        source_model = tf.keras.models.load_model(
            model_dir / "frozen_model.keras"
        )
        models_to_compare = {
            "Frozen_Source": source_model,
            "Target_From_Scratch": train_ann_from_scratch(
                source_model,
                prepared
            ),
            "Weight_Transfer": fine_tune_ann(
                source_model,
                prepared
            )
        }

        output_dir = (
            Path(TRANSFER_OUTPUT)
            / "ANN"
            / configuration
            / season
        )
        output_dir.mkdir(parents=True, exist_ok=True)

        for method, model in models_to_compare.items():
            predictions = ann_predictions(model, prepared)
            metrics = evaluate_forecast(
                predictions["actual"],
                predictions["forecast"]
            )
            metrics.update(interval_metrics(predictions))
            _store_transfer_result(
                "ANN",
                configuration,
                season,
                method,
                prepared,
                metrics
            )
            chunk_rows.append({
                "ModelFamily": "ANN",
                "Configuration": configuration,
                "Season": season,
                "AdaptationMethod": method,
                **metrics
            })

            stem = f"ANN_{configuration}_{season}_{method}"
            predictions.to_csv(
                output_dir / f"{stem}_predictions.csv"
            )
            pd.DataFrame([{
                "Dataset": DATASET_NAME,
                "ModelFamily": "ANN",
                "Configuration": configuration,
                "Season": season,
                "AdaptationMethod": method,
                **metrics
            }]).to_csv(
                output_dir / f"{stem}_metrics.csv",
                index=False
            )
            plot_true_holdout_forecast(
                predictions=predictions,
                model_family="ANN",
                configuration=configuration,
                season=season,
                adaptation_method=method,
                output_path=(
                    output_dir
                    / f"{stem}_true_holdout_forecast.png"
                ),
                interval_label=(
                    "95% MC-dropout forecast interval"
                )
            )
            if method != "Frozen_Source":
                model.save(output_dir / f"{stem}.keras")

    return chunk_rows


def _run_lstm_chunk(configuration, season):
    chunk_rows = []
    model_matches = _matching_metadata_paths(
        "LSTM",
        configuration,
        season
    )

    for metadata_path, metadata in model_matches:
        missing_columns = [
            column
            for column in metadata["model_cols"]
            if column not in losfiltroscaguas.columns
        ]
        if missing_columns:
            print(
                "Skipping LSTM",
                configuration,
                season,
                "- unavailable inputs:",
                missing_columns
            )
            continue

        model_dir = metadata_path.parent
        prepared = prepare_lstm_target_split(
            model_dir,
            losfiltroscaguas
        )
        source_model = tf.keras.models.load_model(
            model_dir / "frozen_model.keras"
        )
        models_to_compare = {
            "Frozen_Source": source_model,
            "Target_From_Scratch": train_lstm_from_scratch(
                source_model,
                prepared
            ),
            "Weight_Transfer": fine_tune_lstm(
                source_model,
                prepared
            )
        }

        output_dir = (
            Path(TRANSFER_OUTPUT)
            / "LSTM"
            / configuration
            / season
        )
        output_dir.mkdir(parents=True, exist_ok=True)

        for method, model in models_to_compare.items():
            predictions = lstm_predictions(model, prepared)
            metrics = evaluate_forecast(
                predictions["actual"],
                predictions["forecast"]
            )
            metrics.update(interval_metrics(predictions))
            _store_transfer_result(
                "LSTM",
                configuration,
                season,
                method,
                prepared,
                metrics
            )
            chunk_rows.append({
                "ModelFamily": "LSTM",
                "Configuration": configuration,
                "Season": season,
                "AdaptationMethod": method,
                **metrics
            })

            stem = f"LSTM_{configuration}_{season}_{method}"
            predictions.to_csv(
                output_dir / f"{stem}_predictions.csv"
            )
            pd.DataFrame([{
                "Dataset": DATASET_NAME,
                "ModelFamily": "LSTM",
                "Configuration": configuration,
                "Season": season,
                "AdaptationMethod": method,
                **metrics
            }]).to_csv(
                output_dir / f"{stem}_metrics.csv",
                index=False
            )
            plot_true_holdout_forecast(
                predictions=predictions,
                model_family="LSTM",
                configuration=configuration,
                season=season,
                adaptation_method=method,
                output_path=(
                    output_dir
                    / f"{stem}_true_holdout_forecast.png"
                ),
                interval_label=(
                    "95% MC-dropout forecast interval"
                )
            )
            if method != "Frozen_Source":
                model.save(output_dir / f"{stem}.keras")

    return chunk_rows


def _run_tft_chunk(configuration, season):
    chunk_rows = []
    model_matches = _matching_metadata_paths(
        "TFT",
        configuration,
        season
    )

    for metadata_path, metadata in model_matches:
        required_columns = list(dict.fromkeys(
            metadata["model_cols"]
            + metadata["static_numeric_cols"]
        ))
        missing_columns = [
            column
            for column in required_columns
            if column not in losfiltroscaguas.columns
        ]
        if missing_columns:
            print(
                "Skipping TFT",
                configuration,
                season,
                "- unavailable inputs:",
                missing_columns
            )
            continue

        model_dir = metadata_path.parent
        prepared = prepare_tft_target_split(
            model_dir,
            losfiltroscaguas
        )
        source_model = build_saved_tft(
            prepared["metadata"]
        ).to(DEVICE)
        source_state = torch.load(
            model_dir / "frozen_model_state_dict.pt",
            map_location=DEVICE,
            weights_only=True
        )
        source_model.load_state_dict(source_state)
        models_to_compare = {
            "Frozen_Source": source_model,
            "Target_From_Scratch": train_tft_from_scratch(
                prepared
            ),
            "Weight_Transfer": fine_tune_tft(
                source_model,
                prepared
            )
        }

        output_dir = (
            Path(TRANSFER_OUTPUT)
            / "TFT"
            / configuration
            / season
        )
        output_dir.mkdir(parents=True, exist_ok=True)

        for method, model in models_to_compare.items():
            predictions = tft_predictions(model, prepared)
            metrics = evaluate_forecast(
                predictions["actual"],
                predictions["forecast"]
            )
            metrics.update(interval_metrics(predictions))
            _store_transfer_result(
                "TFT",
                configuration,
                season,
                method,
                prepared,
                metrics
            )
            chunk_rows.append({
                "ModelFamily": "TFT",
                "Configuration": configuration,
                "Season": season,
                "AdaptationMethod": method,
                **metrics
            })

            stem = f"TFT_{configuration}_{season}_{method}"
            predictions.to_csv(
                output_dir / f"{stem}_predictions.csv"
            )
            pd.DataFrame([{
                "Dataset": DATASET_NAME,
                "ModelFamily": "TFT",
                "Configuration": configuration,
                "Season": season,
                "AdaptationMethod": method,
                **metrics
            }]).to_csv(
                output_dir / f"{stem}_metrics.csv",
                index=False
            )
            plot_true_holdout_forecast(
                predictions=predictions,
                model_family="TFT",
                configuration=configuration,
                season=season,
                adaptation_method=method,
                output_path=(
                    output_dir
                    / f"{stem}_true_holdout_forecast.png"
                ),
                interval_label=(
                    "80% quantile forecast interval (q10-q90)"
                )
            )
            if method != "Frozen_Source":
                torch.save(
                    model.state_dict(),
                    output_dir / f"{stem}_state_dict.pt"
                )

    return chunk_rows


def run_model_transfer_chunk(
    model_family,
    configuration,
    season
):
    model_family = model_family.upper()
    configuration = normalize_configuration_name(
        configuration
    )
    valid_families = ["ANN", "LSTM", "TFT"]
    valid_seasons = [
        "Winter",
        "Spring",
        "Summer",
        "Autumn"
    ]
    if model_family not in valid_families:
        raise ValueError(
            f"Model family must be one of {valid_families}."
        )
    if season not in valid_seasons:
        raise ValueError(
            f"Season must be one of {valid_seasons}."
        )

    print("\n" + "=" * 78)
    print(
        "los filtros TRANSFER CHUNK:",
        model_family,
        "-",
        configuration,
        "-",
        season
    )
    print("=" * 78)

    if not RUN_TRANSFER_LEARNING:
        print("RUN_TRANSFER_LEARNING is False; chunk skipped.")
        return pd.DataFrame()

    family_runners = {
        "ANN": _run_ann_chunk,
        "LSTM": _run_lstm_chunk,
        "TFT": _run_tft_chunk
    }
    chunk_rows = family_runners[model_family](
        configuration,
        season
    )

    _save_current_transfer_tables()
    chunk_results = pd.DataFrame(chunk_rows)

    if chunk_results.empty:
        print(
            "No compatible saved model ran for this chunk. "
            "Check saved metadata and available variables."
        )
    else:
        display(
            chunk_results.sort_values(
                ["AdaptationMethod"]
            ).reset_index(drop=True)
        )

    return chunk_results


def run_external_transfer_chunk(configuration, season):
    configuration = normalize_configuration_name(
        configuration
    )
    valid_seasons = [
        "Winter",
        "Spring",
        "Summer",
        "Autumn"
    ]
    if season not in valid_seasons:
        raise ValueError(
            f"Season must be one of {valid_seasons}."
        )

    print("\n" + "=" * 78)
    print(
        "los filtros TRANSFER CHUNK:",
        configuration,
        "-",
        season
    )
    print("=" * 78)

    if not RUN_TRANSFER_LEARNING:
        print("RUN_TRANSFER_LEARNING is False; chunk skipped.")
        return pd.DataFrame()

    chunk_rows = []
    chunk_rows.extend(
        _run_ann_chunk(configuration, season)
    )
    chunk_rows.extend(
        _run_lstm_chunk(configuration, season)
    )
    chunk_rows.extend(
        _run_tft_chunk(configuration, season)
    )

    _save_current_transfer_tables()
    chunk_results = pd.DataFrame(chunk_rows)

    if chunk_results.empty:
        print(
            "No compatible saved models ran for this chunk. "
            "Check saved metadata and available variables."
        )
    else:
        display(
            chunk_results.sort_values(
                ["ModelFamily", "AdaptationMethod"]
            ).reset_index(drop=True)
        )

    return chunk_results


## 11. LSTM transfer-learning runs


### LSTM — Univariate


#### Chunk 1: Winter


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="LSTM",
    configuration="univariate",
    season="Winter"
)


#### Chunk 2: Spring


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="LSTM",
    configuration="univariate",
    season="Spring"
)


#### Chunk 3: Summer


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="LSTM",
    configuration="univariate",
    season="Summer"
)


#### Chunk 4: Autumn


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="LSTM",
    configuration="univariate",
    season="Autumn"
)


### LSTM — Turbidity


#### Chunk 1: Winter


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="LSTM",
    configuration="turbidity",
    season="Winter"
)


#### Chunk 2: Spring


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="LSTM",
    configuration="turbidity",
    season="Spring"
)


#### Chunk 3: Summer


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="LSTM",
    configuration="turbidity",
    season="Summer"
)


#### Chunk 4: Autumn


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="LSTM",
    configuration="turbidity",
    season="Autumn"
)


### LSTM — Flow-rate


> **Paused:** this configuration requires `finishwater`. Leave these cells commented until flow-rate data are received and mapped.


#### Chunk 1: Winter


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="LSTM",
#     configuration="flowrate",
#     season="Winter"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 2: Spring


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="LSTM",
#     configuration="flowrate",
#     season="Spring"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 3: Summer


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="LSTM",
#     configuration="flowrate",
#     season="Summer"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 4: Autumn


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="LSTM",
#     configuration="flowrate",
#     season="Autumn"
# )
print("Chunk paused until finishwater is available.")


### LSTM — Flow-rate + turbidity


> **Paused:** this configuration requires `finishwater`. Leave these cells commented until flow-rate data are received and mapped.


#### Chunk 1: Winter


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="LSTM",
#     configuration="flowrate_turbidity",
#     season="Winter"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 2: Spring


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="LSTM",
#     configuration="flowrate_turbidity",
#     season="Spring"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 3: Summer


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="LSTM",
#     configuration="flowrate_turbidity",
#     season="Summer"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 4: Autumn


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="LSTM",
#     configuration="flowrate_turbidity",
#     season="Autumn"
# )
print("Chunk paused until finishwater is available.")


### LSTM — Full model


> **Paused:** this configuration requires `finishwater`. Leave these cells commented until flow-rate data are received and mapped.


#### Chunk 1: Winter


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="LSTM",
#     configuration="full_model",
#     season="Winter"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 2: Spring


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="LSTM",
#     configuration="full_model",
#     season="Spring"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 3: Summer


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="LSTM",
#     configuration="full_model",
#     season="Summer"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 4: Autumn


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="LSTM",
#     configuration="full_model",
#     season="Autumn"
# )
print("Chunk paused until finishwater is available.")


## 12. ANN transfer-learning runs


### ANN — Univariate


#### Chunk 1: Winter


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="ANN",
    configuration="univariate",
    season="Winter"
)


#### Chunk 2: Spring


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="ANN",
    configuration="univariate",
    season="Spring"
)


#### Chunk 3: Summer


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="ANN",
    configuration="univariate",
    season="Summer"
)


#### Chunk 4: Autumn


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="ANN",
    configuration="univariate",
    season="Autumn"
)


### ANN — Turbidity


#### Chunk 1: Winter


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="ANN",
    configuration="turbidity",
    season="Winter"
)


#### Chunk 2: Spring


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="ANN",
    configuration="turbidity",
    season="Spring"
)


#### Chunk 3: Summer


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="ANN",
    configuration="turbidity",
    season="Summer"
)


#### Chunk 4: Autumn


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="ANN",
    configuration="turbidity",
    season="Autumn"
)


### ANN — Flow-rate


> **Paused:** this configuration requires `finishwater`. Leave these cells commented until flow-rate data are received and mapped.


#### Chunk 1: Winter


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="ANN",
#     configuration="flowrate",
#     season="Winter"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 2: Spring


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="ANN",
#     configuration="flowrate",
#     season="Spring"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 3: Summer


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="ANN",
#     configuration="flowrate",
#     season="Summer"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 4: Autumn


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="ANN",
#     configuration="flowrate",
#     season="Autumn"
# )
print("Chunk paused until finishwater is available.")


### ANN — Flow-rate + turbidity


> **Paused:** this configuration requires `finishwater`. Leave these cells commented until flow-rate data are received and mapped.


#### Chunk 1: Winter


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="ANN",
#     configuration="flowrate_turbidity",
#     season="Winter"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 2: Spring


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="ANN",
#     configuration="flowrate_turbidity",
#     season="Spring"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 3: Summer


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="ANN",
#     configuration="flowrate_turbidity",
#     season="Summer"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 4: Autumn


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="ANN",
#     configuration="flowrate_turbidity",
#     season="Autumn"
# )
print("Chunk paused until finishwater is available.")


### ANN — Full model


> **Paused:** this configuration requires `finishwater`. Leave these cells commented until flow-rate data are received and mapped.


#### Chunk 1: Winter


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="ANN",
#     configuration="full_model",
#     season="Winter"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 2: Spring


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="ANN",
#     configuration="full_model",
#     season="Spring"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 3: Summer


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="ANN",
#     configuration="full_model",
#     season="Summer"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 4: Autumn


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="ANN",
#     configuration="full_model",
#     season="Autumn"
# )
print("Chunk paused until finishwater is available.")


## 13. TFT transfer-learning runs


### TFT — Univariate


#### Chunk 1: Winter


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="TFT",
    configuration="univariate",
    season="Winter"
)


#### Chunk 2: Spring


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="TFT",
    configuration="univariate",
    season="Spring"
)


#### Chunk 3: Summer


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="TFT",
    configuration="univariate",
    season="Summer"
)


#### Chunk 4: Autumn


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="TFT",
    configuration="univariate",
    season="Autumn"
)


### TFT — Turbidity


#### Chunk 1: Winter


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="TFT",
    configuration="turbidity",
    season="Winter"
)


#### Chunk 2: Spring


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="TFT",
    configuration="turbidity",
    season="Spring"
)


#### Chunk 3: Summer


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="TFT",
    configuration="turbidity",
    season="Summer"
)


#### Chunk 4: Autumn


In [ ]:
chunk_results = run_model_transfer_chunk(
    model_family="TFT",
    configuration="turbidity",
    season="Autumn"
)


### TFT — Flow-rate


> **Paused:** this configuration requires `finishwater`. Leave these cells commented until flow-rate data are received and mapped.


#### Chunk 1: Winter


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="TFT",
#     configuration="flowrate",
#     season="Winter"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 2: Spring


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="TFT",
#     configuration="flowrate",
#     season="Spring"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 3: Summer


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="TFT",
#     configuration="flowrate",
#     season="Summer"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 4: Autumn


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="TFT",
#     configuration="flowrate",
#     season="Autumn"
# )
print("Chunk paused until finishwater is available.")


### TFT — Flow-rate + turbidity


> **Paused:** this configuration requires `finishwater`. Leave these cells commented until flow-rate data are received and mapped.


#### Chunk 1: Winter


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="TFT",
#     configuration="flowrate_turbidity",
#     season="Winter"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 2: Spring


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="TFT",
#     configuration="flowrate_turbidity",
#     season="Spring"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 3: Summer


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="TFT",
#     configuration="flowrate_turbidity",
#     season="Summer"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 4: Autumn


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="TFT",
#     configuration="flowrate_turbidity",
#     season="Autumn"
# )
print("Chunk paused until finishwater is available.")


### TFT — Full model


> **Paused:** this configuration requires `finishwater`. Leave these cells commented until flow-rate data are received and mapped.


#### Chunk 1: Winter


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="TFT",
#     configuration="full_model",
#     season="Winter"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 2: Spring


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="TFT",
#     configuration="full_model",
#     season="Spring"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 3: Summer


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="TFT",
#     configuration="full_model",
#     season="Summer"
# )
print("Chunk paused until finishwater is available.")


#### Chunk 4: Autumn


In [ ]:
# Flow rate is not available yet. After mapping the
# source flow-rate column to finishwater, uncomment:
# chunk_results = run_model_transfer_chunk(
#     model_family="TFT",
#     configuration="full_model",
#     season="Autumn"
# )
print("Chunk paused until finishwater is available.")


## 14. Combined deep-learning transfer results


In [ ]:
if TRANSFER_RESULT_RECORDS:
    _save_current_transfer_tables()
    display(transfer_results)

    plt.figure(figsize=(14, 7))
    sns.barplot(
        data=transfer_results,
        x="AdaptationMethod",
        y="RMSE",
        hue="ModelFamily"
    )
    plt.title(
        "los filtrosransfer-Learning RMSE Comparison"
    )
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig(
        Path(TRANSFER_OUTPUT)
        / "transfer_learning_rmse_comparison.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    method_summary = (
        transfer_results
        .groupby(
            ["ModelFamily", "AdaptationMethod"],
            as_index=False
        )[[
            "RMSE",
            "MAE",
            "R2",
            "MAPE",
            "IntervalCoverage",
            "MeanIntervalWidth"
        ]]
        .mean()
    )
    display(method_summary)
    method_summary.to_csv(
        Path(TRANSFER_OUTPUT)
        / "transfer_learning_method_summary.csv",
        index=False
    )
else:
    transfer_results = pd.DataFrame()
    print(
        "No chunks have been run in this runtime. "
        "Run the desired configuration/season cells first."
    )


## Deep-learning statistical comparison — Friedman and Wilcoxon

Within each predictor configuration, season and adaptation method, ANN, LSTM and TFT are compared using paired absolute residuals from their identical true-holdout timestamps. A significant Friedman test is followed by pairwise Wilcoxon signed-rank tests with Holm correction.


In [ ]:
# Discover nested true-holdout prediction files
COMPARISON_OUTPUT = (
    Path(TRANSFER_OUTPUT)
    / "Statistical_Comparisons"
)
COMPARISON_OUTPUT.mkdir(parents=True, exist_ok=True)

TRANSFER_METHODS = [
    "Frozen_Source",
    "Target_From_Scratch",
    "Weight_Transfer"
]


def identify_transfer_method(file_name):
    for method in TRANSFER_METHODS:
        if method.lower() in file_name.lower():
            return method
    return None


def load_transfer_holdout_residuals():
    records = []
    prediction_files = sorted(
        Path(TRANSFER_OUTPUT).glob(
            "*/*/*/*_predictions.csv"
        )
    )

    for prediction_path in prediction_files:
        relative = prediction_path.relative_to(
            Path(TRANSFER_OUTPUT)
        )
        if len(relative.parts) < 4:
            continue

        model_family, configuration, season = (
            relative.parts[:3]
        )
        if model_family not in {"ANN", "LSTM", "TFT"}:
            continue

        method = identify_transfer_method(
            prediction_path.name
        )
        if method is None:
            continue

        frame = pd.read_csv(prediction_path)
        if not {"actual", "forecast"}.issubset(frame.columns):
            print(
                "Skipped prediction file missing actual/forecast:",
                prediction_path
            )
            continue

        timestamp_column = frame.columns[0]
        frame["Timestamp"] = pd.to_datetime(
            frame[timestamp_column],
            utc=True,
            errors="coerce"
        )
        frame["AbsoluteResidual"] = (
            pd.to_numeric(frame["actual"], errors="coerce")
            - pd.to_numeric(frame["forecast"], errors="coerce")
        ).abs()
        frame = frame.dropna(
            subset=["Timestamp", "AbsoluteResidual"]
        )

        for timestamp, residual in zip(
            frame["Timestamp"],
            frame["AbsoluteResidual"]
        ):
            records.append({
                "Timestamp": timestamp,
                "ModelFamily": model_family,
                "Configuration": configuration,
                "Season": season,
                "AdaptationMethod": method,
                "AbsoluteResidual": float(residual),
                "SourceFile": str(prediction_path)
            })

    return pd.DataFrame(records)


transfer_residuals_long = load_transfer_holdout_residuals()

if transfer_residuals_long.empty:
    print(
        "No nested transfer prediction files are available yet. "
        "Run the desired deep-learning chunks first."
    )
else:
    display(
        transfer_residuals_long.groupby([
            "Configuration",
            "Season",
            "AdaptationMethod",
            "ModelFamily"
        ]).size().rename("HoldoutObservations")
    )


In [ ]:
# Friedman plus Wilcoxon/Holm comparisons
def holm_adjustment(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    sorted_values = p_values[order]
    count = len(sorted_values)

    adjusted_sorted = np.array([
        min((count - index) * value, 1.0)
        for index, value in enumerate(sorted_values)
    ])
    adjusted_sorted = np.maximum.accumulate(
        adjusted_sorted
    )

    adjusted = np.empty(count)
    adjusted[order] = adjusted_sorted
    return adjusted


friedman_rows = []
posthoc_rows = []
residual_summary_rows = []

if not transfer_residuals_long.empty:
    grouping_columns = [
        "Configuration",
        "Season",
        "AdaptationMethod"
    ]

    for group_values, group in (
        transfer_residuals_long.groupby(grouping_columns)
    ):
        configuration, season, method = group_values

        descriptive = (
            group.groupby("ModelFamily")["AbsoluteResidual"]
            .agg(["count", "mean", "median", "std", "min", "max"])
            .reset_index()
        )
        descriptive["Configuration"] = configuration
        descriptive["Season"] = season
        descriptive["AdaptationMethod"] = method
        residual_summary_rows.append(descriptive)

        families = sorted(group["ModelFamily"].unique())
        if len(families) < 3:
            print(
                "Friedman skipped:",
                configuration,
                season,
                method,
                "— fewer than three model families."
            )
            continue

        paired = group.pivot_table(
            index="Timestamp",
            columns="ModelFamily",
            values="AbsoluteResidual",
            aggfunc="mean"
        ).dropna(subset=families)

        if paired.empty:
            print(
                "Friedman skipped — no paired timestamps:",
                configuration,
                season,
                method
            )
            continue

        statistic, p_value = friedmanchisquare(
            *[paired[family] for family in families]
        )
        friedman_rows.append({
            "Configuration": configuration,
            "Season": season,
            "AdaptationMethod": method,
            "ModelsCompared": ", ".join(families),
            "PairedObservations": len(paired),
            "FriedmanStatistic": statistic,
            "FriedmanPValue": p_value,
            "SignificantAt0.05": p_value < 0.05
        })

        if p_value < 0.05:
            temporary_rows = []
            for family_1, family_2 in combinations(families, 2):
                difference = paired[family_1] - paired[family_2]
                if np.allclose(difference, 0):
                    pair_statistic, pair_p = 0.0, 1.0
                else:
                    pair_statistic, pair_p = wilcoxon(
                        paired[family_1],
                        paired[family_2],
                        zero_method="wilcox",
                        alternative="two-sided"
                    )
                temporary_rows.append({
                    "Configuration": configuration,
                    "Season": season,
                    "AdaptationMethod": method,
                    "Model1": family_1,
                    "Model2": family_2,
                    "WilcoxonStatistic": pair_statistic,
                    "RawPValue": pair_p
                })

            adjusted = holm_adjustment([
                row["RawPValue"]
                for row in temporary_rows
            ])
            for row, adjusted_p in zip(
                temporary_rows,
                adjusted
            ):
                row["HolmAdjustedP"] = adjusted_p
                row["SignificantAfterHolm"] = adjusted_p < 0.05
                posthoc_rows.append(row)

friedman_summary = pd.DataFrame(friedman_rows)
wilcoxon_posthoc_summary = pd.DataFrame(posthoc_rows)
absolute_residual_summary = (
    pd.concat(residual_summary_rows, ignore_index=True)
    if residual_summary_rows
    else pd.DataFrame()
)

display(friedman_summary)
display(wilcoxon_posthoc_summary)
display(absolute_residual_summary)

friedman_summary.to_csv(
    COMPARISON_OUTPUT / "friedman_summary.csv",
    index=False
)
wilcoxon_posthoc_summary.to_csv(
    COMPARISON_OUTPUT / "wilcoxon_holm_posthoc_summary.csv",
    index=False
)
absolute_residual_summary.to_csv(
    COMPARISON_OUTPUT / "absolute_residual_summary.csv",
    index=False
)
transfer_residuals_long.to_csv(
    COMPARISON_OUTPUT / "paired_holdout_residuals_used.csv",
    index=False
)

print("Statistical comparisons saved to:", COMPARISON_OUTPUT)


## Deep-learning seasonal radar plots

Radar plots reproduce the Alexander Orr seasonal comparison format. They are generated separately for every predictor configuration and adaptation method, with ANN, LSTM and TFT plotted across Winter, Summer, Spring and Autumn.


In [ ]:
# Seasonal radar-plot helpers
RADAR_OUTPUT = Path(TRANSFER_OUTPUT) / "Radar_Plots"
RADAR_OUTPUT.mkdir(parents=True, exist_ok=True)

RADAR_SEASONS = [
    "Winter",
    "Summer",
    "Spring",
    "Autumn"
]
RADAR_METRICS = ["RMSE", "R2", "MAPE", "MAE"]


def make_transfer_radar_plot(
    metric,
    metric_values,
    configuration,
    adaptation_method
):
    angles = np.linspace(
        0,
        2 * np.pi,
        len(RADAR_SEASONS),
        endpoint=False
    ).tolist()
    closed_angles = angles + angles[:1]

    all_values = [
        value
        for family_values in metric_values.values()
        for value in family_values
    ]
    minimum = min(all_values)
    maximum = max(all_values)

    if metric == "R2" and minimum < 0:
        radial_minimum = minimum * 1.10
        radial_maximum = (
            maximum * 1.20
            if maximum > 0
            else maximum * 0.90
        )
    else:
        radial_minimum = 0
        radial_maximum = maximum * 1.25

    if radial_maximum == radial_minimum:
        radial_maximum = radial_minimum + 1

    figure = plt.figure(figsize=(18, 18))
    axis = plt.subplot(111, polar=True)
    axis.set_theta_offset(pi / 2)
    axis.set_theta_direction(-1)
    axis.set_xticks(angles)
    axis.set_xticklabels(
        RADAR_SEASONS,
        fontsize=24,
        fontweight="bold"
    )
    axis.set_ylim(radial_minimum, radial_maximum)
    axis.grid(True, linewidth=1.3, alpha=0.30)

    for family, values in metric_values.items():
        closed_values = values + values[:1]
        axis.plot(
            closed_angles,
            closed_values,
            marker="o",
            linewidth=3.5,
            markersize=9,
            label=family
        )
        axis.fill(closed_angles, closed_values, alpha=0.05)

    plt.title(
        f"Seasonal Model Performance Comparison ({metric})\n"
        f"{configuration} — {adaptation_method}",
        fontsize=36,
        fontweight="bold",
        pad=90
    )
    axis.legend(
        loc="upper left",
        bbox_to_anchor=(0.97, 1.10),
        frameon=False,
        fontsize=20
    )

    table_data = []
    for season_index, season in enumerate(RADAR_SEASONS):
        row = [season]
        for family in metric_values:
            row.append(
                f"{metric_values[family][season_index]:.4f}"
            )
        table_data.append(row)

    table = plt.table(
        cellText=table_data,
        colLabels=["Season"] + list(metric_values.keys()),
        cellLoc="center",
        colLoc="center",
        loc="bottom",
        bbox=[-0.10, -0.40, 1.18, 0.28]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(16)
    plt.subplots_adjust(
        bottom=0.32,
        top=0.76,
        left=0.08,
        right=0.82
    )

    output_dir = (
        RADAR_OUTPUT
        / configuration
        / adaptation_method
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = (
        output_dir
        / f"seasonal_radar_{metric.lower()}.png"
    )
    plt.savefig(output_path, dpi=1200, bbox_inches="tight")
    plt.show()
    plt.close(figure)
    print("Saved:", output_path)


In [ ]:
# Generate radar plots directly from nested seasonal metrics files
def load_nested_transfer_metrics():
    metric_frames = []
    metric_files = sorted(
        Path(TRANSFER_OUTPUT).glob(
            "*/*/*/*_metrics.csv"
        )
    )

    for metric_path in metric_files:
        relative = metric_path.relative_to(
            Path(TRANSFER_OUTPUT)
        )
        if len(relative.parts) < 4:
            continue

        family, configuration, season = relative.parts[:3]
        if family not in {"ANN", "LSTM", "TFT"}:
            continue

        method = identify_transfer_method(metric_path.name)
        if method is None:
            continue

        frame = pd.read_csv(metric_path)
        if frame.empty:
            continue

        frame = frame.iloc[[0]].copy()
        frame["ModelFamily"] = family
        frame["Configuration"] = configuration
        frame["Season"] = season
        frame["AdaptationMethod"] = method
        frame["SourceMetricsFile"] = str(metric_path)
        metric_frames.append(frame)

    if not metric_frames:
        return pd.DataFrame()

    return pd.concat(metric_frames, ignore_index=True)


radar_results = load_nested_transfer_metrics()

if radar_results.empty:
    print(
        "No nested transfer metrics files are available yet. "
        "Run the desired seasonal chunks first."
    )
else:
    print(
        "Radar metrics loaded from nested folders:",
        len(radar_results)
    )
    display(
        radar_results[[
            "ModelFamily",
            "Configuration",
            "Season",
            "AdaptationMethod",
            "SourceMetricsFile"
        ]]
    )

    for configuration in sorted(
        radar_results["Configuration"].unique()
    ):
        for method in TRANSFER_METHODS:
            subset = radar_results.loc[
                (radar_results["Configuration"] == configuration)
                & (radar_results["AdaptationMethod"] == method)
            ].copy()

            if subset.empty:
                continue

            for metric in RADAR_METRICS:
                metric_values = {}
                for family in ["ANN", "LSTM", "TFT"]:
                    family_rows = subset.loc[
                        subset["ModelFamily"] == family
                    ]
                    season_values = []
                    complete = True

                    for season in RADAR_SEASONS:
                        value_rows = family_rows.loc[
                            family_rows["Season"] == season,
                            metric
                        ]
                        if value_rows.empty:
                            complete = False
                            break
                        season_values.append(
                            float(value_rows.iloc[0])
                        )

                    if complete:
                        metric_values[family] = season_values
                    else:
                        print(
                            "Radar skipped for incomplete family:",
                            family,
                            configuration,
                            method,
                            metric
                        )

                if metric_values:
                    make_transfer_radar_plot(
                        metric=metric,
                        metric_values=metric_values,
                        configuration=configuration,
                        adaptation_method=method
                    )


## 15. Transfer-learning interpretation

- `Frozen_Source` measures strict Alexander Orr to los filtros transportability.
- `Target_From_Scratch` uses the saved source-selected architecture without
  transferring Alexander Orr weights.
- `Weight_Transfer` adapts the prediction head and then performs
  low-learning-rate full-network fine-tuning.
- For each season, training uses continuous earlier losfiltros history, the second
  month is used for validation, and the last month is the untouched holdout.
- A transfer benefit is supported only when `Weight_Transfer` improves on
  `Target_From_Scratch` on the identical monthly holdout.
- Improvement over `Frozen_Source` demonstrates target-domain adaptation, not
  zero-shot generalization.
- The lower losfiltros chlorine range is learned only from the target training
  period; validation and holdout observations never update model weights.


# Percentile-normalized distribution-adapted transfer exploration

This optional section consumes the percentile-normalized ANN, LSTM, and TFT
source artifacts created in the Alexander Orr notebook. It leaves every
existing raw-scale transfer, baseline, EDA, and Los Filtros Caguas classification result
unchanged.

For each saved source model, los filtros percentile mappings are fitted using
the target-domain training window only. Frozen source weights, target-from-
scratch training, and weight transfer are then compared on the same UTC target
timestamps and untouched seasonal holdout. Results are reported on both the
percentile scale and after inverse mapping to los filtros mg/L.

Because historical los filtros observations define the target percentile
mapping, this is distribution-adapted transfer rather than strict zero-shot
external validation.


In [ ]:
# Percentile-normalized transfer settings and helpers.
PERCENTILE_SOURCE_MODEL_DIR = os.path.join(
    SAVED_MODEL_DIR,
    "Percentile_Normalized",
    "DeepLearning"
)

PERCENTILE_TRANSFER_OUTPUT = os.path.join(
    EXTERNAL_OUTPUT,
    "Percentile_Normalized_Transfer"
)

os.makedirs(PERCENTILE_TRANSFER_OUTPUT, exist_ok=True)


def fit_target_percentile_transformers(
    data,
    metadata
):
    # Use the same seasonal target-training window definition as the transfer
    # experiment. No validation or holdout value fits these transformations.
    season = metadata["season"]
    evaluation_year = latest_complete_season_year(
        data.index,
        season
    )
    validation_month = SEASONAL_EVALUATION_MONTHS[season][
        "validation_month"
    ]
    validation_start, _ = month_bounds(
        evaluation_year,
        validation_month,
        data.index.tz
    )
    requested_train_months = saved_training_months(metadata)
    train_start = (
        validation_start
        - pd.DateOffset(months=requested_train_months)
    )
    training_frame = data.loc[
        (data.index >= train_start)
        & (data.index < validation_start)
    ].copy()

    transformed_columns = [
        column
        for column in metadata["percentile_transformed_columns"]
        if column in data.columns
    ]
    missing_columns = sorted(
        set(metadata["percentile_transformed_columns"])
        - set(transformed_columns)
    )
    if missing_columns:
        raise ValueError(
            "los filtross missing percentile-model inputs: "
            f"{missing_columns}"
        )

    transformers = {}
    for column in transformed_columns:
        observed_training = training_frame
        fill_flag = f"{column}_forward_filled"
        if fill_flag in observed_training.columns:
            observed_training = observed_training.loc[
                ~observed_training[fill_flag].astype(bool)
            ]

        values = pd.to_numeric(
            observed_training[column],
            errors="coerce"
        ).dropna().to_numpy().reshape(-1, 1)

        if len(values) < 2:
            raise ValueError(
                f"Insufficient Los Filtros Caguas training observations for {column}."
            )

        transformer = QuantileTransformer(
            n_quantiles=min(1000, len(values)),
            output_distribution="uniform",
            subsample=None,
            random_state=SEED
        )
        transformer.fit(values)
        transformers[column] = transformer

    return transformers, train_start, validation_start


def apply_target_percentile_transformers(data, transformers):
    transformed = data.copy()

    for column, transformer in transformers.items():
        values = pd.to_numeric(
            transformed[column],
            errors="coerce"
        ).to_numpy().reshape(-1, 1)
        transformed[column] = transformer.transform(values).reshape(-1)

    return transformed


def inverse_target_percentiles(values, transformer):
    values = np.asarray(values, dtype=float).reshape(-1)
    clipped_values = np.clip(values, 0.0, 1.0)
    return transformer.inverse_transform(
        clipped_values.reshape(-1, 1)
    ).reshape(-1)

def percentile_band_metrics(actual_percentiles, forecast_percentiles):
    # Report error separately in low, typical, and high operating regions.
    frame = pd.DataFrame({
        "actual_percentile": np.asarray(actual_percentiles, dtype=float),
        "forecast_percentile": np.asarray(forecast_percentiles, dtype=float)
    }).dropna()

    band_edges = [0.0, 0.10, 0.30, 0.70, 0.90, 1.0]
    band_labels = [
        "0th-10th",
        "10th-30th",
        "30th-70th",
        "70th-90th",
        "90th-100th"
    ]
    frame["PercentileBand"] = pd.cut(
        frame["actual_percentile"],
        bins=band_edges,
        labels=band_labels,
        include_lowest=True
    )

    rows = []
    for band in band_labels:
        part = frame.loc[frame["PercentileBand"] == band]
        if part.empty:
            rows.append({
                "PercentileBand": band,
                "Observations": 0,
                "RMSE_Percentile": np.nan,
                "MAE_Percentile": np.nan,
                "Bias_Percentile": np.nan
            })
            continue

        errors = (
            part["forecast_percentile"]
            - part["actual_percentile"]
        )
        rows.append({
            "PercentileBand": band,
            "Observations": int(len(part)),
            "RMSE_Percentile": float(np.sqrt(np.mean(errors ** 2))),
            "MAE_Percentile": float(np.mean(np.abs(errors))),
            "Bias_Percentile": float(np.mean(errors))
        })

    return pd.DataFrame(rows)



def percentile_model_required_columns(metadata):
    family = metadata["model_family"].lower()

    if family == "ann":
        return list(dict.fromkeys(
            metadata["numeric_base_cols"]
            + metadata["categorical_cols"]
            + [metadata["target_col"]]
        ))
    if family == "lstm":
        return list(metadata["model_cols"])
    if family == "tft":
        return list(dict.fromkeys(
            metadata["model_cols"]
            + metadata["static_numeric_cols"]
        ))
    raise ValueError(f"Unsupported percentile model family: {family}")


def discover_percentile_source_models():
    records = []
    root = Path(PERCENTILE_SOURCE_MODEL_DIR)

    for metadata_path in sorted(root.glob("*/*/*/model_metadata.json")):
        with open(metadata_path, encoding="utf-8") as file:
            metadata = json.load(file)

        family = metadata["model_family"].upper()
        configuration = metadata["config_name"]
        season = metadata["season"]

        if ACTIVE_MODEL_FAMILIES is not None and family not in ACTIVE_MODEL_FAMILIES:
            continue
        if ACTIVE_CONFIGURATIONS is not None and normalize_configuration_name(
            configuration
        ) not in [
            normalize_configuration_name(value)
            for value in ACTIVE_CONFIGURATIONS
        ]:
            continue
        if ACTIVE_SEASONS is not None and season not in ACTIVE_SEASONS:
            continue

        records.append((metadata_path.parent, metadata))

    return records


def percentile_predictions_to_mg_l(
    percentile_predictions,
    target_transformer
):
    converted = pd.DataFrame(index=percentile_predictions.index)
    converted["actual"] = inverse_target_percentiles(
        percentile_predictions["actual"],
        target_transformer
    )
    converted["forecast"] = inverse_target_percentiles(
        percentile_predictions["forecast"],
        target_transformer
    )

    for column in ["lower_ci", "upper_ci", "q10", "q50", "q90"]:
        if column in percentile_predictions.columns:
            converted[column] = inverse_target_percentiles(
                percentile_predictions[column],
                target_transformer
            )

    return converted


def run_one_percentile_transfer_model(model_dir, metadata):
    family = metadata["model_family"].lower()
    configuration = metadata["config_name"]
    season = metadata["season"]

    required_columns = percentile_model_required_columns(metadata)
    missing_columns = [
        column for column in required_columns
        if column not in losfiltroscaguas.columns
    ]
    if missing_columns:
        print(
            "Skipping percentile model",
            family,
            configuration,
            season,
            "because inputs are unavailable:",
            missing_columns
        )
        return []

    target_transformers, transform_train_start, transform_train_end = (
        fit_target_percentile_transformers(
            losfiltroscaguas,
            metadata
        )
    )
    percentile_data = apply_target_percentile_transformers(
        losfiltroscaguas,
        target_transformers
    )

    if family == "ann":
        prepared = prepare_ann_target_split(model_dir, percentile_data)
        source_model = tf.keras.models.load_model(
            Path(model_dir) / "frozen_model.keras"
        )
        models_to_compare = {
            "Percentile_Frozen_Source": source_model,
            "Percentile_Target_From_Scratch": train_ann_from_scratch(
                source_model,
                prepared
            ),
            "Percentile_Weight_Transfer": fine_tune_ann(
                source_model,
                prepared
            )
        }
        prediction_function = ann_predictions

    elif family == "lstm":
        prepared = prepare_lstm_target_split(model_dir, percentile_data)
        source_model = tf.keras.models.load_model(
            Path(model_dir) / "frozen_model.keras"
        )
        models_to_compare = {
            "Percentile_Frozen_Source": source_model,
            "Percentile_Target_From_Scratch": train_lstm_from_scratch(
                source_model,
                prepared
            ),
            "Percentile_Weight_Transfer": fine_tune_lstm(
                source_model,
                prepared
            )
        }
        prediction_function = lstm_predictions

    elif family == "tft":
        prepared = prepare_tft_target_split(model_dir, percentile_data)
        source_model = build_saved_tft(metadata).to(DEVICE)
        source_state = torch.load(
            Path(model_dir) / "frozen_model_state_dict.pt",
            map_location=DEVICE,
            weights_only=True
        )
        source_model.load_state_dict(source_state)
        models_to_compare = {
            "Percentile_Frozen_Source": source_model,
            "Percentile_Target_From_Scratch": train_tft_from_scratch(
                prepared
            ),
            "Percentile_Weight_Transfer": fine_tune_tft(
                source_model,
                prepared
            )
        }
        prediction_function = tft_predictions

    else:
        raise ValueError(f"Unsupported model family: {family}")

    output_dir = (
        Path(PERCENTILE_TRANSFER_OUTPUT)
        / family.upper()
        / configuration
        / season
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(
        target_transformers,
        output_dir / "losfiltros_training_percentile_transformers.joblib"
    )

    rows = []
    target_transformer = target_transformers[metadata["target_col"]]

    for method, model in models_to_compare.items():
        percentile_predictions = prediction_function(model, prepared)
        percentile_metrics = evaluate_forecast(
            percentile_predictions["actual"],
            percentile_predictions["forecast"]
        )
        percentile_metrics.update(interval_metrics(percentile_predictions))

        mg_l_predictions = percentile_predictions_to_mg_l(
            percentile_predictions,
            target_transformer
        )
        mg_l_metrics = evaluate_forecast(
            mg_l_predictions["actual"],
            mg_l_predictions["forecast"]
        )
        mg_l_metrics.update(interval_metrics(mg_l_predictions))

        row = {
            "Dataset": DATASET_NAME,
            "ModelFamily": family.upper(),
            "Configuration": configuration,
            "Season": season,
            "AdaptationMethod": method,
            "PercentileTransformTrainStart": transform_train_start,
            "PercentileTransformTrainEndExclusive": transform_train_end,
            "StrictZeroShot": False,
            **{
                f"Percentile_{key}": value
                for key, value in percentile_metrics.items()
            },
            **{
                f"mg_L_{key}": value
                for key, value in mg_l_metrics.items()
            }
        }
        rows.append(row)

        stem = f"{family.upper()}_{configuration}_{season}_{method}"
        percentile_predictions.to_csv(
            output_dir / f"{stem}_predictions_percentile.csv"
        )
        percentile_band_metrics(
            percentile_predictions["actual"],
            percentile_predictions["forecast"]
        ).assign(
            ModelFamily=family.upper(),
            Configuration=configuration,
            Season=season,
            AdaptationMethod=method
        ).to_csv(
            output_dir / f"{stem}_metrics_by_percentile_band.csv",
            index=False
        )
        mg_l_predictions.to_csv(
            output_dir / f"{stem}_predictions_mg_L.csv"
        )
        # pd.DataFrame([row]).to_csv(
        #     output_dir / f"{stem}_metrics_both_scales.csv",
        #     index=False
        # )

        # plot_true_holdout_forecast(
        #     predictions=mg_l_predictions,
        #     model_family=family.upper(),
        #     configuration=f"{configuration} percentile-normalized",
        #     season=season,
        #     adaptation_method=method,
        #     output_path=output_dir / f"{stem}_holdout_mg_L.png",
        #     interval_label=(
        #         "Inverse-mapped percentile forecast interval"
        #     )
        # )

        # Save this transfer method's percentile-scale and mg/L metrics.
        method_metrics_df = pd.DataFrame([
            row
        ])

        method_metrics_df.to_csv(
            output_dir / f"{stem}_metrics_both_scales.csv",
            index=False
        )

        # Display the metrics immediately after the method finishes.
        print(
            "\nPercentile-transfer metrics:",
            family.upper(),
            configuration,
            season,
            method
        )

        display(method_metrics_df)


        # Save and display percentile-scale plots using the Alexander Orr style.
        save_forecast_plots(
            pred_df=percentile_predictions,
            output_dir=str(output_dir),
            file_stem=f"{stem}_percentile",
            model_label=(
                f"{family.upper()} "
                f"{configuration} "
                f"{season} "
                f"{method} — Percentile Scale"
            ),
            actual_col="actual",
            pred_col="forecast",
            ylabel="Residual Chlorine Percentile",
            show_plot=True
        )


        # Save and display inverse-transformed plots in mg/L.
        save_forecast_plots(
            pred_df=mg_l_predictions,
            output_dir=str(output_dir),
            file_stem=f"{stem}_mg_L",
            model_label=(
                f"{family.upper()} "
                f"{configuration} "
                f"{season} "
                f"{method} — Original Scale"
            ),
            actual_col="actual",
            pred_col="forecast",
            ylabel="Residual Chlorine (mg/L)",
            show_plot=True
        )





        if method != "Percentile_Frozen_Source":
            if family in {"ann", "lstm"}:
                model.save(output_dir / f"{stem}.keras")
            else:
                torch.save(
                    model.state_dict(),
                    output_dir / f"{stem}_state_dict.pt"
                )

    with open(
        output_dir / "percentile_transfer_metadata.json",
        "w",
        encoding="utf-8"
    ) as file:
        json.dump({
            "source_model_dir": str(model_dir),
            "model_family": family,
            "configuration": configuration,
            "season": season,
            "timeline_timezone": "UTC",
            "target_timestamp_alignment": "t_plus_horizon",
            "target_distribution_used": "Los Filtros Caguas training window only",
            "strict_zero_shot": False,
            "interpretation": (
                "Percentile-normalized distribution-adapted transfer"
            )
        }, file, indent=4, default=str)

    return rows


def run_all_percentile_transfer_experiments():
    all_rows = []
    discovered_models = discover_percentile_source_models()

    if not discovered_models:
        raise FileNotFoundError(
            "No percentile-normalized Alexander Orr source artifacts were found. "
            "Run the new Alexander Orr percentile section first."
        )

    for model_dir, metadata in discovered_models:
        try:
            all_rows.extend(
                run_one_percentile_transfer_model(
                    model_dir,
                    metadata
                )
            )
        except Exception as error:
            print(
                "Percentile transfer skipped:",
                metadata.get("model_family"),
                metadata.get("config_name"),
                metadata.get("season"),
                "-",
                error
            )

    results = pd.DataFrame(all_rows)
    results.to_csv(
        Path(PERCENTILE_TRANSFER_OUTPUT)
        / "all_percentile_transfer_metrics_both_scales.csv",
        index=False
    )
    return results


# Run exactly one percentile-transfer family/configuration/season chunk.
def run_percentile_transfer_chunk(
    model_family,
    configuration,
    season
):
    requested_family = str(model_family).strip().upper()
    requested_configuration = normalize_configuration_name(
        configuration
    )
    requested_season = str(season).strip().title()

    matches = []
    root = Path(PERCENTILE_SOURCE_MODEL_DIR)

    for metadata_path in sorted(
        root.glob("*/*/*/model_metadata.json")
    ):
        with open(metadata_path, encoding="utf-8") as file:
            metadata = json.load(file)

        saved_family = str(
            metadata["model_family"]
        ).strip().upper()
        saved_configuration = normalize_configuration_name(
            metadata["config_name"]
        )
        saved_season = str(
            metadata["season"]
        ).strip().title()

        if (
            saved_family == requested_family
            and saved_configuration == requested_configuration
            and saved_season == requested_season
        ):
            matches.append((metadata_path.parent, metadata))

    if len(matches) == 0:
        raise FileNotFoundError(
            "No saved percentile model matched "
            f"{requested_family}/{requested_configuration}/"
            f"{requested_season} under {root}."
        )

    if len(matches) > 1:
        matched_folders = [
            str(model_dir)
            for model_dir, _ in matches
        ]
        raise ValueError(
            "Expected exactly one saved percentile model, "
            f"but found {len(matches)}: {matched_folders}"
        )

    model_dir, metadata = matches[0]

    print("=" * 78)
    print(
        "PERCENTILE TRANSFER CHUNK:",
        requested_family,
        requested_configuration,
        requested_season
    )
    print("Saved source model:", model_dir)
    print("=" * 78)

    rows = run_one_percentile_transfer_model(
        model_dir=model_dir,
        metadata=metadata
    )

    results = pd.DataFrame(rows)
    if not results.empty:
        display(results)

        chunk_output = (
            Path(PERCENTILE_TRANSFER_OUTPUT)
            / requested_family
            / metadata["config_name"]
            / requested_season
        )
        chunk_output.mkdir(parents=True, exist_ok=True)
        results.to_csv(
            chunk_output / "percentile_chunk_metrics.csv",
            index=False
        )

    return results



## Run percentile-normalized transfer models

Run this only after the desired percentile-normalized Alexander Orr source
artifacts have been created. The runner discovers every available ANN, LSTM,
and TFT configuration and season, while respecting the notebook's existing
active-model filters.


#LSTM Univariate

#Chunk 1-Winter

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="LSTM",
    configuration="univariate",
    season="Winter"
)

#Chunk 2-Spring

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="LSTM",
    configuration="univariate",
    season="Spring"
)

#Chunk 3-Summer

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="LSTM",
    configuration="univariate",
    season="Summer"
)

#Chunk 4-Autumn

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="LSTM",
    configuration="univariate",
    season="Autumn"
)

#LSTM Turbidity

#Chunk 1-Winter

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="LSTM",
    configuration="turbidity",
    season="Winter"
)

#Chunk 2-Spring

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="LSTM",
    configuration="turbidity",
    season="spring"
)

#Chunk 3-Summer

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="LSTM",
    configuration="turbidity",
    season="summer"
)

#Chunk 4-Autumn

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="LSTM",
    configuration="turbidity",
    season="autumn"
)

#ANN Univariate

#Chunk 1-Winter

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="ANN",
    configuration="univariate",
    season="Winter"
)


#Chunk 2-Spring

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="ANN",
    configuration="univariate",
    season="Spring"
)


#Chunk 3-Summer

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="ANN",
    configuration="univariate",
    season="Summer"
)

#Chunk 4-Autumn

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="ANN",
    configuration="univariate",
    season="Autumn"
)

#ANN Turbidity

#Chunk 1-Winter

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="ANN",
    configuration="turbidity",
    season="Winter"
)


#Chunk 2-Spring

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="ANN",
    configuration="turbidity",
    season="Spring"
)

#Chunk 3-Summer

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="ANN",
    configuration="turbidity",
    season="Summer"
)

#Chunk 4-Autumn

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="ANN",
    configuration="turbidity",
    season="Autumn"
)

#TFT Univariate

#Chunk 1-Winter

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="TFT",
    configuration="univariate",
    season="Winter"
)


#Chunk 2-Spring

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="TFT",
    configuration="univariate",
    season="Spring"
)


#Chunk 3-Summer

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="TFT",
    configuration="univariate",
    season="Summer"
)


#Chunk 4-Autumn

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="TFT",
    configuration="univariate",
    season="Autumn"
)


#TFT Flowrate

#Chunk 1-Winter

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="TFT",
    configuration="turbidity",
    season="Winter"
)


#Chunk 2-Spring

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="TFT",
    configuration="turbidity",
    season="Spring"
)

#Chunk 3-Summer

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="TFT",
    configuration="turbidity",
    season="Summer"
)

#Chunk 4-Autumn

In [ ]:
# Run one percentile-transfer chunk.
# Duplicate this cell and change the three selections for another run.
percentile_chunk_results = run_percentile_transfer_chunk(
    model_family="TFT",
    configuration="turbidity",
    season="Autumn"
)

# Final section: Los Filtros validation of Alexander Orr classification hyperparameters

It evaluates whether the **winning Alexander Orr classifier hyperparameters** remain useful for identifying the same 12-hour-ahead residual-chlorine exceedance condition at Los Filtros:


The Alexander Orr estimator types and winning hyperparameters are reused without conducting a new Los Filtros hyperparameter search. Because Los Filtros does not contain flowrate or chlorine-conversion information, the validation uses only predictors that can be constructed consistently at this plant:

- residual chlorine (`totalchlorine`)
- turbidity (`turbidity`)
- lagged, rolling, change, and UTC calendar features derived from those variables

Development cross-validation is used to describe performance with the locked Alexander Orr hyperparameters. An independent Los Filtros calibration period selects each probability threshold using F2, and the final chronological holdout remains untouched until evaluation. This is **configuration-transfer validation**, because the hyperparameters are transferred but the classifiers must be fitted using the compatible Los Filtros feature space.


In [ ]:
# ============================================================
# LOS FILTROS CLASSIFICATION SETTINGS
# ============================================================
LOSFILTROS_ML_HORIZON = 12
LOSFILTROS_ML_LAGS = 24
LOSFILTROS_ML_HOLDOUT_START = pd.Timestamp("2026-01-01", tz="UTC")
LOSFILTROS_ML_CALIBRATION_HOURS = 90 * 24
LOSFILTROS_ML_CV_SPLITS = 5
LOSFILTROS_ML_CV_GAP = LOSFILTROS_ML_HORIZON
LOSFILTROS_ML_THRESHOLD_BETA = 2.0
LOSFILTROS_ML_THRESHOLD_GRID = np.round(np.arange(0.05, 0.96, 0.05), 2)

LOSFILTROS_ML_OUTPUT = Path(EXTERNAL_OUTPUT) / "Machine_Learning_Classification"
LOSFILTROS_ML_OUTPUT.mkdir(parents=True, exist_ok=True)

# This is the folder created by the Alexander Orr classification section.
ALEXANDER_ORR_CLASSIFICATION_DIR = Path(SAVED_MODEL_DIR) / "Classification"

# Run every available Alexander Orr winner, or provide a subset such as:
# ["LogisticRegression", "RandomForest"]
ACTIVE_LOSFILTROS_ML_MODELS = None

print("Los Filtros ML output:", LOSFILTROS_ML_OUTPUT)
print("Alexander Orr winner folder:", ALEXANDER_ORR_CLASSIFICATION_DIR)
print("Forecast horizon:", LOSFILTROS_ML_HORIZON, "hours")
print("Available raw predictors: totalchlorine and turbidity")


In [ ]:
# ============================================================
# FEATURE, SPLIT, HYPERPARAMETER, AND EVALUATION HELPERS
# ============================================================
def _json_safe(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (list, tuple)):
        return [_json_safe(item) for item in value]
    if isinstance(value, dict):
        return {str(key): _json_safe(item) for key, item in value.items()}
    return str(value)


def discover_alexander_orr_winning_estimators():
    """Load only estimator classes and winning hyperparameters from AO."""
    model_paths = sorted(
        ALEXANDER_ORR_CLASSIFICATION_DIR.glob("*_frozen_pipeline.joblib")
    )
    if not model_paths:
        model_paths = sorted(
            ALEXANDER_ORR_CLASSIFICATION_DIR.glob("*_pipeline.joblib")
        )
    if not model_paths:
        raise FileNotFoundError(
            "No saved Alexander Orr classifier pipelines were found in "
            f"{ALEXANDER_ORR_CLASSIFICATION_DIR}. Run or copy the Alexander "
            "Orr classification winners before this section."
        )

    winners = {}
    audit_rows = []
    for model_path in model_paths:
        model_name = model_path.name
        for suffix in ["_frozen_pipeline.joblib", "_pipeline.joblib"]:
            model_name = model_name.replace(suffix, "")
        if (
            ACTIVE_LOSFILTROS_ML_MODELS is not None
            and model_name not in ACTIVE_LOSFILTROS_ML_MODELS
        ):
            continue

        source_pipeline = joblib.load(model_path)
        if hasattr(source_pipeline, "named_steps") and "model" in source_pipeline.named_steps:
            source_estimator = source_pipeline.named_steps["model"]
        else:
            source_estimator = source_pipeline

        source_params = source_estimator.get_params(deep=False)
        estimator = clone(source_estimator).set_params(**source_params)
        winners[model_name] = estimator
        audit_rows.append({
            "Model": model_name,
            "SourcePipeline": str(model_path),
            "EstimatorClass": type(source_estimator).__name__,
            "AlexanderOrrWinningHyperparameters": json.dumps(
                _json_safe(source_params), sort_keys=True
            )
        })

    if not winners:
        raise ValueError("No Alexander Orr winning classifiers are active.")
    audit = pd.DataFrame(audit_rows)
    audit.to_csv(
        LOSFILTROS_ML_OUTPUT / "alexander_orr_winning_hyperparameters_reused.csv",
        index=False
    )
    with open(
        LOSFILTROS_ML_OUTPUT / "alexander_orr_winning_hyperparameters_reused.json",
        "w"
    ) as file:
        json.dump(audit_rows, file, indent=4)
    display(audit)
    return winners, audit


def build_losfiltros_ml_feature_frame(data):
    work = data.copy().sort_index()
    if not isinstance(work.index, pd.DatetimeIndex):
        raise ValueError("losfiltroscaguas must have a DatetimeIndex.")
    if str(work.index.tz) != "UTC":
        raise ValueError("Los Filtros ML features require a UTC index.")

    required = ["totalchlorine", "turbidity"]
    missing = [column for column in required if column not in work.columns]
    if missing:
        raise ValueError(f"Los Filtros ML columns are missing: {missing}")
    for column in required:
        work[column] = pd.to_numeric(work[column], errors="coerce")

    for column in required:
        for lag in range(1, LOSFILTROS_ML_LAGS + 1):
            work[f"{column}_lag_{lag}"] = work[column].shift(lag)

    work["chlorine_roll_mean_6"] = work["totalchlorine"].rolling(6).mean()
    work["chlorine_roll_std_6"] = work["totalchlorine"].rolling(6).std()
    work["chlorine_roll_max_12"] = work["totalchlorine"].rolling(12).max()
    work["chlorine_roll_min_12"] = work["totalchlorine"].rolling(12).min()
    work["chlorine_diff_1"] = work["totalchlorine"].diff(1)
    work["chlorine_diff_3"] = work["totalchlorine"].diff(3)
    work["turbidity_roll_mean_6"] = work["turbidity"].rolling(6).mean()
    work["turbidity_roll_std_6"] = work["turbidity"].rolling(6).std()
    work["turbidity_diff_1"] = work["turbidity"].diff(1)

    work["utc_hour"] = work.index.hour
    work["utc_hour_sin"] = np.sin(2 * np.pi * work["utc_hour"] / 24.0)
    work["utc_hour_cos"] = np.cos(2 * np.pi * work["utc_hour"] / 24.0)
    work["utc_day_of_week"] = work.index.dayofweek
    work["utc_month"] = work.index.month
    work["utc_dow_sin"] = np.sin(2 * np.pi * work["utc_day_of_week"] / 7.0)
    work["utc_dow_cos"] = np.cos(2 * np.pi * work["utc_day_of_week"] / 7.0)

    feature_cols = [
        "totalchlorine", "turbidity",
        "chlorine_roll_mean_6", "chlorine_roll_std_6",
        "chlorine_roll_max_12", "chlorine_roll_min_12",
        "chlorine_diff_1", "chlorine_diff_3",
        "turbidity_roll_mean_6", "turbidity_roll_std_6",
        "turbidity_diff_1", "utc_hour", "utc_hour_sin", "utc_hour_cos",
        "utc_day_of_week", "utc_month", "utc_dow_sin", "utc_dow_cos"
    ] + [
        f"{column}_lag_{lag}"
        for column in required
        for lag in range(1, LOSFILTROS_ML_LAGS + 1)
    ]
    feature_cols = list(dict.fromkeys(feature_cols))

    work["future_chlorine"] = work["totalchlorine"].shift(-LOSFILTROS_ML_HORIZON)
    work["target_timestamp"] = pd.Series(
        work.index, index=work.index
    ).shift(-LOSFILTROS_ML_HORIZON)
    if "totalchlorine_forward_filled" in work.columns:
        work["target_was_forward_filled"] = work[
            "totalchlorine_forward_filled"
        ].shift(-LOSFILTROS_ML_HORIZON)
    else:
        work["target_was_forward_filled"] = False

    frame = work.dropna(
        subset=feature_cols + ["future_chlorine", "target_timestamp"]
    ).copy()
    frame = frame.loc[
        ~frame["target_was_forward_filled"].astype(bool)
    ].copy()
    return frame, feature_cols


def split_losfiltros_ml_frame(frame):
    calibration_start = (
        LOSFILTROS_ML_HOLDOUT_START
        - pd.Timedelta(hours=LOSFILTROS_ML_CALIBRATION_HOURS)
    )
    development = frame.loc[
        frame["target_timestamp"] < calibration_start
    ].copy()
    calibration = frame.loc[
        (frame["target_timestamp"] >= calibration_start)
        & (frame["target_timestamp"] < LOSFILTROS_ML_HOLDOUT_START)
    ].copy()
    holdout = frame.loc[
        frame["target_timestamp"] >= LOSFILTROS_ML_HOLDOUT_START
    ].copy()
    if any(part.empty for part in [development, calibration, holdout]):
        raise ValueError(
            "The configured dates leave an empty development, calibration, "
            "or holdout period."
        )
    return development, calibration, holdout


def add_task_target(frame, task_name, cutoff):
    if task_name != "EPA_Upper_Exceedance_4mgL":
        raise ValueError(
            "This notebook validates only the original 4.0 mg/L "
            "Alexander Orr exceedance definition."
        )
    result = frame.copy()
    result["target"] = (
        result["future_chlorine"] >= cutoff
    ).astype(int)
    return result


def classifier_probabilities(fitted_pipeline, X):
    if hasattr(fitted_pipeline, "predict_proba"):
        return fitted_pipeline.predict_proba(X)[:, 1]
    scores = fitted_pipeline.decision_function(X)
    return 1.0 / (1.0 + np.exp(-scores))


def classification_metrics(y_true, probability, threshold):
    y_true = np.asarray(y_true, dtype=int)
    prediction = (np.asarray(probability) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        "Accuracy": accuracy_score(y_true, prediction),
        "Precision": precision_score(y_true, prediction, zero_division=0),
        "Recall": recall_score(y_true, prediction, zero_division=0),
        "F1": f1_score(y_true, prediction, zero_division=0),
        "F2": fbeta_score(
            y_true, prediction, beta=LOSFILTROS_ML_THRESHOLD_BETA,
            zero_division=0
        ),
        "AveragePrecision": (
            average_precision_score(y_true, probability)
            if np.unique(y_true).size == 2 else np.nan
        ),
        "ROC_AUC": (
            roc_auc_score(y_true, probability)
            if np.unique(y_true).size == 2 else np.nan
        ),
        "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)
    }


In [ ]:
# ============================================================
# FIXED-HYPERPARAMETER TARGET-PLANT VALIDATION RUNNER
# ============================================================
def run_losfiltros_fixed_hyperparameter_task(
    task_name,
    cutoff,
    winning_estimators,
    base_frame,
    feature_cols
):
    task_dir = LOSFILTROS_ML_OUTPUT / task_name
    models_dir = task_dir / "models"
    metrics_dir = task_dir / "metrics"
    predictions_dir = task_dir / "predictions"
    for folder in [models_dir, metrics_dir, predictions_dir]:
        folder.mkdir(parents=True, exist_ok=True)

    labeled = add_task_target(base_frame, task_name, cutoff)
    development, calibration, holdout = split_losfiltros_ml_frame(labeled)
    partitions = {
        "Development": development,
        "IndependentCalibration": calibration,
        "UntouchedHoldout": holdout
    }
    prevalence = pd.DataFrame([
        {
            "Task": task_name,
            "CutoffMgL": cutoff,
            "Subset": name,
            "Rows": len(part),
            "PositiveCount": int(part["target"].sum()),
            "NegativeCount": int((part["target"] == 0).sum()),
            "PositiveRate": float(part["target"].mean())
        }
        for name, part in partitions.items()
    ])
    prevalence.to_csv(metrics_dir / "class_prevalence.csv", index=False)
    display(prevalence)

    metadata = {
        "dataset": DATASET_NAME,
        "task": task_name,
        "cutoff_mg_L": float(cutoff),
        "cutoff_fitted_from": (
            "fixed EPA upper threshold"
            if task_name == "EPA_Upper_Exceedance_4mgL"
            else "development-period future chlorine values only"
        ),
        "forecast_horizon_hours": LOSFILTROS_ML_HORIZON,
        "lag_hours": LOSFILTROS_ML_LAGS,
        "holdout_start_utc": LOSFILTROS_ML_HOLDOUT_START.isoformat(),
        "calibration_hours": LOSFILTROS_ML_CALIBRATION_HOURS,
        "raw_predictors": ["totalchlorine", "turbidity"],
        "excluded_unavailable_predictors": [
            "finishwater", "chlorine_conversion"
        ],
        "feature_columns": feature_cols,
        "hyperparameter_policy": (
            "Reuse Alexander Orr winning estimator hyperparameters without "
            "a new Los Filtros hyperparameter search"
        ),
        "threshold_policy": (
            "Select probability threshold by maximum F2 on the independent "
            "Los Filtros calibration period"
        )
    }
    with open(task_dir / "modeling_metadata.json", "w") as file:
        json.dump(metadata, file, indent=4)
    with open(task_dir / "feature_columns.json", "w") as file:
        json.dump(feature_cols, file, indent=4)

    one_class_parts = [
        name for name, part in partitions.items()
        if part["target"].nunique() < 2
    ]
    if one_class_parts:
        status = pd.DataFrame([{
            "Task": task_name,
            "Status": "Skipped",
            "Reason": (
                "Both classes are required in every partition; one class "
                f"was found in: {', '.join(one_class_parts)}"
            )
        }])
        status.to_csv(metrics_dir / "task_status.csv", index=False)
        display(status)
        return {"status": "skipped", "prevalence": prevalence}

    X_development = development[feature_cols]
    y_development = development["target"].astype(int)
    X_calibration = calibration[feature_cols]
    y_calibration = calibration["target"].astype(int)
    X_holdout = holdout[feature_cols]
    y_holdout = holdout["target"].astype(int)

    # Development CV is diagnostic only: hyperparameters remain locked to AO.
    tscv = TimeSeriesSplit(
        n_splits=LOSFILTROS_ML_CV_SPLITS,
        gap=LOSFILTROS_ML_CV_GAP
    )
    cv_rows = []
    result_rows = []
    locked_rows = []

    for model_name, source_estimator in winning_estimators.items():
        preprocessor = ColumnTransformer(
            [("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), feature_cols)],
            remainder="drop"
        )
        for fold, (train_idx, validation_idx) in enumerate(
            tscv.split(X_development), start=1
        ):
            y_train_fold = y_development.iloc[train_idx]
            y_validation_fold = y_development.iloc[validation_idx]
            if y_train_fold.nunique() < 2 or y_validation_fold.nunique() < 2:
                cv_rows.append({
                    "Task": task_name, "Model": model_name, "Fold": fold,
                    "Status": "SkippedOneClass"
                })
                continue
            pipeline = Pipeline([
                ("preprocessor", clone(preprocessor)),
                ("model", clone(source_estimator))
            ])
            pipeline.fit(X_development.iloc[train_idx], y_train_fold)
            fold_probability = classifier_probabilities(
                pipeline, X_development.iloc[validation_idx]
            )
            cv_rows.append({
                "Task": task_name, "Model": model_name, "Fold": fold,
                "Status": "Completed",
                **classification_metrics(
                    y_validation_fold, fold_probability, 0.50
                )
            })

        fitted_pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", clone(source_estimator))
        ])
        fitted_pipeline.fit(X_development, y_development)
        calibration_probability = classifier_probabilities(
            fitted_pipeline, X_calibration
        )
        threshold_candidates = []
        for probability_threshold in LOSFILTROS_ML_THRESHOLD_GRID:
            values = classification_metrics(
                y_calibration, calibration_probability, probability_threshold
            )
            threshold_candidates.append({
                "Model": model_name,
                "ProbabilityThreshold": float(probability_threshold),
                **values
            })
        threshold_table = pd.DataFrame(threshold_candidates).sort_values(
            ["F2", "Recall", "Precision", "ProbabilityThreshold"],
            ascending=[False, False, False, True]
        )
        threshold_table.to_csv(
            metrics_dir / f"{model_name}_calibration_threshold_search.csv",
            index=False
        )
        locked_threshold = float(threshold_table.iloc[0]["ProbabilityThreshold"])
        locked_rows.append({
            "Task": task_name,
            "Model": model_name,
            "LockedProbabilityThreshold": locked_threshold,
            "SelectionSubset": "IndependentCalibration",
            "SelectionMetric": "F2"
        })

        holdout_probability = classifier_probabilities(
            fitted_pipeline, X_holdout
        )
        holdout_metrics = classification_metrics(
            y_holdout, holdout_probability, locked_threshold
        )
        result_rows.append({
            "Dataset": DATASET_NAME,
            "Task": task_name,
            "CutoffMgL": cutoff,
            "Model": model_name,
            "Hyperparameters": json.dumps(
                _json_safe(source_estimator.get_params(deep=False)),
                sort_keys=True
            ),
            "LockedProbabilityThreshold": locked_threshold,
            **holdout_metrics
        })
        prediction = (holdout_probability >= locked_threshold).astype(int)
        pd.DataFrame({
            "target_timestamp": pd.DatetimeIndex(holdout["target_timestamp"]),
            "actual_future_chlorine_mg_L": holdout["future_chlorine"].to_numpy(),
            "actual_class": y_holdout.to_numpy(),
            "predicted_probability": holdout_probability,
            "predicted_class": prediction
        }).to_csv(
            predictions_dir / f"{model_name}_untouched_holdout_predictions.csv",
            index=False
        )
        joblib.dump(
            fitted_pipeline,
            models_dir / f"{model_name}_fitted_pipeline.joblib"
        )

    cv_results = pd.DataFrame(cv_rows)
    locked_thresholds = pd.DataFrame(locked_rows)
    holdout_results = pd.DataFrame(result_rows)
    cv_results.to_csv(metrics_dir / "fixed_hyperparameter_cv_diagnostics.csv", index=False)
    locked_thresholds.to_csv(metrics_dir / "locked_probability_thresholds.csv", index=False)
    holdout_results.to_csv(metrics_dir / "final_untouched_holdout_results.csv", index=False)
    display(holdout_results)
    return {
        "status": "completed",
        "prevalence": prevalence,
        "cv_results": cv_results,
        "locked_thresholds": locked_thresholds,
        "holdout_results": holdout_results
    }


## Load the Alexander Orr winners and construct the Los Filtros validation frame

Run this once before the validation cell. The saved CSV and JSON tables provide a permanent audit of the exact Alexander Orr winning hyperparameters reused at Los Filtros.


In [ ]:
alexander_orr_winning_estimators, reused_hyperparameter_table = (
    discover_alexander_orr_winning_estimators()
)

losfiltros_ml_frame, losfiltros_ml_features = build_losfiltros_ml_feature_frame(
    losfiltroscaguas
)

validation_definition = pd.DataFrame([{
    "ValidationTask": "EPA_Upper_Exceedance_4mgL",
    "PositiveClassDefinition": "totalchlorine at t+12 hours >= 4.0 mg/L",
    "CutoffMgL": 4.0,
    "TransferredElement": "Alexander Orr winning estimator hyperparameters",
    "LosFiltrosRawPredictors": "totalchlorine, turbidity",
    "UnavailablePredictorsExcluded": "finishwater, chlorine_conversion"
}])
validation_definition.to_csv(
    LOSFILTROS_ML_OUTPUT / "classification_validation_definition.csv",
    index=False
)
display(validation_definition)


## Validate the Alexander Orr hyperparameters at Los Filtros

The positive class is residual chlorine at or above 4.0 mg/L 12 hours ahead. The class distribution is audited separately in the development, calibration, and untouched holdout periods before evaluation.


In [ ]:
losfiltros_exceedance_results = run_losfiltros_fixed_hyperparameter_task(
    task_name="EPA_Upper_Exceedance_4mgL",
    cutoff=4.0,
    winning_estimators=alexander_orr_winning_estimators,
    base_frame=losfiltros_ml_frame,
    feature_cols=losfiltros_ml_features
)


## Final Los Filtros hyperparameter-validation registry

Run this after the 4.0 mg/L validation. It inventories the saved Alexander Orr hyperparameters, locked Los Filtros probability thresholds, evaluation metrics, predictions, and fitted validation pipelines.


In [ ]:
registry_rows = []
for task_dir in sorted(LOSFILTROS_ML_OUTPUT.iterdir()):
    if not task_dir.is_dir():
        continue
    for model_path in sorted((task_dir / "models").glob("*_fitted_pipeline.joblib")):
        model_name = model_path.name.replace("_fitted_pipeline.joblib", "")
        registry_rows.append({
            "Dataset": DATASET_NAME,
            "Task": task_dir.name,
            "Model": model_name,
            "FittedPipeline": str(model_path),
            "Metadata": str(task_dir / "modeling_metadata.json"),
            "HoldoutMetrics": str(
                task_dir / "metrics" / "final_untouched_holdout_results.csv"
            ),
            "LockedThresholds": str(
                task_dir / "metrics" / "locked_probability_thresholds.csv"
            ),
            "HoldoutPredictions": str(
                task_dir / "predictions"
                / f"{model_name}_untouched_holdout_predictions.csv"
            )
        })

losfiltros_ml_registry = pd.DataFrame(registry_rows)
losfiltros_ml_registry.to_csv(
    LOSFILTROS_ML_OUTPUT / "losfiltros_ml_saved_model_registry.csv",
    index=False
)
display(losfiltros_ml_registry)
print("All Los Filtros ML outputs saved under:", LOSFILTROS_ML_OUTPUT)
